In [ ]:
# C02 - Installs, versions, directories, timer, seeds
import os, sys, time, json, math, gc, inspect, random, subprocess, platform, tempfile, hashlib, shutil, re, zipfile
from importlib import metadata as _md

T0 = time.time()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("./p1_work")
_tmp_root = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else tempfile.gettempdir()
RAW = os.path.join(_tmp_root, "p1_raw")
HF_HOME = os.path.join(RAW, "hf_home")
os.environ["HF_HOME"] = HF_HOME  # keep Hugging Face download caches OUT of /kaggle/working (20 GB cap)
PREP_DIR = os.path.join(WORK, "data")
OUT_DIR = os.path.join(WORK, "project1_output")
EXPORT_DIR, CKPT_DIR, EVAL_DIR, METRICS_DIR, HUMAN_DIR, TRAIN_DIR, MAN_DIR = [
    os.path.join(OUT_DIR, d) for d in ("final_model", "checkpoints", "predictions", "metrics", "human_eval", "logs", "manifests")]
# EVAL_DIR = predictions/ (the IN22 sample files); TRAIN_DIR = logs/ (training logs, run summary)
BPCC_DIR, BENCH_DIR, MODEL_SRC_DIR, SCRATCH = [os.path.join(RAW, d) for d in ("bpcc", "bench", "model", "scratch")]
for d in (WORK, RAW, HF_HOME, PREP_DIR, OUT_DIR, EXPORT_DIR, CKPT_DIR, EVAL_DIR, METRICS_DIR, HUMAN_DIR, TRAIN_DIR, MAN_DIR, BPCC_DIR, BENCH_DIR, MODEL_SRC_DIR, SCRATCH):
    os.makedirs(d, exist_ok=True)
_T0_FILE = os.path.join(WORK, ".session_t0")
try:  # a kernel restart INSIDE the same Kaggle session must not reset the 12 h clock (a stale file from another day is ignored)
    _t_saved = float(open(_T0_FILE).read())
    if not 0 <= T0 - _t_saved < 13 * 3600:
        raise ValueError("stale session start")
    T0 = _t_saved
except Exception:
    with open(_T0_FILE, "w") as _f:
        _f.write(repr(T0))
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)


def _pip(*pkgs, extra=()):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *pkgs], capture_output=True, text=True)
    return r.returncode == 0, (r.stderr or "")[-300:]


def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None


# transformers>=4.51 is required for qwen3 (older raises KeyError 'qwen3'); prefer the 4.57.x line.
# One resolver call so huggingface_hub / datasets / transformers stay mutually compatible (gradio is installed lazily in C29).
_LIBS = ["bitsandbytes>=0.43.3", "datasets", "pyarrow", "pandas", "tqdm", "safetensors", "matplotlib", "sacrebleu==2.3.1", "pyyaml"]
ok, err = _pip("transformers>=4.57,<4.58", *_LIBS)
assert ok, ("could not install the pinned transformers 4.57.x + libraries (no silent fallback to another transformers version: "
            f"a different Qwen3 implementation would make the run non-reproducible). pip said: {err}")
ok, _ = _pip("fasttext-wheel")
if not ok:
    ok, err = _pip("fasttext")
    print("fasttext installed" if ok else f"WARNING: fasttext unavailable (prep LID filter will fail): {err}")
ok, _ = _pip("indic-nlp-library", extra=("--no-deps",))  # only trivial_tokenize is needed
try:
    from indicnlp.tokenize import indic_tokenize  # noqa: F401
except Exception as e:
    print(f"WARNING: indicnlp not importable ({e!r}); light tokenizer fallback will be used outside the helper module")

import numpy as np
import torch
import transformers
major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
assert (major, minor) >= (4, 51) and major < 5, f"transformers {transformers.__version__} unsupported (need >=4.51,<5)"


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(42)
VERSIONS = {p: _ver(p) for p in ("torch", "transformers", "huggingface_hub", "datasets", "bitsandbytes", "safetensors",
                                 "pandas", "pyarrow", "numpy", "sacrebleu", "fasttext-wheel", "fasttext")}
print("python", platform.python_version(), "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpus", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print({k: v for k, v in VERSIONS.items() if v})
print(f"WORK={WORK} RAW={RAW} | free(WORK)={shutil.disk_usage(WORK).free / 1e9:.1f} GB")


In [ ]:
# C03 - CONFIG (one dict)
CONFIG = dict(
    # ---- identity (Hugging Face ids are READ-ONLY download sources; nothing is ever written to Hugging Face) ------------------------
    MODEL_ID="Qwen/Qwen3-1.7B-Base", DATA_ID="ai4bharat/BPCC", LID_ID="facebook/fasttext-language-identification",
    INDIC=["guj_Gujr", "hin_Deva", "mar_Deva"],
    # ---- data ----------------------------------------------------------------------------------
    POOL=264_000, N=249_600, GLOBAL_BATCH=64, STEPS=3_900, WARMUP_STEPS=156,
    HUMAN_CAP=0.35, LABSE_MIN=0.8, LID_MIN_PROB=0.6, EN_WORDS=(3, 60), MAX_TOKENS=512,
    DEV_PAIRS_PER_DIR=300, FLORES_DEV_PER_DIR=50,
    HUMAN_SUBSET_REGEX=r"seed|daily|wiki|ilci|massive|human|bpcc-h",   # real subsets: bpcc-seed-*, nllb_seed, daily, wiki, ilci, massive = human; printed in C06
    MINED_PARSE_CAP=400_000,     # random line cap per (pair, mined subset) while parsing (memory guard; ~95k pairs/language are needed)
    # IN22 is a TEST set.  Default False = it only drives decontamination; training data is shaped by BPCC statistics alone.
    # (True restores the spec's variants: length-histogram matching in C14 / digit-convention audit in C13.)
    USE_IN22_LENGTH_MATCH=False, USE_IN22_DIGIT_AUDIT=False,
    # mined pairs that carry no LaBSE score: "drop" (quality-controlled; if that would starve a language they are kept for that
    # language only, with a loud warning + manifest entry) or "keep".
    MINED_WITHOUT_LABSE="drop",
    CODE_VERSION="p1-2026-10-08.1",
    # when too few LaBSE-scored mined pairs exist for a language: "keep_unscored" (keeps N, weaker quality control for that language,
    # disclosed in manifest/data card/model card) or "shrink" (never re-add unscored pairs; the pool then cannot reach N and the run stops)
    # 'shrink' is REQUIRED for this run: a language with too few LaBSE-scored pairs makes preparation STOP with the numbers instead of
    # silently re-adding unscored (weaker-quality) pairs to reach N.
    ON_LABSE_STARVATION="shrink",
    SOURCE_CAP=0.20,
    RESERVE_PER_LANG=4_500,      # extra pairs per language used only to refill after >512-token drops
    # ---- model / training ----------------------------------------------------------------------
    TRAINED_LAYERS=10, LR_PEAK=3e-5, LR_MIN=3e-6, BETAS=(0.9, 0.95), EPS=1e-8, WD=0.01, CLIP=1.0,
    SCALER_INIT=2 ** 14, MICRO_BATCH_TOKENS=6000, LOSS_CHUNK=1024,
    OOM_LADDER=[(10, 6000), (10, 4500), (10, 3000), (8, 6000), (8, 4500), (8, 3000)],
    DEV_LOSS_EVERY=650, GEN_EVAL_STEPS=[975, 1950, 2925, 3900], LOCAL_CKPT_MIN=60, TRAIN_WALL_H=8.0,
    # training wall cap = min(TRAIN_WALL_H, SESSION_CAP_H - elapsed - POST_TRAIN_H - SAFETY_H), computed when training starts (prep shares the session)
    SESSION_CAP_H=12.0, POST_TRAIN_H=2.0, SAFETY_H=1.0,
    PRE_TRAIN_H=0.25,   # model download + build + validation between the end of preparation and the first training step
    NONFINITE_REPLAYS=3,   # a global batch whose gradients overflow in fp16 (the scaler skips that update) is replayed this many times before it counts as skipped
    EXPECTED_EX_S=10.49, OVERHEAD_FACTOR=1.08,   # measured median step throughput of the previous run; +8% for dev evaluations / checkpoints
    GEN_BATCH=32, SEED=42,
    # ---- evaluation ----------------------------------------------------------------------------
    BOOT_N=1000, BOOT_FALLBACK_N=300, HUMAN_N_EN2X=40, HUMAN_N_X2EN=20,
)
CONFIG["PAIRS"] = [f"eng_Latn-{c}" for c in CONFIG["INDIC"]]
CONFIG["DIRECTIONS"] = CONFIG["PAIRS"] + [f"{c}-eng_Latn" for c in CONFIG["INDIC"]]

assert CONFIG["N"] % CONFIG["GLOBAL_BATCH"] == 0, "N must be a multiple of GLOBAL_BATCH"
assert CONFIG["STEPS"] == CONFIG["N"] // CONFIG["GLOBAL_BATCH"], "STEPS != N // GLOBAL_BATCH"
assert CONFIG["N"] <= CONFIG["POOL"] and CONFIG["POOL"] % 6 == 0 and CONFIG["POOL"] % 3 == 0
assert CONFIG["WARMUP_STEPS"] == round(0.04 * CONFIG["STEPS"]), "WARMUP_STEPS must be 4% of STEPS"
assert CONFIG["GEN_EVAL_STEPS"] == [round(CONFIG["STEPS"] * f) for f in (0.25, 0.5, 0.75, 1.0)]
assert CONFIG["MINED_WITHOUT_LABSE"] in ("drop", "keep") and CONFIG["ON_LABSE_STARVATION"] in ("keep_unscored", "shrink")
assert CONFIG["TRAINED_LAYERS"] == 10 and CONFIG["MAX_TOKENS"] == 512 and CONFIG["GLOBAL_BATCH"] == 64
assert CONFIG["OOM_LADDER"][0] == (CONFIG["TRAINED_LAYERS"], CONFIG["MICRO_BATCH_TOKENS"])
print("N =", CONFIG["N"], "| steps =", CONFIG["STEPS"], "| global batch =", CONFIG["GLOBAL_BATCH"], "| trained layers =", CONFIG["TRAINED_LAYERS"], "+ final norm")


**Changed (C03):** `N` 204,800 -> **249,600** (3,900 steps = 64 x 3,900; warm-up 4% = 156; FLORES checkpoints at 975 / 1,950 / 2,925 / 3,900; dev loss every 650 steps so the last step is also evaluated), `POOL` 216,000 -> 264,000 (same ~6% head-room for the >512-token drops), `RESERVE_PER_LANG` 3,000 -> 4,500, `MINED_PARSE_CAP` 250,000 -> 400,000, `TRAIN_WALL_H` 7 -> 8, `ON_LABSE_STARVATION` -> `"shrink"`, new `EXPECTED_EX_S` / `OVERHEAD_FACTOR` (measured throughput). **Why:** +22% unique pairs from the SAME filters — reading more lines of the mined files (the cap is a memory/time control, not a quality filter) — with the exact-N rule and a strict LaBSE policy (a language that cannot supply enough LaBSE-scored pairs stops preparation with the numbers instead of silently re-adding unscored pairs). **Expected effect:** more unique English->Indic examples (about 41.6K per direction instead of 34.1K) at a training time of about 7.1 h.

In [ ]:
# C03b - Kaggle GPU + environment verification
assert torch.cuda.is_available(), "Kaggle GPU is required: Settings > Accelerator > GPU T4 x2, then Run All again."
_props = [torch.cuda.get_device_properties(i) for i in range(torch.cuda.device_count())]
print(f"GPU: {[p.name for p in _props]} | count {len(_props)} | VRAM {[round(p.total_memory / 1e9, 1) for p in _props]} GB each | "
      f"CUDA {torch.version.cuda} | PyTorch {torch.__version__}")
assert _props[0].total_memory >= 14e9, f"{_props[0].name} has {_props[0].total_memory / 1e9:.0f} GB: at least a 16 GB GPU (T4) is needed for the 6000-token micro-batches"
print(f"training uses cuda:0; {'cuda:1 additionally runs half of the IN22 generation jobs' if len(_props) > 1 else 'a single GPU runs all generation jobs'}")


In [ ]:
# C04 - Auth (read-only Hugging Face access), pinned revisions, gated-access test
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

S = {"revs": {}, "errors": [], "manifest": {}}


def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)


# HF_TOKEN = READ / DOWNLOAD authentication only (the BPCC dataset is gated).  No Hugging Face repository is created or modified by this
# notebook, the token needs no write permission, and no personal namespace (user name, dataset repo, model repo) is ever looked up.
HF_TOKEN = get_secret("HF_TOKEN")
assert HF_TOKEN, ("HF_TOKEN is required for gated Hugging Face resources.\n"
                  "Add it through Kaggle > Add-ons > Secrets.\n"
                  "No Hugging Face repository write access is required.\n"
                  "Make sure the account has accepted the BPCC terms.")
os.environ["HF_TOKEN"] = HF_TOKEN  # picked up by huggingface_hub / datasets for every download (no login call, no token file, no account lookup)


class _ReadOnlyHf:
    """The ONLY Hugging Face client of this notebook: exposes read-only metadata calls, nothing else."""
    _ALLOWED = frozenset({"model_info", "dataset_info", "list_repo_tree", "list_repo_files"})

    def __init__(self, inner):
        self._inner = inner

    def __getattr__(self, name):
        if name not in self._ALLOWED:
            raise PermissionError(f"Hugging Face is read-only in this notebook: '{name}' is not allowed")
        return getattr(self._inner, name)


api = _ReadOnlyHf(HfApi(token=HF_TOKEN))
S["revs"]["model"] = api.model_info(CONFIG["MODEL_ID"]).sha
S["revs"]["data"] = api.dataset_info(CONFIG["DATA_ID"]).sha  # listing works without accepting the terms
print("HF_TOKEN: read/download authentication only; no Hugging Face repository is created or modified.")
print("pinned model sha:", S["revs"]["model"], "| pinned BPCC sha:", S["revs"]["data"])


def list_bpcc():
    """{path: size} for every file of the pinned BPCC revision (recursive listing)."""
    if "inventory" not in S:
        S["inventory"] = {e.path: (getattr(e, "size", None) or 0) for e in api.list_repo_tree(
            CONFIG["DATA_ID"], repo_type="dataset", revision=S["revs"]["data"], recursive=True) if hasattr(e, "size")}
    return S["inventory"]


def gated_test():
    """Download the smallest BPCC text file: 401/403 means the terms were not accepted for this token."""
    inv = list_bpcc()
    cand = sorted((sz, fn) for fn, sz in inv.items() if sz > 0 and not fn.startswith("additional/") and not fn.lower().endswith(".zip")
                  and (any(fn.endswith(f"/{l}.tsv") for l in CONFIG["INDIC"]) or any(f"/{p}/" in fn for p in CONFIG["PAIRS"])))
    assert cand, ("no <subset>/<lang>.tsv or <subset>/eng_Latn-<lang>/* file found: the BPCC layout differs from what this notebook "
                  f"understands; top-level entries: {sorted({f.split('/')[0] for f in inv})[:30]}")
    try:
        hf_hub_download(CONFIG["DATA_ID"], cand[0][1], repo_type="dataset", revision=S["revs"]["data"], local_dir=BPCC_DIR)
    except (GatedRepoError, HfHubHTTPError) as e:
        raise RuntimeError("Cannot download BPCC. Accept the BPCC terms at https://huggingface.co/datasets/ai4bharat/BPCC "
                           f"with the account behind HF_TOKEN, then re-run. ({type(e).__name__})") from None
    os.remove(os.path.join(BPCC_DIR, cand[0][1]))
    print("gated access OK via", cand[0][1], f"({cand[0][0]} bytes)")


gated_test()
# the BPCC source-file manifest (path + size of every listed file of the pinned revision) is part of the data identity (manifest / checkpoint signature)
_inv = list_bpcc()
S["revs"]["bpcc_files"] = hashlib.sha256(json.dumps(sorted((f, s) for f, s in _inv.items() if not f.startswith("additional/"))).encode()).hexdigest()[:16]
S["revs"]["bpcc_n_files"] = len(_inv)
print(f"BPCC file manifest: {len(_inv)} files, sha {S['revs']['bpcc_files']}")


def fetch_model(tokenizer_only=False):
    """snapshot_download of the PINNED base-model commit into MODEL_SRC_DIR."""
    pats = ["tokenizer*", "vocab.json", "merges.txt", "config.json", "generation_config.json"]
    if not tokenizer_only:
        pats += ["*.safetensors", "*.safetensors.index.json"]
    return snapshot_download(CONFIG["MODEL_ID"], revision=S["revs"]["model"], local_dir=MODEL_SRC_DIR,
                             allow_patterns=pats, token=HF_TOKEN)


Helper code lives **inside the notebook** as the string `UTILS_SRC` (the next cells, C05b-d). It is `exec`'d into a module in C05e, and the two generation workers and the metrics subprocess receive the very same source **on stdin** (`python - <command> <config.json>`), so the same functions run everywhere and **no `.py` file is ever written** (a plain-text copy of the source is stored from memory as `p1_utils_source.txt` inside the prep artifact and the training outputs). Its import-time dependencies are stdlib + numpy only; torch / transformers / sacrebleu / indicnlp are imported lazily. Cell C05e runs assert-based unit tests of every **pure** function on **synthetic data** (no real-model forward/backward); the hand-written fixtures are *not* IN22 text.

In [ ]:
# U01 - helper source part 1/10 (kept as a string; no .py file is ever written)
UTILS_SRC = r'''# ==== PART 1: constants and text helpers
"""p1_utils - helpers shared by the p1-qwen3-indic-translator notebook, its worker
subprocesses (worker / metrics).  Import-time dependencies: stdlib + numpy only
(torch / transformers / sacrebleu / indicnlp are imported lazily inside functions)."""
import hashlib
import json
import math
import os
import re
import shutil
import time
import unicodedata

LANGS = {
    "eng_Latn": {"name": "English", "tok": "en"},
    "guj_Gujr": {"name": "Gujarati", "tok": "gu"},
    "hin_Deva": {"name": "Hindi", "tok": "hi"},
    "mar_Deva": {"name": "Marathi", "tok": "mr"},
}
NAME2CODE = {v["name"]: k for k, v in LANGS.items()}
INDIC = ["guj_Gujr", "hin_Deva", "mar_Deva"]
DIRECTIONS = [f"eng_Latn-{l}" for l in INDIC] + [f"{l}-eng_Latn" for l in INDIC]
SCRIPT_RANGES = {
    "guj_Gujr": [(0x0A80, 0x0AFF)],
    "hin_Deva": [(0x0900, 0x097F)],
    "mar_Deva": [(0x0900, 0x097F)],
    "eng_Latn": [(0x41, 0x5A), (0x61, 0x7A), (0xC0, 0x24F)],
}
LEN_EDGES = (8, 14, 22, 35, 60)  # English word-count buckets: <=8, 9-14, 15-22, 23-35, 36-60
LEAK_MARKERS = ("Translate ", "English:", "Gujarati:", "Hindi:", "Marathi:")
EOS_TOKEN = "<|endoftext|>"

_DROP = re.compile("[\u0000-\u0008\u000e-\u001f\u007f-\u009f\u00ad\u061c\u180e\u200b\u200e\u200f"
                   "\u202a-\u202e\u2060-\u206f\ufeff\ufff9-\ufffb]")
_WS = re.compile(r"\s+")
URL_RE = re.compile(r"(?:https?://|ftp://|www\.)\S+|\b[\w-]+\.(?:com|org|net|gov|edu|nic\.in|gov\.in)\b/?\S*", re.I)
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
HTML_RE = re.compile(r"<\s*/?\s*[a-zA-Z][^>]{0,100}>|&(?:[a-zA-Z]{2,8}|#\d{1,6});")
PHONE_RE = re.compile(r"(?<!\d)\+?\d(?:[\s\-]?\d){9,}(?!\d)")
_NUM_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")
_DIGIT_BASES = {"guj_Gujr": 0x0AE6, "hin_Deva": 0x0966, "mar_Deva": 0x0966}
_TO_ASCII = {b + i: 0x30 + i for b in (0x0966, 0x0AE6) for i in range(10)}
_CAT = {}
_SCR_CACHE = {}


def split_direction(direction):
    src, tgt = direction.split("-")
    assert src in LANGS and tgt in LANGS, f"bad direction {direction}"
    return src, tgt


def _cat1(ch):
    v = _CAT.get(ch)
    if v is None:
        v = _CAT[ch] = unicodedata.category(ch)[0]
    return v


def norm_text(s):
    """NFC; drop BOM/soft-hyphen/zero-width-space/bidi marks/control chars; NBSP and tabs become spaces;
    collapse whitespace.  U+200C (ZWNJ) and U+200D (ZWJ) are KEPT (they matter for Indic shaping)."""
    if s is None:
        return ""
    s = unicodedata.normalize("NFC", str(s))
    return _WS.sub(" ", _DROP.sub("", s)).strip()


def _classify(ch, code):
    cache = _SCR_CACHE.setdefault(code, {})
    v = cache.get(ch)
    if v is None:
        if unicodedata.category(ch)[0] in "LM":
            o = ord(ch)
            v = 1 if any(lo <= o <= hi for lo, hi in SCRIPT_RANGES[code]) else 0
        else:
            v = -1
        cache[ch] = v
    return v


def script_ratio(text, code):
    """Share of letters+combining marks that belong to the script of `code` (digits, punctuation,
    danda, ZWJ/ZWNJ are ignored).  0.0 when the text has no letters."""
    hit = tot = 0
    for ch in text:
        v = _classify(ch, code)
        if v >= 0:
            tot += 1
            hit += v
    return hit / tot if tot else 0.0


def to_ascii_digits(s):
    return s.translate(_TO_ASCII)


def to_native_digits(s, code):
    base = _DIGIT_BASES[code]
    return to_ascii_digits(s).translate({0x30 + i: base + i for i in range(10)})


def digit_counts(texts):
    """(ascii_digit_chars, native_digit_chars) over an iterable of strings."""
    a = n = 0
    for t in texts:
        for ch in t:
            o = ord(ch)
            if 0x30 <= o <= 0x39:
                a += 1
            elif o in _TO_ASCII:
                n += 1
    return a, n
'''


In [ ]:
# U02 - helper source part 2/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

def numerals(s):
    """Numbers in ASCII/Devanagari/Gujarati digits, mapped to ASCII; thousands commas removed."""
    out = []
    for m in _NUM_RE.findall(to_ascii_digits(s)):
        m = m.rstrip(",").replace(",", "")
        if m:
            out.append(m)
    return out


def numeral_check(src, tgt, min_frac=0.8):
    sn = numerals(src)
    if not sn:
        return True
    tn = set(numerals(tgt))
    return sum(n in tn for n in sn) / len(sn) >= min_frac


def dedup_key(s):
    s = to_ascii_digits(norm_text(s)).lower()
    return "".join(ch for ch in s if _cat1(ch) in "LMN")


def word_ngrams(s, n=8):
    """Set of word n-grams (normalised words); empty set when the text has fewer than n words."""
    words = []
    for w in to_ascii_digits(norm_text(s)).lower().split():
        w = "".join(ch for ch in w if _cat1(ch) in "LMN")
        if w:
            words.append(w)
    if len(words) < n:
        return set()
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


def has_noise(s):
    """True for URLs, e-mails, HTML tags/entities or phone-like (>=10 digit) strings."""
    return bool(URL_RE.search(s) or EMAIL_RE.search(s) or HTML_RE.search(s) or PHONE_RE.search(to_ascii_digits(s)))


def sym_digit_frac(s):
    """Fraction of non-space characters that are not letters/marks (digits, punctuation, symbols)."""
    chars = [c for c in s if not c.isspace()]
    if not chars:
        return 1.0
    return sum(1 for c in chars if _cat1(c) not in "LM") / len(chars)


def len_bucket(n_words):
    for i, e in enumerate(LEN_EDGES[:-1]):
        if n_words <= e:
            return i
    return len(LEN_EDGES) - 1


# ==== PART 2: prompt, loss alignment, batching
def build_prompt(src_name, tgt_name, src):
    return f"Translate {src_name} to {tgt_name}.\n{src_name}: {src}\n{tgt_name}:"


def encode_examples(tok, src_names, tgt_names, srcs, tgts, eos_id):
    """Prompt and target are tokenised SEPARATELY and concatenated; the target is ' ' + text + EOS.
    Returns (list_of_input_ids, list_of_prompt_len).  Same prompt path is used at inference."""
    prompts = [build_prompt(a, b, s) for a, b, s in zip(src_names, tgt_names, srcs)]
    p_ids = tok(prompts, add_special_tokens=False)["input_ids"]
    t_ids = tok([" " + t for t in tgts], add_special_tokens=False)["input_ids"]
    return [p + t + [eos_id] for p, t in zip(p_ids, t_ids)], [len(p) for p in p_ids]


def encode_example(tok, src_name, tgt_name, src, tgt, eos_id):
    ids, plen = encode_examples(tok, [src_name], [tgt_name], [src], [tgt], eos_id)
    return ids[0], plen[0]


def target_position_mask(prompt_len, length, T):
    """True at positions t in [p-1, L-2]: hidden state t predicts token t+1.  Built from LENGTHS only
    (never from the pad id).  Scalars -> (T,) ; 1-D tensors of size B -> (B,T)."""
    import torch
    t = torch.arange(T)
    p = torch.as_tensor(prompt_len)
    L = torch.as_tensor(length)
    if p.dim() == 1:
        p, L = p[:, None], L[:, None]
    return (t >= p - 1) & (t <= L - 2)


def gather_targets(hidden, ids, mask):
    """hidden (B,T,H), ids (B,T), mask (B,T) bool -> (h (n,H), labels (n,)) with labels = ids[t+1]."""
    import torch
    shifted = torch.zeros_like(ids)
    shifted[:, :-1] = ids[:, 1:]
    return hidden[mask], shifted[mask]


def chunked_ce(h, labels, W, chunk=1024, reduction="sum", use_ckpt=True):
    """Cross-entropy of h (n,H) against tied matrix W (V,H) in chunks; each chunk is re-computed in
    backward (torch.utils.checkpoint) so full logits are never stored; CE is computed in fp32."""
    import torch
    import torch.nn.functional as F
    from torch.utils.checkpoint import checkpoint

    def _ce(hc, lc):
        return F.cross_entropy((hc @ W.t()).float(), lc, reduction="none")

    parts = []
    for i in range(0, h.shape[0], chunk):
        hc, lc = h[i:i + chunk], labels[i:i + chunk]
        if use_ckpt and torch.is_grad_enabled() and hc.requires_grad:
            parts.append(checkpoint(_ce, hc, lc, use_reentrant=False))
        else:
            parts.append(_ce(hc, lc))
    loss = torch.cat(parts) if parts else h.new_zeros(0, dtype=torch.float32)
    return loss.sum() if reduction == "sum" else loss
'''


In [ ]:
# U03 - helper source part 3/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

def group_by_budget(lengths, budget):
    """Group example indices (longest first) so that max_len_in_group * group_size <= budget.
    A single example longer than the budget gets its own group."""
    order = sorted(range(len(lengths)), key=lambda i: -lengths[i])
    groups, cur = [], []
    for i in order:
        if cur and lengths[cur[0]] * (len(cur) + 1) > budget:
            groups.append(cur)
            cur = []
        cur.append(i)
    if cur:
        groups.append(cur)
    return groups


def lr_schedule(step, steps, warmup, lr_peak, lr_min):
    """Learning rate of the optimizer step `step` (0-based): linear warm-up to lr_peak, then cosine decay to lr_min.
    A pure function of the step - it does NOT depend on the wall clock, so every machine / run follows the identical trajectory;
    the last step (steps-1) uses exactly lr_min.  Always clamped to [lr_min, lr_peak]."""
    if step < warmup:
        return min(max(lr_peak * (step + 1) / warmup, lr_min), lr_peak)  # clamp: the first few warm-up steps would be < lr_min
    x = min(max((step + 1 - warmup) / (steps - warmup), 0.0), 1.0)
    return min(max(lr_min + 0.5 * (lr_peak - lr_min) * (1.0 + math.cos(math.pi * x)), lr_min), lr_peak)


def restore_order(items, order):
    """items[k] was produced for original index order[k]; return the list in original order."""
    assert len(items) == len(order) and sorted(order) == list(range(len(order))), "order is not a permutation"
    out = [None] * len(items)
    for k, i in enumerate(order):
        out[i] = items[k]
    return out


def postprocess_hyp(text):
    return unicodedata.normalize("NFC", text.replace("\r", "\n").split("\n", 1)[0].strip())


# ==== PART 3: sampling helpers, tokenisation for metrics, failure flags
def waterfill(avail, k, cap=None):
    """Allocate k items across keys as evenly as possible; avail[key] is the supply, cap the per-key max."""
    cap_of = {key: min(n, cap) if cap is not None else n for key, n in avail.items()}
    alloc = {key: 0 for key in avail}
    remaining = min(k, sum(cap_of.values()))
    active = [key for key in avail if cap_of[key] > 0]
    while remaining > 0 and active:
        share = max(1, remaining // len(active))
        nxt = []
        for key in active:
            give = min(share, cap_of[key] - alloc[key], remaining)
            alloc[key] += give
            remaining -= give
            if alloc[key] < cap_of[key]:
                nxt.append(key)
            if remaining == 0:
                break
        active = nxt
    return alloc


def largest_remainder(weights, total):
    """Integer allocation of `total` proportional to weights (sums exactly to total)."""
    s = float(sum(weights))
    raw = [w / s * total for w in weights]
    base = [int(math.floor(r)) for r in raw]
    for i in sorted(range(len(raw)), key=lambda j: raw[j] - base[j], reverse=True)[: total - sum(base)]:
        base[i] += 1
    return base


def light_tok(text):
    """Fallback tokenizer: runs of letters/marks/digits, every other non-space char on its own."""
    toks, cur = [], []
    for ch in text:
        if _cat1(ch) in "LMN":
            cur.append(ch)
        else:
            if cur:
                toks.append("".join(cur))
                cur = []
            if not ch.isspace():
                toks.append(ch)
    if cur:
        toks.append("".join(cur))
    return " ".join(toks)


_INDIC_NORMALIZERS = {}


def _indic_normalizer(code):
    n = _INDIC_NORMALIZERS.get(code)
    if n is None:
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
        n = _INDIC_NORMALIZERS[code] = IndicNormalizerFactory().get_normalizer(LANGS[code]["tok"])
    return n


def tok_indic(text, code, strict=False):
    """IndicTrans2's evaluation protocol for Indic targets: IndicNormalizer.normalize(line.strip()) and then indicnlp
    trivial_tokenize (scripts/preprocess_translate.py of AI4Bharat/IndicTrans2); light_tok fallback unless strict."""
    try:
        from indicnlp.tokenize import indic_tokenize
        return " ".join(indic_tokenize.trivial_tokenize(_indic_normalizer(code).normalize(text.strip()), LANGS[code]["tok"]))
    except ImportError:
        if strict:
            raise
        return light_tok(text)
'''


In [ ]:
# U04 - helper source part 4/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

def prep_for_metric(texts, tgt_code, strict=False):
    """Protocol: Indic targets are tokenised with tok_indic (hyp AND ref); English is left untouched."""
    if tgt_code == "eng_Latn":
        return list(texts)
    return [tok_indic(t, tgt_code, strict) for t in texts]


def score_chrf(hyps, refs, tgt_code):
    """Corpus chrF++ (word_order=2) with the Indic tokenisation protocol; returns a float."""
    from sacrebleu.metrics import CHRF
    return CHRF(word_order=2).corpus_score(prep_for_metric(hyps, tgt_code), [prep_for_metric(refs, tgt_code)]).score


_CHAR_LOOP = re.compile(r"(.{2,12}?)\1{4,}")


def has_loop(text, n=4, reps=4):
    """True if a word n-gram repeats >= reps times, or a 2-12 character unit repeats >= 5 times in a row (sub-word loops such as
    'ababababab'); only letters and combining marks are considered for the second test (digits / punctuation are ignored)."""
    if _CHAR_LOOP.search("".join(ch for ch in text if _cat1(ch) in "LM")):
        return True
    w = text.split()
    counts = {}
    for i in range(len(w) - n + 1):
        g = tuple(w[i:i + n])
        counts[g] = counts.get(g, 0) + 1
        if counts[g] >= reps:
            return True
    return False


def has_leak(text):
    return any(m in text for m in LEAK_MARKERS)


def failure_flags(hyp, hyp_raw, src, stop, chrf_fn=None):
    """Failure counters of section 8, all on the scored hypothesis.  `stop` is how decoding ended: 'eos' (clean), 'newline' (the model
    started a new line instead of emitting EOS -> counted as `no_eos`, a diagnostic of EOS learning), 'cap' (hit its length cap ->
    `unfinished`)."""
    copy = (hyp.strip() == src.strip()) or bool(chrf_fn and hyp and chrf_fn(hyp, src) >= 80.0)
    return {"empty": not hyp.strip(), "loop": has_loop(hyp), "leak": has_leak(hyp),
            "no_eos": stop == "newline", "unfinished": stop == "cap", "bad_bytes": "\ufffd" in hyp_raw, "source_copy": bool(copy)}


# ==== PART 4: files, logging, timing, disk
def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)


def atomic_save(path, obj):
    """Write via temp file + os.replace.  obj: callable(tmp_path) | bytes | str | JSON-serialisable."""
    d = os.path.dirname(path) or "."
    os.makedirs(d, exist_ok=True)
    tmp = f"{path}.tmp{os.getpid()}"
    try:
        if callable(obj):
            obj(tmp)
        elif isinstance(obj, bytes):
            with open(tmp, "wb") as f:
                f.write(obj)
        elif isinstance(obj, str):
            with open(tmp, "w", encoding="utf-8") as f:
                f.write(obj)
        else:
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump(obj, f, ensure_ascii=False, indent=1, default=_json_default)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)
    return path


class JsonlLogger:
    def __init__(self, path):
        self.path = path
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def log(self, **kw):
        kw.setdefault("ts", time.time())
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(kw, ensure_ascii=False, default=_json_default) + "\n")

    def read(self):
        if not os.path.exists(self.path):
            return []
        with open(self.path, encoding="utf-8") as f:
            return [json.loads(x) for x in f if x.strip()]


class Timer:
    def __init__(self, start=None):
        self.t0 = time.time() if start is None else start

    def elapsed(self):
        return time.time() - self.t0

    def hours(self):
        return self.elapsed() / 3600.0


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()
'''


In [ ]:
# U05 - helper source part 5/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

def free_gb(path):
    return shutil.disk_usage(path).free / 1e9


def dir_gb(path):
    tot = 0
    for root, _, files in os.walk(path):
        for fn in files:
            try:
                tot += os.path.getsize(os.path.join(root, fn))
            except OSError:
                pass
    return tot / 1e9


def check_disk(path, need_gb, label=""):
    f = free_gb(path)
    if f < need_gb:
        raise RuntimeError(f"Not enough disk for {label or path}: need {need_gb:.1f} GB, free {f:.1f} GB.")
    return f


# ==== PART 5: batched greedy generation (shared by training dev-eval, workers and the Translator)
def newline_token_ids(tok):
    """Ids of every vocabulary entry that contains a line break (byte-level BPE spells LF as U+010A and CR as U+010D).
    The scored hypothesis is the text before the first line break, so decoding may stop there (see generate_batch)."""
    ids = getattr(tok, "_p1_nl_ids", None)
    if ids is None:
        marks = (chr(0x010A), chr(0x010D))
        ids = sorted(i for t, i in tok.get_vocab().items() if any(m in t for m in marks))
        try:
            tok._p1_nl_ids = ids
        except Exception:
            pass
    return ids


def cut_at_stop(toks, eos, nl_ids):
    """Greedy continuation -> (kept_tokens, stop_reason).  stop_reason: 'eos' (the model emitted <|endoftext|>), 'newline' (it started a new
    line instead of emitting EOS: the text after the break is never scored), 'cap' (neither within the length cap).
    A newline token is kept (its text before the break belongs to the hypothesis); EOS is dropped."""
    nl = nl_ids if isinstance(nl_ids, (set, frozenset)) else set(nl_ids)
    for j, t in enumerate(toks):
        if t == eos:
            return toks[:j], "eos"
        if t in nl:
            return toks[:j + 1], "newline"
    return list(toks), "cap"


def generate_batch(model, tok, items, batch_size=32, max_len=512):
    """items: list of (src_text, src_code, tgt_code).  Left-padded greedy decoding, longest prompts first,
    results returned in INPUT order as dicts: hyp_raw, hyp, n_prompt_tok, n_gen_tok, stop ('eos' | 'newline' | 'cap'),
    finished_by_eos (stop == 'eos'), seconds.  Each row stops at EOS (<|endoftext|>), at the first line break (the hypothesis is the text
    before it, so nothing scored changes - it only avoids decoding a continuation that is thrown away) or at its per-example cap."""
    import torch
    eos = tok.convert_tokens_to_ids(EOS_TOKEN)
    nl_ids = newline_token_ids(tok)
    nl_set = set(nl_ids)
    dev = next(model.parameters()).device
    prompts = [build_prompt(LANGS[a]["name"], LANGS[b]["name"], s) for s, a, b in items]
    p_ids = tok(prompts, add_special_tokens=False)["input_ids"]
    s_len = [len(x) for x in tok([s for s, _, _ in items], add_special_tokens=False)["input_ids"]]
    caps = []
    for (s, a, b), p, sl in zip(items, p_ids, s_len):
        mult = 8.0 if a == "eng_Latn" else 0.5
        caps.append(max(0, min(max_len - len(p), int(mult * sl + 32))))
    order = sorted(range(len(items)), key=lambda i: -len(p_ids[i]))
    was_training, prev_cache = model.training, model.config.use_cache
    model.eval()
    model.config.use_cache = True
    done = []
    try:
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            live = [i for i in idx if caps[i] > 0]
            res = {}
            for i in idx:
                if caps[i] <= 0:  # prompt already fills the window: nothing can be generated
                    res[i] = ([], "cap", 0.0)
            if live:
                L = max(len(p_ids[i]) for i in live)
                ids = torch.full((len(live), L), eos, dtype=torch.long)
                att = torch.zeros((len(live), L), dtype=torch.long)
                for r, i in enumerate(live):
                    ids[r, L - len(p_ids[i]):] = torch.tensor(p_ids[i])
                    att[r, L - len(p_ids[i]):] = 1
                t0 = time.time()
                with torch.no_grad(), torch.autocast(dev.type, dtype=torch.float16, enabled=(dev.type == "cuda")):
                    out = model.generate(input_ids=ids.to(dev), attention_mask=att.to(dev),
                                         max_new_tokens=max(caps[i] for i in live), do_sample=False, num_beams=1,
                                         temperature=None, top_p=None, top_k=None, eos_token_id=[eos] + nl_ids,
                                         pad_token_id=eos, use_cache=True)
                if dev.type == "cuda":
                    torch.cuda.synchronize(dev)
                dt = (time.time() - t0) / len(live)
                new = out[:, L:].tolist()
                for r, i in enumerate(live):
                    gen, stop = cut_at_stop(new[r][:caps[i]], eos, nl_set)
                    res[i] = (gen, stop, dt)
            for i in idx:
                gen, stop, dt = res[i]
                raw = tok.decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=False)
                done.append({"hyp_raw": raw, "hyp": postprocess_hyp(raw), "n_prompt_tok": len(p_ids[i]),
                             "n_gen_tok": len(gen), "stop": stop, "finished_by_eos": stop == "eos", "seconds": dt})
    finally:
        model.config.use_cache = prev_cache
        if was_training:
            model.train()
    return restore_order(done, order)
'''


In [ ]:
# U06 - helper source part 6/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

DEMO_FIXTURES = {
    "eng_Latn": "The government has announced a new scheme to provide free health insurance to all families.",
    "hin_Deva": "सरकार ने सभी परिवारों को मुफ्त स्वास्थ्य बीमा देने के लिए एक नई योजना की घोषणा की है।",
    "guj_Gujr": "સરકારે તમામ પરિવારોને મફત આરોગ્ય વીમો આપવા માટે એક નવી યોજનાની જાહેરાત કરી છે.",
    "mar_Deva": "सरकारने सर्व कुटुंबांना मोफत आरोग्य विमा देण्यासाठी एक नवीन योजना जाहीर केली आहे.",
}
DEMO_FIXTURES_2 = {
    "eng_Latn": "Please bring your Aadhaar card and two photographs when you come to the office tomorrow.",
    "hin_Deva": "कृपया कल कार्यालय आते समय अपना आधार कार्ड और दो फोटो साथ लाएँ।",
    "guj_Gujr": "કૃપા કરીને આવતીકાલે ઓફિસ આવો ત્યારે તમારું આધાર કાર્ડ અને બે ફોટા સાથે લાવો.",
    "mar_Deva": "कृपया उद्या कार्यालयात येताना आपले आधार कार्ड आणि दोन छायाचित्रे सोबत आणा.",
}


# ==== PART 6: unit tests on SYNTHETIC data (no model, no GPU)
class _FakeTok:
    """Character-level stand-in for a tokenizer (ids = code points)."""
    def __call__(self, texts, add_special_tokens=False):
        assert add_special_tokens is False
        return {"input_ids": [[ord(c) for c in t] for t in texts]}


def _t_text():
    zwj, zwnj = "क्\u200dष", "क्\u200cत"
    assert "\u200d" in norm_text(zwj) and "\u200c" in norm_text(zwnj)
    assert norm_text("\ufeffa\u00a0b\u00adc\x07d\t e\n") == "a bcd e" and norm_text(None) == ""
    EN, HI = DEMO_FIXTURES["eng_Latn"], DEMO_FIXTURES["hin_Deva"]
    GU, MR = DEMO_FIXTURES["guj_Gujr"], DEMO_FIXTURES["mar_Deva"]
    assert script_ratio(GU, "guj_Gujr") > 0.98 and script_ratio(GU, "hin_Deva") < 0.02
    assert script_ratio(HI, "hin_Deva") > 0.98 and script_ratio(HI, "guj_Gujr") < 0.02
    assert script_ratio(MR, "mar_Deva") > 0.98 and script_ratio(EN, "eng_Latn") > 0.98
    assert script_ratio(EN, "hin_Deva") < 0.02 and script_ratio("123 ,.", "eng_Latn") == 0.0
    assert script_ratio("क्\u200dष", "hin_Deva") == 1.0  # ZWJ ignored, marks counted
    assert numerals("२०२४ and ૨૦૨૪ and 2024") == ["2024"] * 3
    assert numerals("1,00,000 and 3.5, 7") == ["100000", "3.5", "7"]
    assert numeral_check("In 2024 we sold 5 items", "२०२४ में ५ सामान") and not numeral_check("In 2024", "abc")
    assert numeral_check("no numbers here", "कोई संख्या नहीं")
    assert to_native_digits("2024", "guj_Gujr") == "૨૦૨૪" and to_ascii_digits("૨૦૨૪ ५") == "2024 5"
    assert digit_counts(["a1b", "२३"]) == (1, 2)
    assert dedup_key("Hello, World!") == dedup_key("hello  world") and "्" in dedup_key("कार्ड")
    assert dedup_key("७ cats") == dedup_key("7 cats")
    g1 = word_ngrams("One two three, four five six seven eight nine.", 8)
    assert len(g1) == 2 and word_ngrams("one two three four five six seven", 8) == set()
    assert word_ngrams("ONE two three four five six seven eight", 8) <= g1
    assert has_noise("see https://x.org/a") and has_noise("mail a.b@c.com") and has_noise("a <b>bold</b>")
    assert has_noise("call 98765 43210 now") and has_noise("९८७६५४३२१०१") and not has_noise("In 2019 2020 we grew.")
    assert not has_noise(EN) and not has_noise(HI)
    assert sym_digit_frac("abc") == 0.0 and sym_digit_frac("12 34") == 1.0
    assert len_bucket(8) == 0 and len_bucket(9) == 1 and len_bucket(36) == 4
    return "norm_text (ZWJ/ZWNJ kept), script_ratio, numerals/digits, dedup_key, word_ngrams, noise regexes, buckets"


def _t_prompt_and_loss():
    import torch
    import torch.nn.functional as F
    assert build_prompt("English", "Hindi", "Hi.") == "Translate English to Hindi.\nEnglish: Hi.\nHindi:"
    ids, p = encode_example(_FakeTok(), "English", "Hindi", "Hi.", "नमस्ते।", eos_id=7)
    prompt = build_prompt("English", "Hindi", "Hi.")
    assert p == len(prompt) and ids[:p] == [ord(c) for c in prompt] and ids[p:] == [ord(c) for c in " नमस्ते।"] + [7]
    EOS, T = 9, 8  # pad id == EOS id on purpose
    a, b = [1, 2, 3, 4, EOS], [1, 2, 3, 5, 6, 7, 8, EOS]
    ids_t = torch.tensor([a + [EOS] * (T - len(a)), b])
    lens, plens = [len(a), len(b)], [2, 3]
    m = target_position_mask(plens, lens, T)
    assert m.sum(1).tolist() == [3, 5] == [lens[i] - plens[i] for i in range(2)]
    assert m[0].tolist() == [False, True, True, True, False, False, False, False]
    assert m[1].tolist() == [False, False, True, True, True, True, True, False]
    hid = torch.randn(2, T, 4)
    h, lab = gather_targets(hid, ids_t, m)
    assert lab.tolist() == [3, 4, EOS, 5, 6, 7, 8, EOS] and h.shape == (8, 4)
    assert torch.equal(h[0], hid[0, 1]) and torch.equal(h[3], hid[1, 2])
    assert target_position_mask(2, 5, 8).tolist() == m[0].tolist()
    torch.manual_seed(0)
    hh = torch.randn(53, 8, requires_grad=True)
    W, y = torch.randn(30, 8), torch.randint(0, 30, (53,))
    full = F.cross_entropy(hh @ W.t(), y, reduction="sum")
    g_full, = torch.autograd.grad(full, hh)
    ch = chunked_ce(hh, y, W, chunk=7)
    g_ch, = torch.autograd.grad(ch, hh)
    assert torch.allclose(full, ch, atol=1e-4) and torch.allclose(g_full, g_ch, atol=1e-5)
    assert chunked_ce(hh, y, W, chunk=7, reduction="none").shape == (53,) and float(chunked_ce(hh[:0], y[:0], W)) == 0.0
    return "build_prompt/encode_example, target_position_mask (last label EOS even when pad==EOS), chunked_ce value+grad"


def _t_accumulation():
    """The training objective is invariant to HOW a global batch is split into micro-batches and into loss chunks: the gradient of
    sum-CE / (total target tokens of the global batch) is the same for one pass, uneven splits and one-example micro-batches."""
    import torch
    g, B, T, H, V = torch.Generator().manual_seed(1), 10, 7, 6, 30
    hid = torch.randn(B, T, H, generator=g, dtype=torch.float64)
    ids = torch.randint(0, V, (B, T), generator=g)
    pl, L = torch.randint(1, 4, (B,), generator=g), torch.randint(5, 8, (B,), generator=g)
    W = torch.randn(V, H, generator=g, dtype=torch.float64, requires_grad=True)
    denom = float((L - pl).sum())

    def grad_of(groups, chunk):
        W.grad = None
        for grp in groups:
            m = target_position_mask(pl[grp], L[grp], T)
            h, lab = gather_targets(hid[grp], ids[grp], m)
            (chunked_ce(h, lab, W, chunk=chunk, reduction="sum") / denom).backward()
        return W.grad.clone()

    ref = grad_of([list(range(B))], 10_000)
    for groups, chunk in (([[0, 1, 2, 3], [4, 5, 6, 7, 8, 9]], 3), ([[i] for i in range(B)], 1), ([[9, 3], [0], [1, 2, 4, 5, 6, 7, 8]], 5)):
        assert torch.allclose(grad_of(groups, chunk), ref, atol=1e-6), "gradient depends on the micro-batch split / loss chunk"
    return "gradient accumulation invariant to micro-batch split and loss chunk"
'''


In [ ]:
# U07 - helper source part 7/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

def _t_batching():
    import numpy as np
    L = np.random.RandomState(0).randint(20, 400, 64).tolist()
    groups = group_by_budget(L, 3000)
    assert sorted(i for g in groups for i in g) == list(range(64))
    assert all(len(g) == 1 or max(L[i] for i in g) * len(g) <= 3000 for g in groups) and group_by_budget([600], 500) == [[0]]
    for S_, W_ in ((3200, 128), (3900, 156)):
        lr = lambda s: lr_schedule(s, S_, W_, 3e-5, 3e-6)  # noqa: E731
        assert abs(lr(W_ - 1) - 3e-5) < 1e-15 and abs(lr(W_) - 3e-5) < 1e-9 and abs(lr(S_ - 1) - 3e-6) < 1e-15
        assert all(3e-6 - 1e-15 <= lr(s) <= 3e-5 + 1e-15 for s in range(0, S_))
        assert all(lr(s) >= lr(s + 1) - 1e-15 for s in range(W_ - 1, S_ - 1)) and all(lr(s) <= lr(s + 1) + 1e-15 for s in range(0, W_ - 1))
        assert abs(lr(W_ + (S_ - W_) // 2) - 1.65e-5) < 5e-7  # cosine mid-point = (peak + min) / 2
    items, order = list("abcdef"), [3, 0, 5, 1, 4, 2]
    assert restore_order([items[i] for i in order], order) == items
    try:
        restore_order([1, 2], [0, 0])
        raise RuntimeError("restore_order accepted a non-permutation")
    except AssertionError as e:
        assert "permutation" in str(e)
    al = waterfill({"a": 10, "b": 3, "c": 50}, 25, cap=15)
    assert sum(al.values()) == 25 and al["b"] == 3 and max(al.values()) <= 15
    assert sum(waterfill({"a": 10, "b": 3, "c": 50}, 99, cap=15).values()) == 28 and sum(waterfill({"a": 2, "b": 1}, 10).values()) == 3
    lr_ = largest_remainder([1, 1, 1], 10)
    assert sum(lr_) == 10 and sorted(lr_) == [3, 3, 4]
    return "group_by_budget, lr_schedule (step-based: warm-up / peak / monotone cosine / exact minimum / clamp), restore_order, waterfill, largest_remainder"


def _t_outputs():
    HI = DEMO_FIXTURES["hin_Deva"]
    assert postprocess_hyp("  नमस्ते \nEnglish: x") == "नमस्ते" and postprocess_hyp("a\r\nb") == "a"
    assert has_loop("a b c d " * 4) and not has_loop("a b c d e f g h") and has_leak("x\nEnglish: y")
    assert has_loop("abababababab") and has_loop("xx " + "है" * 6) and not has_loop("1000000000000, ........")
    assert not any(has_loop(v) for v in list(DEMO_FIXTURES.values()) + list(DEMO_FIXTURES_2.values()))
    f = failure_flags("a b", "a b", "a b", "cap", chrf_fn=lambda h, s: 100.0)
    assert f["unfinished"] and not f["no_eos"] and f["source_copy"] and not f["empty"] and not f["leak"]
    assert failure_flags("a b", "a b", "s", "newline")["no_eos"] and not failure_flags("a b", "a b", "s", "eos")["no_eos"]
    assert failure_flags("English: x", "English: x", "s", "eos")["leak"]
    assert failure_flags("", "", "src", "eos")["empty"] and failure_flags("x\ufffd", "x\ufffd", "s", "eos")["bad_bytes"]
    assert cut_at_stop([5, 6, 7, 2, 9], 2, {7}) == ([5, 6, 7], "newline") and cut_at_stop([5, 6, 2, 7], 2, {7}) == ([5, 6], "eos")
    assert cut_at_stop([5, 6], 2, {7}) == ([5, 6], "cap") and cut_at_stop([], 2, {7}) == ([], "cap") and cut_at_stop([2], 2, {7}) == ([], "eos")
    assert light_tok("नमस्ते, दुनिया।") == "नमस्ते , दुनिया ।" and prep_for_metric(["a b"], "eng_Latn") == ["a b"]
    try:
        import indicnlp  # noqa: F401
        assert "।" in tok_indic(HI, "hin_Deva", strict=True).split()
    except ImportError:
        pass
    assert split_direction("eng_Latn-guj_Gujr") == ("eng_Latn", "guj_Gujr") and len(DIRECTIONS) == 6
    return "postprocess_hyp, failure flags, tokenisation helpers"


def _t_files():
    import tempfile
    import numpy as np
    with tempfile.TemporaryDirectory() as d:
        p1 = atomic_save(os.path.join(d, "x", "a.json"), {"k": [1, 2]})
        atomic_save(os.path.join(d, "x", "b.txt"), "hello")

        def _w(tmp):
            with open(tmp, "wb") as fh:
                fh.write(b"zz")
        atomic_save(os.path.join(d, "x", "c.bin"), _w)
        assert json.load(open(p1))["k"] == [1, 2] and not [x for x in os.listdir(os.path.join(d, "x")) if ".tmp" in x]
        assert sha256_file(os.path.join(d, "x", "b.txt")) == hashlib.sha256(b"hello").hexdigest()
        lg = JsonlLogger(os.path.join(d, "l.jsonl"))
        lg.log(step=np.int64(3), loss=np.float32(0.5))
        lg.log(step=4)
        assert [r["step"] for r in lg.read()] == [3, 4] and check_disk(d, 0.0) > 0 and dir_gb(d) >= 0
        assert 0.99 < Timer(start=time.time() - 3600).hours() < 1.01
    return "atomic_save, JsonlLogger, sha256_file, Timer, disk helpers"


def run_unit_tests(verbose=True):
    tests = [_t_text, _t_prompt_and_loss, _t_accumulation, _t_batching, _t_outputs, _t_files]
    try:
        import sacrebleu  # noqa: F401
        tests.append(run_metric_tests)
    except ImportError:
        if verbose:
            print("  --  metric tests skipped (sacrebleu not installed here; they run in the metrics step)")
    for t in tests:
        what = t()
        if verbose:
            print(f"  ok  {what}")
    if verbose:
        print(f"{len(tests)} unit-test groups passed")
    return len(tests)


def run_metric_tests():
    """Metric-side tests on synthetic data (need sacrebleu; also run in the metrics step)."""
    from sacrebleu.metrics import CHRF

    class _FakeLid:
        class _F:
            def predict(self, t, k, thr, mode):
                return [(0.9, "__label__hin_Deva")]
        f = _F()

        def predict(self, texts, k=1):
            raise ValueError("numpy 2 copy error")

    assert lid_top1(_FakeLid(), ["a", "b"]) == [("hin_Deva", 0.9)] * 2
    base = ["the cat sat on the mat", "hello world again", "good morning to you all", "this is a test sentence", "nothing here"]
    hyps = [base[i % 5] + (" extra" if i % 3 == 0 else "") for i in range(40)]
    refs = [base[(i + (i % 4 == 0)) % 5] for i in range(40)]
    m = CHRF(word_order=2)
    ci, method = bootstrap_chrf(m, hyps, refs, seed=42, n_boot=200)
    sc = m.corpus_score(hyps, [refs]).score
    assert method.startswith("sufficient_statistics") and ci[0] < ci[1] and ci[0] - 1e-9 <= sc <= ci[1] + 1e-9, (ci, sc, method)
    assert ci == bootstrap_chrf(m, hyps, refs, seed=42, n_boot=200)[0]
    assert abs(score_chrf(hyps, refs, "eng_Latn") - sc) < 1e-9
    assert abs(_marker_frac(["यह है", "और वह"], HI_MARKERS) - 1.0) < 1e-9 and _marker_frac(["यह वह"], HI_MARKERS) == 0.0
    return "lid_top1 NumPy-2 fallback, bootstrap_chrf (aggregate == corpus_score, seeded CI brackets the score), marker words"
'''


In [ ]:
# U08 - helper source part 8/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

# ==== PART 7: subprocess entry points - generation workers
def lid_top1(model, texts):
    """[(label_without_prefix, prob)] for a fastText LID model; falls back to model.f.predict when
    model.predict breaks under NumPy 2 [CHECK]."""
    texts = [t.replace("\n", " ") or "." for t in texts]
    try:
        labels, probs = model.predict(texts, k=1)
        return [(l[0].replace("__label__", ""), float(p[0])) for l, p in zip(labels, probs)]
    except ValueError:
        out = []
        for t in texts:
            r = model.f.predict(t + "\n", 1, 0.0, "strict")
            out.append((r[0][1].replace("__label__", ""), float(r[0][0])) if r else ("", 0.0))
        return out


def _write_lines(path, rows):
    atomic_save(path, "\n".join(json.dumps(x, ensure_ascii=False) for x in rows) + "\n")


def _load_for_generation(model_dir):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.float16).to("cuda:0").eval()
    return model, tok


def gen_worker(cfg):
    """cfg: jobs_path (json list of {direction,set,rows:[{id,src,ref}]}), model_dir, out_dir, batch_size.
    Runs on the single GPU made visible via CUDA_VISIBLE_DEVICES; one samples_<direction>_<set>.jsonl per job."""
    model, tok = _load_for_generation(cfg["model_dir"])
    jobs = json.load(open(cfg["jobs_path"], encoding="utf-8"))
    os.makedirs(cfg["out_dir"], exist_ok=True)
    for job in jobs:
        sc, tc = split_direction(job["direction"])
        t0 = time.time()
        res = generate_batch(model, tok, [(r["src"], sc, tc) for r in job["rows"]], batch_size=cfg["batch_size"], max_len=512)
        lines = [{"id": r["id"], "direction": job["direction"], "set": job["set"], "src": r["src"], "ref": r["ref"], **o}
                 for r, o in zip(job["rows"], res)]
        _write_lines(os.path.join(cfg["out_dir"], f"samples_{job['direction']}_{job['set']}.jsonl"), lines)
        print(f"[worker] {job['direction']} {job['set']}: {len(lines)} sentences in {time.time() - t0:.0f}s", flush=True)


# ==== PART 8: metrics (sacrebleu 2.3.1 / indicnlp / fasttext)
SCRIPT_MIN = {"eng_Latn": 0.90, "guj_Gujr": 0.85, "hin_Deva": 0.85, "mar_Deva": 0.85}
MR_MARKERS = ["आहे", "आहेत", "आणि", "करण्यासाठी"]
HI_MARKERS = ["है", "हैं", "और", "के लिए", "नहीं"]
EVAL_SETS = ("in22_gen", "in22_conv")


def _boot_one(args):
    hyps, refs, idx = args
    from sacrebleu.metrics import CHRF
    return CHRF(word_order=2).corpus_score([hyps[i] for i in idx], [[refs[i] for i in idx]]).score


def bootstrap_chrf(metric, hyps, refs, seed=42, n_boot=1000, n_fallback=300):
    """95% bootstrap CI of corpus chrF++ from per-segment sufficient statistics (private sacrebleu API, verified here
    against corpus_score; otherwise 300 resamples with multiprocessing).  The SAME resampled indices serve hyps and refs."""
    import numpy as np
    full = metric.corpus_score(hyps, [refs]).score
    n, rng = len(hyps), np.random.RandomState(seed)
    try:
        stats = np.array(metric._extract_corpus_statistics(hyps, [refs]), dtype=np.float64)
        agg = metric._aggregate_and_compute(stats.tolist())
        if abs(agg.score - full) < 1e-6:
            sc = [metric._aggregate_and_compute([stats[rng.randint(0, n, n)].sum(0).tolist()]).score for _ in range(n_boot)]
            return [float(np.percentile(sc, 2.5)), float(np.percentile(sc, 97.5))], f"sufficient_statistics_{n_boot}"
    except Exception as e:  # private API changed
        print(f"[bootstrap] sufficient-statistics path unavailable: {type(e).__name__}: {e}", flush=True)
    jobs = [(hyps, refs, rng.randint(0, n, n)) for _ in range(n_fallback)]
    try:
        import multiprocessing as mp
        with mp.Pool(min(4, os.cpu_count() or 1)) as pool:
            sc = pool.map(_boot_one, jobs)
    except Exception:
        sc = [_boot_one(j) for j in jobs]
    return [float(np.percentile(sc, 2.5)), float(np.percentile(sc, 97.5))], f"corpus_score_{n_fallback}"


def _load_samples(eval_dir, d, s):
    p = os.path.join(eval_dir, f"samples_{d}_{s}.jsonl")
    if not os.path.exists(p):
        return None
    with open(p, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]


def _marker_frac(tok_hyps, markers):
    pad = [" " + t + " " for t in tok_hyps]
    return sum(any(" " + m + " " in t for m in markers) for t in pad) / max(1, len(pad))


def generation_stats(rows):
    """Decoding diagnostics of one (direction, set): generated-token statistics and how decoding ended - the evidence needed to tell a
    length-cap problem ('cap'), weak EOS learning ('newline': a new line instead of EOS) and verbose output (token / character ratios)."""
    import numpy as np
    g = np.array([r["n_gen_tok"] for r in rows], dtype=float)
    stops = [r.get("stop") or ("eos" if r["finished_by_eos"] else "cap") for r in rows]
    n = max(1, len(rows))
    return {"n_gen_tok_mean": float(g.mean()), "n_gen_tok_median": float(np.median(g)), "n_gen_tok_p95": float(np.percentile(g, 95)),
            "n_gen_tok_max": int(g.max()), "gen_per_prompt_tok_median": float(np.median(g / np.maximum(1.0, [r["n_prompt_tok"] for r in rows]))),
            "len_ratio_median": float(np.median([len(r["hyp"]) / max(1, len(r["ref"])) for r in rows])),
            "stop_eos": stops.count("eos") / n, "stop_newline": stops.count("newline") / n,
            "stop_cap": stops.count("cap") / n, "hyp_chars_mean": float(np.mean([len(r["hyp"]) for r in rows])),
            "ref_chars_mean": float(np.mean([len(r["ref"]) for r in rows]))}
'''


In [ ]:
# U09 - helper source part 9/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

def analyze_set(rows, src_c, tgt_c, lid_model, strict_tok=True, seed=42, boot_n=1000, boot_fallback=300):
    from sacrebleu.metrics import CHRF, BLEU
    hyps, refs = [r["hyp"] for r in rows], [r["ref"] for r in rows]
    hyp_t, ref_t = prep_for_metric(hyps, tgt_c, strict_tok), prep_for_metric(refs, tgt_c, strict_tok)
    chrf = CHRF(word_order=2)
    score = chrf.corpus_score(hyp_t, [ref_t]).score
    ci, method = bootstrap_chrf(chrf, hyp_t, ref_t, seed, boot_n, boot_fallback)
    bleu = BLEU(tokenize="none") if tgt_c != "eng_Latn" else BLEU()
    out = {"n": len(rows), "chrf_pp": score, "ci95": ci, "ci_method": method, "chrf_signature": str(chrf.get_signature()),
           "chrf_plain": CHRF(word_order=2).corpus_score(hyps, [refs]).score, "bleu": bleu.corpus_score(hyp_t, [ref_t]).score,
           "bleu_signature": str(bleu.get_signature())}
    sent = CHRF(word_order=2)
    segs, num_tot, num_keep, acr_tot, acr_keep = [], 0, 0, 0, 0
    labels = lid_top1(lid_model, hyps) if lid_model is not None else [("", 0.0)] * len(rows)
    for r, ht, rt, (lab, _) in zip(rows, hyp_t, ref_t, labels):
        h = r["hyp"]
        fl = failure_flags(h, r["hyp_raw"], r["src"], r.get("stop") or ("eos" if r["finished_by_eos"] else "cap"),
                           chrf_fn=lambda a, b: sent.sentence_score(a, [b]).score)
        ratio = len(h) / max(1, len(r["ref"]))
        sn = numerals(r["src"])
        hn = set(numerals(h))
        num_tot += len(sn)
        num_keep += sum(x in hn for x in sn)
        acr = re.findall(r"\b[A-Z]{2,}\b", r["src"]) if src_c == "eng_Latn" else re.findall(r"[A-Za-z][A-Za-z0-9]+", r["src"])
        acr_tot += len(acr)
        acr_keep += sum(a in h for a in acr)
        segs.append({"id": r["id"], "set": r["set"], "bucket": len_bucket(len(r["src"].split())), "chrf_sent": sent.sentence_score(ht, [rt]).score,
                     "script_ok": script_ratio(h, tgt_c) >= SCRIPT_MIN[tgt_c], "len_ratio": ratio, "len_ok": 0.7 <= ratio <= 1.4,
                     "lid": lab, "lid_ok": lab == tgt_c, "flags": fl, "hyp_t": ht})
    out["_segs"], out["_num"], out["_acr"], out["_pair"] = segs, (num_keep, num_tot), (acr_keep, acr_tot), (hyp_t, ref_t)
    out["generation"] = generation_stats(rows)
    return out
'''


In [ ]:
# U10 - helper source part 10/10 (kept as a string; no .py file is ever written)
UTILS_SRC += r'''

def metrics_base(cfg):
    """cfg: eval_dir, out_dir, lid_path (optional), seed, boot_n, boot_fallback."""
    lid = None
    if cfg.get("lid_path"):
        try:
            import fasttext
            lid = fasttext.load_model(cfg["lid_path"])
        except Exception as e:
            print(f"[metrics] LID unavailable: {type(e).__name__}: {e}", flush=True)
    res, seg_out, confusion = {}, [], {}
    pooled = {"in22_gen": ([], []), "in22_conv": ([], []), "all": ([], [])}  # every direction pooled (tokenised hyp / ref lists)
    for d in DIRECTIONS:
        sc, tc = split_direction(d)
        entry, all_segs, nums, acrs = {}, [], [0, 0], [0, 0]
        for s in EVAL_SETS:
            try:
                rows = _load_samples(cfg["eval_dir"], d, s)
                if rows is None:
                    entry[s] = {"error": "samples file missing"}
                    continue
                n_exp = (cfg.get("expected") or {}).get(f"{d}|{s}")
                if n_exp is not None and [r["id"] for r in rows] != [f"{s}-{i:04d}" for i in range(n_exp)]:
                    entry[s] = {"error": f"sample file has {len(rows)} rows / unexpected ids, the benchmark has exactly {n_exp}: refusing to score a partial or foreign file"}
                    continue
                a = analyze_set(rows, sc, tc, lid, True, cfg.get("seed", 42), cfg.get("boot_n", 1000), cfg.get("boot_fallback", 300))
                for seg in a.pop("_segs"):
                    seg.update(direction=d)
                    all_segs.append(seg)
                nums[0] += a["_num"][0]; nums[1] += a["_num"][1]; acrs[0] += a["_acr"][0]; acrs[1] += a["_acr"][1]
                a.pop("_num"); a.pop("_acr")
                for k in (s, "all"):
                    pooled[k][0].extend(a["_pair"][0])
                    pooled[k][1].extend(a["_pair"][1])
                a.pop("_pair")
                entry[s] = a
            except Exception as e:  # one failing metric/set must not stop the others
                entry[s] = {"error": f"{type(e).__name__}: {e}"}
        if all_segs:
            n = len(all_segs)
            fl = {k: sum(x["flags"][k] for x in all_segs) for k in all_segs[0]["flags"]}
            entry.update(script_acc=sum(x["script_ok"] for x in all_segs) / n, len_ratio_ok=sum(x["len_ok"] for x in all_segs) / n,
                         lid_lang_acc=(sum(x["lid_ok"] for x in all_segs) / n) if lid is not None else None,
                         failures={**fl, **{f"{k}_rate": v / n for k, v in fl.items()}, "n": n},
                         numeral_preservation=(nums[0] / nums[1]) if nums[1] else None,
                         acronym_copy_rate=(acrs[0] / acrs[1]) if acrs[1] else None)
            if lid is not None:
                conf = {}
                for x in all_segs:
                    conf[x["lid"]] = conf.get(x["lid"], 0) + 1
                confusion[d] = conf
            if tc in ("hin_Deva", "mar_Deva"):
                th = [x["hyp_t"] for x in all_segs]
                entry["marker_words"] = {"hindi_marker_frac": _marker_frac(th, HI_MARKERS), "marathi_marker_frac": _marker_frac(th, MR_MARKERS)}
            by_b = {}
            for b in range(len(LEN_EDGES)):
                idx = [i for i, x in enumerate(all_segs) if x["bucket"] == b]
                by_b[str(b)] = {"n": len(idx), "mean_sentence_chrf": (sum(all_segs[i]["chrf_sent"] for i in idx) / len(idx)) if idx else None}
            entry["by_source_length_bucket"] = by_b
        res[d] = entry
        for x in all_segs:
            x.pop("hyp_t")
            seg_out.append(x)
        print(f"[metrics] {d}: " + ", ".join(f"{s}: {entry[s].get('chrf_pp', 'n/a')}" for s in EVAL_SETS), flush=True)
    from sacrebleu.metrics import CHRF
    overall = {}  # "overall" = all six directions pooled into ONE corpus (micro average) per set and over both sets; BLEU overall = mean of the six direction scores
    for k, (hs, rs) in pooled.items():
        if not hs:
            continue
        sets = EVAL_SETS if k == "all" else (k,)
        chrf = CHRF(word_order=2)
        ci, method = bootstrap_chrf(chrf, hs, rs, cfg.get("seed", 42), cfg.get("boot_n", 1000), cfg.get("boot_fallback", 300))
        bl = [res[d][s]["bleu"] for d in DIRECTIONS for s in sets if isinstance(res[d].get(s), dict) and "bleu" in res[d][s]]
        cf = [res[d][s]["chrf_pp"] for d in DIRECTIONS for s in sets if isinstance(res[d].get(s), dict) and "chrf_pp" in res[d][s]]
        overall[k] = {"n": len(hs), "chrf_pp_pooled": chrf.corpus_score(hs, [rs]).score, "ci95_pooled": ci, "ci_method": method,
                      "chrf_pp_macro_mean": sum(cf) / len(cf) if cf else None, "bleu_macro_mean": sum(bl) / len(bl) if bl else None}
        print(f"[metrics] overall {k}: chrF++ pooled {overall[k]['chrf_pp_pooled']:.2f} CI {ci}", flush=True)
    os.makedirs(cfg["out_dir"], exist_ok=True)
    atomic_save(os.path.join(cfg["out_dir"], "overall.json"), overall)
    atomic_save(os.path.join(cfg["out_dir"], "results_base.json"), res)
    _write_lines(os.path.join(cfg["out_dir"], "segments.jsonl"), seg_out)
    atomic_save(os.path.join(cfg["out_dir"], "lid_confusion.json"), confusion)


def _cli(argv):
    """usage (source on stdin): python - <worker|metrics-base> <config.json>"""
    cmds = {"worker": gen_worker, "metrics-base": metrics_base}
    if len(argv) != 3 or argv[1] not in cmds:
        raise SystemExit(_cli.__doc__)
    cmds[argv[1]](json.load(open(argv[2], encoding="utf-8")))


if __name__ == "__main__":
    import sys
    _cli(sys.argv)
'''


**Changed (helpers):** (1) `lr_schedule` is a pure function of the optimizer step; (2) `generate_batch` additionally stops a row at the first line break (the scored hypothesis is the text before it, so scores are unchanged) and records HOW decoding ended — `stop` = eos / newline / cap — and `generation_stats` summarises generated tokens per direction and set; (3) `failure_flags` reports `no_eos` (a new line instead of EOS: a direct diagnostic of EOS learning) and `unfinished` (= hit the length cap) on the scored text; (4) `tok_indic` normalises with IndicNormalizer before tokenising, as AI4Bharat's evaluation does; (5) metrics add pooled overall chrF++ with CI. **Why:** the old `unfinished` / `leak` columns mixed harmless post-newline continuation with real failures, and the cause of English->Indic failures could not be told apart. **Effect:** unchanged hypotheses and translation scores up to the Indic normalisation; sharper diagnostics, less wasted decoding.

In [ ]:
# C05e - Build the helper module from UTILS_SRC, run unit tests, import helpers
import types
p1_utils = types.ModuleType("p1_utils")
exec(compile(UTILS_SRC, "p1_utils", "exec"), p1_utils.__dict__)
sys.modules["p1_utils"] = p1_utils
n_groups = p1_utils.run_unit_tests(verbose=True)
from p1_utils import (LANGS, NAME2CODE, INDIC, DIRECTIONS, EOS_TOKEN, DEMO_FIXTURES, DEMO_FIXTURES_2, split_direction,
                      norm_text, script_ratio, to_ascii_digits, to_native_digits, digit_counts, numerals, numeral_check,
                      dedup_key, word_ngrams, has_noise, sym_digit_frac, len_bucket, build_prompt, encode_examples,
                      target_position_mask, gather_targets, chunked_ce, group_by_budget, lr_schedule,
                      waterfill, largest_remainder, score_chrf, generate_batch, lid_top1, atomic_save, JsonlLogger, Timer, sha256_file,
                      free_gb, dir_gb, check_disk)
assert DIRECTIONS == CONFIG["DIRECTIONS"] and INDIC == CONFIG["INDIC"], "CONFIG and p1_utils disagree on directions"
SESSION = Timer(start=T0)
print(f"{n_groups} unit-test groups passed; helpers imported from the in-memory p1_utils module")


In [ ]:
# C05f - Subprocess helpers, configuration signature
RUN_LOG = JsonlLogger(os.path.join(TRAIN_DIR, "run_log.jsonl"))


def run_utils(python_exe, cmd, cfg_path, timeout=None, env=None):
    """Run a helper command (worker | metrics-base) in a subprocess: the helper SOURCE is passed on
    stdin (`python - cmd cfg`), so no .py file exists anywhere.  Returns the CompletedProcess (stdout/stderr captured)."""
    return subprocess.run([python_exe, "-", cmd, cfg_path], input=UTILS_SRC, encoding="utf-8", errors="replace", capture_output=True,
                          timeout=timeout, env={**os.environ, "PYTHONIOENCODING": "utf-8", **(env or {})}, cwd=WORK)


def popen_utils(python_exe, cmd, cfg_path, log_file, env=None):
    """Start a long-running helper command (a generation worker) with the helper source on stdin."""
    p = subprocess.Popen([python_exe, "-", cmd, cfg_path], stdin=subprocess.PIPE, stdout=log_file, stderr=subprocess.STDOUT,
                         encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8", **(env or {})}, cwd=WORK)
    p.stdin.write(UTILS_SRC)
    p.stdin.close()
    return p


_PREP_KEYS = ["MODEL_ID", "DATA_ID", "INDIC", "POOL", "N", "GLOBAL_BATCH", "HUMAN_CAP", "LABSE_MIN", "LID_MIN_PROB", "EN_WORDS", "MAX_TOKENS",
              "DEV_PAIRS_PER_DIR", "FLORES_DEV_PER_DIR", "HUMAN_SUBSET_REGEX", "MINED_PARSE_CAP", "SOURCE_CAP", "RESERVE_PER_LANG",
              "USE_IN22_LENGTH_MATCH", "USE_IN22_DIGIT_AUDIT", "MINED_WITHOUT_LABSE", "ON_LABSE_STARVATION", "SEED"]
_TRAIN_KEYS = ["STEPS", "WARMUP_STEPS", "TRAINED_LAYERS", "LR_PEAK", "LR_MIN", "BETAS", "EPS", "WD", "CLIP", "SCALER_INIT", "MICRO_BATCH_TOKENS",
               "LOSS_CHUNK", "OOM_LADDER", "DEV_LOSS_EVERY", "GEN_EVAL_STEPS", "NONFINITE_REPLAYS"]  # (the wall cap does not define the experiment)
_SIG_KEYS = _PREP_KEYS + _TRAIN_KEYS


def config_sig():
    """Hash of every data / training setting + the pinned model / BPCC revisions + the code identity (CODE_VERSION, helper-source hash).
    A local checkpoint is resumed ONLY if its recorded signature equals this one."""
    payload = {"cfg": {k: CONFIG[k] for k in sorted(_SIG_KEYS)}, "revs": S["revs"],
               "code": [CONFIG["CODE_VERSION"], hashlib.sha256(UTILS_SRC.encode("utf-8")).hexdigest()[:12]]}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


def prep_sig():
    """Identity of the PREPARED DATA: the data settings, the pinned model / BPCC revisions (incl. the BPCC file manifest) and the code identity.
    The processed data in /kaggle/working/data is reused (instead of re-parsing hours of BPCC text) only when its manifest carries this signature
    AND every file still matches its sha256."""
    payload = {"cfg": {k: CONFIG[k] for k in sorted(_PREP_KEYS)}, "revs": S["revs"],
               "code": [CONFIG["CODE_VERSION"], hashlib.sha256(UTILS_SRC.encode("utf-8")).hexdigest()[:12]]}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


def session_left_h():
    return CONFIG["SESSION_CAP_H"] - SESSION.hours()


In [ ]:
# C05i - Disk monitor, manifest helper
def disk_report(tag=""):
    used, free = dir_gb(WORK), free_gb(WORK)
    print(f"[disk{(' ' + tag) if tag else ''}] /working used {used:.1f} GB (cap ~20) | free {free:.1f} GB | raw free {free_gb(RAW):.1f} GB")
    if used > 18:
        print("WARNING: /kaggle/working above 18 GB - delete caches/checkpoints you no longer need")
    return used


def guard_working(need_gb, label):
    used = dir_gb(WORK)
    if used + need_gb > 20.0:
        raise RuntimeError(f"{label}: /kaggle/working would reach {used + need_gb:.1f} GB (> 20 GB cap)")


DATA_FILES = ["pool.parquet", "dev_loss.parquet", "flores_dev_sample.jsonl"]


def verify_data_dir():
    """Re-read the processed data files and compare them with manifest.json (sha256)."""
    with open(os.path.join(PREP_DIR, "manifest.json"), encoding="utf-8") as f:
        mf = json.load(f)
    for name, sha in mf["sha256"].items():
        assert sha256_file(os.path.join(PREP_DIR, name)) == sha, f"{name}: sha256 differs from manifest.json"
    return mf


def prep_cache_valid():
    """True when /kaggle/working/data already holds EXACTLY the data this configuration would produce (same prep signature = data settings +
    pinned revisions + code identity, same N, every file matches its sha256).  Used after a kernel restart inside the same session: the 1-2 h of
    BPCC parsing is not repeated, and the pool is byte-identical, so a checkpoint of the same run can be resumed."""
    mp = os.path.join(PREP_DIR, "manifest.json")
    if not os.path.exists(mp):
        return False
    try:
        with open(mp, encoding="utf-8") as f:
            mf = json.load(f)
        if mf.get("prep_sig") != prep_sig() or int(mf.get("n_final", -1)) != CONFIG["N"]:
            return False
        verify_data_dir()
        return True
    except Exception as e:
        print(f"[cache] the processed data is not reusable ({type(e).__name__}: {str(e)[:100]}) - preparing it again")
        return False


def time_plan(tag, done_steps=0, trained_s=0.0, pre_reserve_h=0.0):
    """THE runtime budget of the notebook (used by C16b right after preparation and by C21 / compute_wall right before training).
    Projected finish = session elapsed + reserve for the steps still to come before training (model download / build / validation; 0 once
    they are done) + training time still needed + post-training reserve (export, generation, metrics).  Training may only start if the
    time left after ALL reserves (pre-training, post-training, safety) covers the training still to be done.  Returns a dict and prints it."""
    el = SESSION.hours()
    n_left = CONFIG["N"] - done_steps * CONFIG["GLOBAL_BATCH"]
    need = n_left / CONFIG["EXPECTED_EX_S"] / 3600 * CONFIG["OVERHEAD_FACTOR"]
    avail = min(CONFIG["TRAIN_WALL_H"] - trained_s / 3600,
                CONFIG["SESSION_CAP_H"] - el - pre_reserve_h - CONFIG["POST_TRAIN_H"] - CONFIG["SAFETY_H"])
    finish = el + pre_reserve_h + need + CONFIG["POST_TRAIN_H"]
    plan = {"tag": tag, "session_elapsed_h": round(el, 3), "pre_training_reserve_h": pre_reserve_h, "examples_left": n_left, "training_need_h": round(need, 3),
            "post_training_reserve_h": CONFIG["POST_TRAIN_H"], "safety_reserve_h": CONFIG["SAFETY_H"], "training_window_h": round(avail, 3),
            "projected_finish_h": round(finish, 3), "margin_h": round(CONFIG["SESSION_CAP_H"] - finish, 3), "ok": avail >= need}
    print(f"TIME BUDGET ({tag}): session elapsed {el:.2f} h | pre-training reserve {pre_reserve_h:.2f} h | estimated training {need:.2f} h "
          f"({n_left:,} examples at {CONFIG['EXPECTED_EX_S']} ex/s x {CONFIG['OVERHEAD_FACTOR']}) | post-training reserve {CONFIG['POST_TRAIN_H']:.2f} h | "
          f"safety reserve {CONFIG['SAFETY_H']:.2f} h | projected finish {finish:.2f} h of {CONFIG['SESSION_CAP_H']:.0f} h | remaining margin "
          f"{plan['margin_h']:.2f} h (safety reserve {CONFIG['SAFETY_H']:.2f} h) | training window {avail:.2f} h -> {'OK' if plan['ok'] else 'NOT ENOUGH TIME'}")
    return plan


def require_time(plan):
    if not plan["ok"]:
        raise RuntimeError(f"Not enough time left in this {CONFIG['SESSION_CAP_H']:.0f} h session: the {plan['examples_left']:,} remaining examples need about "
                           f"{plan['training_need_h']:.2f} h of training but only {plan['training_window_h']:.2f} h are available after the pre-training, post-training "
                           f"and safety reserves (projected finish {plan['projected_finish_h']:.2f} h). Training is NOT started, because the run would end INCOMPLETE. "
                           "Re-run the notebook in a fresh session (preparation must finish earlier) or lower CONFIG['N'] / ['STEPS'] consistently - the experiment "
                           "never trains on a silently smaller set.")


def timed(label, fn):
    """Run one preparation step and log its duration (the prep timings go into manifest.json)."""
    t0 = time.time()
    out = fn()
    S.setdefault("timing", {})[label] = round(time.time() - t0, 1)
    print(f"[{label}] {(time.time() - t0) / 60:.1f} min | session elapsed {SESSION.hours():.2f} h")
    return out


def mf_set(key, value):
    """Record a value in the data manifest (saved locally at once, finalised in C16b)."""
    S["manifest"][key] = value
    atomic_save(os.path.join(PREP_DIR, "manifest.partial.json"), S["manifest"])


## Part 1 — DATA PREPARATION (BPCC download, filtering, decontamination, exact N)

Builds `pool.parquet` (216,000 disjoint pairs, 36,000 per direction; every complete block of 600 consecutive examples contains exactly 100 per direction, only the block the first N cuts through can deviate slightly), `dev_loss.parquet`, `flores_dev_sample.jsonl` and `manifest.json` (sha256 of every file) into `/kaggle/working/data/`. Everything runs in this one Kaggle GPU session; parsing and filtering the BPCC text is CPU work that costs session time, which the training wall-clock cap (C21) accounts for. Nothing is written to Hugging Face or to any Kaggle Dataset.

**BPCC layout (verified from the live Hub listing).** The text is *not* in `<subset>/<pair>/train.<lang>` files (those exist only inside the 14.8 GB `additional/BPCC.zip`, which is never downloaded). It is in `<subset>/<lang>.tsv` (human: `bpcc-seed-*`, `nllb_seed`, `daily`, `wiki`, `ilci`, `massive`; mined: `samanantar_v2`, `samanantar_v0.3_filtered`, `nllb_filtered`, `comparable`), with side files such as `<subset>/eng_Latn-<lang>/labse_scores.txt`. Because the files are gated their columns could not be inspected in advance, so the TSV layout is **detected at run time** and printed. Two layouts are understood: **Format A** (a header-less or headed file whose English and Indic columns are found by script, with an optional score column) and **Format B** (explicit `src_lang | tgt_lang | src | tgt` columns, e.g. `bpcc-seed-v2`): for B the declared languages are verified against the requested one, `src`/`tgt` are used directly (swapped if declared the other way round), quoted fields — including quoted line breaks — are parsed with the `csv` module, an optional score column is taken by name, and the script-ratio detector is never consulted. Files are fetched **one at a time**, randomly sampled (mined: `MINED_PARSE_CAP` rows per file) and deleted, so peak disk is about the largest single file (~7 GB).

**Selection rules, stated precisely.** The human share is capped at `HUMAN_CAP` of each language's selection overall; the per-subset cap `SOURCE_CAP` is applied **inside each English-length bucket** (so a subset can exceed that share globally; the resulting global share is printed and recorded, and the cap is relaxed when too few subsets exist). Each pair is used in exactly one direction (the two directions of a language get disjoint halves).

Everything in this part is **count-agnostic**: no number that comes from a file is hard-coded; counts after every filter step go into the manifest. By default IN22 text is used here **only for decontamination** (and its length / digit statistics are *printed for information*); the training length distribution and digit convention come from BPCC alone (`USE_IN22_LENGTH_MATCH` / `USE_IN22_DIGIT_AUDIT` = `False`; set them to `True` to restore the original spec's IN22-informed variants, which the model card then discloses). IN22 is never used for training, dev or any other setting.

In [ ]:
# C06 - Inventory of BPCC sources (real layout: <subset>/<lang>.tsv)
HUMAN_RE = re.compile(CONFIG["HUMAN_SUBSET_REGEX"], re.I)
SCORE_RE = re.compile(r"labse|laser|score|sim", re.I)
SKIP_STEM_RE = re.compile(r"dev|test|valid", re.I)


def load_tok():
    """Tokenizer of the PINNED base-model commit (tokenizer files only; weights are fetched in C19)."""
    if "tok" not in S:
        from transformers import AutoTokenizer
        fetch_model(tokenizer_only=True)
        S["tok"] = AutoTokenizer.from_pretrained(MODEL_SRC_DIR)
    return S["tok"]


def build_sources():
    """Every (subset, language) text source of the three pairs.  Two layouts are understood:
    (B) the layout of the Hub repo as listed:   <subset>/<lang>.tsv  (+ <subset>/<pair>/labse_scores.txt for mined data)
    (A) the layout described in the spec/README: <subset>/<pair>/<stem>.eng_Latn + <stem>.<lang> (+ a score file)."""
    inv, srcs = list_bpcc(), []
    for lang in INDIC:
        pair, aligned = f"eng_Latn-{lang}", {}
        for path, size in inv.items():
            parts = path.split("/")
            if parts[0] == "additional" or path.lower().endswith(".zip"):
                continue
            name, folder = parts[-1], "/".join(parts[:-1])
            if name == f"{lang}.tsv":
                score = f"{folder}/{pair}/labse_scores.txt"
                srcs.append({"kind": "tsv", "pair": pair, "lang": lang, "subset": folder or "<root>", "files": [path], "size": size,
                             "score": score if score in inv else None, "score_size": inv.get(score, 0)})
            elif len(parts) >= 2 and parts[-2] == pair:
                sub, (stem, ext) = "/".join(parts[:-2]) or "<root>", os.path.splitext(name)
                if ext[1:] in ("eng_Latn", lang) and not SKIP_STEM_RE.search(stem):
                    aligned.setdefault((sub, stem), {})[ext[1:]] = (path, size)
        for (sub, stem), d in sorted(aligned.items()):
            if len(d) == 2:
                sc = [p for p in inv if os.path.dirname(p) == f"{sub}/{pair}" and SCORE_RE.search(os.path.basename(p)) and os.path.basename(p).startswith(stem + ".")]
                srcs.append({"kind": "aligned", "pair": pair, "lang": lang, "subset": sub, "files": [d["eng_Latn"][0], d[lang][0]],
                             "size": d["eng_Latn"][1] + d[lang][1], "score": sc[0] if sc else None, "score_size": inv.get(sc[0], 0) if sc else 0})
    for s in srcs:
        s["human"] = bool(HUMAN_RE.search(s["subset"]))
    return srcs


def c06():
    srcs = build_sources()
    miss = [l for l in INDIC if not any(s["lang"] == l for s in srcs)]
    if miss:
        raise RuntimeError(f"No BPCC text source found for {miss}. Top-level entries: {sorted({f.split('/')[0] for f in list_bpcc()})[:40]}")
    tab = pd.DataFrame([{"subset": s["subset"], "lang": s["lang"], "kind": s["kind"], "human": s["human"],
                         "gb": round(s["size"] / 1e9, 3), "score_gb": round(s["score_size"] / 1e9, 3)} for s in srcs])
    print(tab.sort_values(["human", "subset", "lang"], ascending=[False, True, True]).to_string(index=False))
    if not tab.human.any():
        print("WARNING: HUMAN_SUBSET_REGEX matched no subset - everything is treated as MINED. Check CONFIG['HUMAN_SUBSET_REGEX'].")
    zips = {f: s for f, s in list_bpcc().items() if f.lower().endswith(".zip") and re.search(r"in22|flores", f, re.I)}
    assert any(re.search("in22", f, re.I) for f in zips) and any(re.search("flores", f, re.I) for f in zips), \
        f"IN22 / FLORES-22 zips not found under additional/ (found: {list(zips)})"
    print("benchmark zips:", {f: round(s / 1e6, 1) for f, s in zips.items()}, "MB  (additional/BPCC.zip is NEVER downloaded)")
    biggest = max(s["size"] + s["score_size"] for s in srcs) / 1e9
    check_disk(RAW, biggest * 1.15 + 2.0, "largest single BPCC file (files are processed one at a time and deleted)")
    S["sources"], S["bench_zips"] = srcs, zips
    mf_set("inventory", {"sources": tab.to_dict("records"), "zips": zips, "total_text_gb": round(sum(s["size"] for s in srcs) / 1e9, 2),
                         "largest_file_gb": round(biggest, 2)})
    print(f"{len(srcs)} sources, {sum(s['size'] for s in srcs) / 1e9:.1f} GB of text in total (streamed one file at a time; peak disk ~{biggest:.1f} GB)")


timed("C06", c06)


In [ ]:
# C07a - Benchmark readers (IN22-Gen, IN22-Conv, FLORES-22 dev)
import zipfile

BENCH_SETS = ("in22_gen", "in22_conv", "flores_dev")


def _read_lines(path):
    with open(path, encoding="utf-8", newline="\n") as f:
        return [l.rstrip("\r\n") for l in f]


LANG_RE = re.compile(r"(?<![A-Za-z])[a-z]{3}_[A-Z][a-z]{3}(?![A-Za-z])")
PAIR_DIR_RE = re.compile(r"^([a-z]{3}_[A-Z][a-z]{3})-([a-z]{3}_[A-Z][a-z]{3})$")
SKIP_EXT = (".json", ".jsonl", ".csv", ".tsv", ".parquet", ".zip")
BENCH_LABEL = {"in22_gen": "IN22-Gen", "in22_conv": "IN22-Conv", "flores_dev": "FLORES-22 dev"}


def _bench_set_of(rel, kind):
    """kind='flores': every file is the FLORES-22 dev set.  kind='in22': the set comes from the path (gen / conv); anything else is ignored."""
    if kind == "flores":
        return "flores_dev"
    low = rel.lower()
    if re.search(r"(?<![a-z])conv(?![a-z])", low):
        return "in22_conv"
    if re.search(r"(?<![a-z])gen(?![a-z])", low):
        return "in22_gen"
    return None


In [ ]:
# C07a2 - Benchmark readers: pair-aware zip sniffer
def _sniff_bench(root, kind):
    """PAIR-AWARE reader of one extracted benchmark zip.  Returns (sets, listing, n_other_pair_dirs) where
    sets[set_name] = {"pairs": {frozenset({lang_a, lang_b}): {"dir", "stem", "lines": {lang_a: [...], lang_b: [...]}}},
                      "langs": {lang: [(relative_path, lines), ...]}}.
    * A directory named LANG1-LANG2 (e.g. all/eng_Latn-guj_Gujr/) is a language-pair directory: its two files (dev.eng_Latn, dev.guj_Gujr)
      are read TOGETHER and must have the same number of lines (no truncation, ever).  Pair directories that involve a language outside
      the four project languages (e.g. eng_Latn-pan_Guru) are not used.
    * Files outside pair directories (e.g. all/eng_Latn/dev.eng_Latn) are plain per-language files.
    Files that belong to different pair directories are NEVER compared with or merged into each other."""
    sets, listing, groups, other_pairs = {}, [], {}, set()
    for dp, _, fns in os.walk(root):
        pm = PAIR_DIR_RE.match(os.path.basename(dp))
        reldir = os.path.relpath(dp, root).replace(os.sep, "/")
        for fn in sorted(fns):
            rel = fn if reldir == "." else reldir + "/" + fn
            listing.append(rel)
            codes = LANG_RE.findall(fn)
            s = _bench_set_of(rel, kind)
            if fn.lower().endswith(SKIP_EXT) or not codes or s is None:
                continue
            code = codes[-1]
            if pm:
                if not (pm.group(1) in LANGS and pm.group(2) in LANGS):
                    other_pairs.add(reldir)
                    continue
                if code not in pm.groups():
                    continue
                stem = fn[:fn.rindex(code)].rstrip("._-") or "data"
                g = groups.setdefault((s, reldir, stem), {"pair": pm.groups(), "files": {}})
                g["files"][code] = (rel, _read_lines(os.path.join(dp, fn)))
            elif code in LANGS:
                sets.setdefault(s, {"pairs": {}, "langs": {}})["langs"].setdefault(code, []).append((rel, _read_lines(os.path.join(dp, fn))))
    for (s, reldir, stem), g in sorted(groups.items()):
        a, b = g["pair"]
        absent = [c for c in (a, b) if c not in g["files"]]
        if absent:
            raise RuntimeError(f"{BENCH_LABEL[s]} pair directory {reldir} (file stem '{stem}') has no file for {absent}")
        (fa, la), (fb, lb) = g["files"][a], g["files"][b]
        if len(la) != len(lb):
            raise RuntimeError(f"{BENCH_LABEL[s]} alignment error:" + chr(10) + f"pair={reldir}" + chr(10) + f"source={fa}" + chr(10) + f"target={fb}" + chr(10) +
                               f"source_lines={len(la)}" + chr(10) + f"target_lines={len(lb)}")
        pairs = sets.setdefault(s, {"pairs": {}, "langs": {}})["pairs"]
        key, rec = frozenset((a, b)), {"dir": reldir, "stem": stem, "lines": {a: la, b: lb}}
        if key in pairs and pairs[key]["lines"] != rec["lines"]:
            raise RuntimeError(f"{BENCH_LABEL[s]}: two directories hold the pair {a}/{b} with different content ({pairs[key]['dir']}/{pairs[key]['stem']} "
                               f"vs {reldir}/{stem}); cannot tell which one is the benchmark")
        pairs.setdefault(key, rec)
    return sets, listing, len(other_pairs)


def _merge_sets(dst, src):
    for s, rec in src.items():
        d = dst.setdefault(s, {"pairs": {}, "langs": {}})
        d["pairs"].update(rec["pairs"])
        for c, copies in rec["langs"].items():
            d["langs"].setdefault(c, []).extend(copies)


def _one_copy(copies, s, code):
    """The single plain per-language file of a set (several plain copies must be identical, otherwise the choice would be a guess)."""
    if not copies:
        return None
    if any(c[1] != copies[0][1] for c in copies[1:]):
        raise RuntimeError(f"{BENCH_LABEL[s]}: several DIFFERENT plain files for {code}: {[c[0] for c in copies]}")
    return copies[0]


In [ ]:
# C07a3 - Benchmark readers: build bench[set][direction] and the alignment checks
def _build_bench(sf):
    """-> bench[set][direction] = [(src, ref), ...], source description per direction, all discovered lines per language (for decontamination),
    and the directions that could not be built.  A direction uses its own language-pair directory when there is one; otherwise the two plain
    per-language files of the set (n-way parallel by construction; equal length is enforced)."""
    bench, where, lines, missing = {}, {}, {}, {}
    for s in BENCH_SETS:
        rec = sf.get(s, {"pairs": {}, "langs": {}})
        bench[s], where[s] = {}, {}
        for d in DIRECTIONS:
            a, b = split_direction(d)
            pr = rec["pairs"].get(frozenset((a, b)))
            if pr:
                bench[s][d] = list(zip(pr["lines"][a], pr["lines"][b]))
                where[s][d] = f"pair directory {pr['dir']} ({pr['stem']})"
                continue
            ca, cb = _one_copy(rec["langs"].get(a), s, a), _one_copy(rec["langs"].get(b), s, b)
            if ca is None or cb is None:
                missing.setdefault(s, []).append(d)
                continue
            if len(ca[1]) != len(cb[1]):
                raise RuntimeError(f"{BENCH_LABEL[s]}: the language files {ca[0]} ({len(ca[1])} lines) and {cb[0]} ({len(cb[1])} lines) are not aligned")
            bench[s][d] = list(zip(ca[1], cb[1]))
            where[s][d] = f"language files {ca[0]} + {cb[0]}"
        lines[s], seen = {c: [] for c in LANGS}, {c: set() for c in LANGS}
        sources = [(c, ls) for pr in rec["pairs"].values() for c, ls in pr["lines"].items()] + [(c, ls) for c, cp in rec["langs"].items() for _, ls in cp]
        for c, ls in sources:  # every benchmark sentence we saw (decontamination wants all of them), each once
            for l in ls:
                if l not in seen[c]:
                    seen[c].add(l)
                    lines[s][c].append(l)
    return bench, where, lines, missing


def _check_alignment(bench, s):
    """Statistical alignment checks inside every English<->Indic pair (the files carry no ids, so alignment can only be made very likely)."""
    label, eng, indic = BENCH_LABEL[s], {}, {}
    for code in INDIC:
        e, x = [a for a, _ in bench[s][f"eng_Latn-{code}"]], [b for _, b in bench[s][f"eng_Latn-{code}"]]
        eng[code], indic[code] = e, x
        we, wx = np.array([len(l.split()) for l in e], dtype=float), np.array([len(l.split()) for l in x], dtype=float)
        hits = [numerals(a) == numerals(b) for a, b in zip(e, x) if numerals(a)]
        rate = sum(hits) / max(1, len(hits))
        corr = float(np.corrcoef(we, wx)[0, 1])  # sentence-length correlation: ~0 for shuffled / shifted lines
        shifted = max(float(np.corrcoef(we[k:], wx[:len(wx) - k])[0, 1]) for k in (1, 2))  # off-by-one / two shifts
        shifted = max(shifted, max(float(np.corrcoef(we[:len(wx) - k], wx[k:])[0, 1]) for k in (1, 2)))
        print(f"{label:<14} n={len(e):<5} eng vs {code}: numeral agreement {rate:.2f} ({len(hits)} sentences) | length correlation {corr:.2f}")
        if rate < 0.5 or corr < 0.5 or corr - shifted < 0.15:
            raise RuntimeError(f"{label}: pair-local alignment check failed for eng_Latn-{code} (numeral agreement {rate:.2f}, length correlation {corr:.2f}, "
                               f"best shifted-by-1-2 correlation {shifted:.2f}). The benchmark files carry no ids, so row alignment can only be "
                               "checked statistically, never proven.")
    if all(eng[c] == eng[INDIC[0]] for c in INDIC):  # one shared English side -> the three Indic sides are row-aligned with each other too
        for code in INDIC:
            wc = np.array([len(l.split()) for l in indic[code]], dtype=float)
            cross = min(float(np.corrcoef(wc, np.array([len(l.split()) for l in indic[o]], dtype=float))[0, 1]) for o in INDIC if o != code)
            if cross < 0.5:
                raise RuntimeError(f"{label}: Indic sides of the three pairs share one English side but {code} is not row-aligned with the others (length correlation {cross:.2f})")
        print(f"{label:<14} the English side is identical in the three project pairs (n-way)")
    else:  # NOT corruption: pair-specific directories may hold different sentence sets / orders - each pair was validated on its own above
        e0 = set(eng[INDIC[0]])
        print(f"NOTE {label}: the English sides of the three project pairs differ (sizes {[len(eng[c]) for c in INDIC]}, sentences shared by all three "
              f"{len(e0.intersection(*[set(eng[c]) for c in INDIC[1:]]))}); each pair is validated on its own (cross-pair English mismatch, not a pair-local alignment failure)")


In [ ]:
# C07a4 - Benchmark readers: Hugging Face datasets fallback for IN22
def load_in22_hf():
    """Fallback: the HF datasets ai4bharat/IN22-Gen and IN22-Conv (auto-gated: accept their terms with the HF_TOKEN account).
    Per the dataset cards the language fields are `sentence_<code>` (not `context_<code>`); tries the n-way `all` config,
    the parquet `default` config, then the per-pair configs.  Anything unexpected raises with the columns that were seen."""
    from datasets import load_dataset
    out = {}
    for s, repo in (("in22_gen", "ai4bharat/IN22-Gen"), ("in22_conv", "ai4bharat/IN22-Conv")):
        per, seen = {}, []
        for cfg in ["all", None] + CONFIG["PAIRS"]:
            try:
                ds = load_dataset(repo, cfg, token=HF_TOKEN) if cfg else load_dataset(repo, token=HF_TOKEN)
            except Exception as e:
                seen.append(f"{cfg}: {type(e).__name__}")
                continue
            if "test" in ds:  # the benchmarks are test sets; never guess among several splits
                d = ds["test"]
            elif len(ds) == 1:
                d = ds[list(ds.keys())[0]]
            else:
                seen.append(f"{cfg}: splits {list(ds.keys())} - no 'test' split")
                continue
            seen.append(f"{cfg}: columns {d.column_names[:12]}")
            for code in LANGS:
                col = next((c for c in (f"sentence_{code}", code) if c in d.column_names), None)
                if col is None:
                    continue
                lines = [str(x) for x in d[col]]
                if code in per and per[code] != lines:
                    raise RuntimeError(f"{repo}: {code} differs between configs - not n-way parallel")
                per[code] = lines
            if set(per) == set(LANGS):
                break
        if set(per) != set(LANGS):
            raise RuntimeError(f"{repo}: could not assemble the 4 languages (have {sorted(per)}). Tried: {seen}")
        out[s] = per
    return out


In [ ]:
# C07b - Benchmarks: download, pair-aware parsing, build bench[set][direction]
def ensure_bench():
    """Downloads/extracts the benchmark zips (once), parses them PAIR-AWARE, validates alignment, and returns
    bench[set][direction] = [(src, ref), ...] for set in (in22_gen, in22_conv, flores_dev) and the six project directions.
    Also sets S['bench_lines'][set][lang] (every benchmark sentence seen, for decontamination) and S['bench_src'] (where each direction came from)."""
    if "bench" in S:
        return S["bench"]
    zips = S.get("bench_zips") or {f: s for f, s in list_bpcc().items()
                                    if f.lower().endswith(".zip") and re.search(r"in22|flores", f, re.I)}
    for fn in zips:
        out = os.path.join(BENCH_DIR, os.path.splitext(os.path.basename(fn))[0])
        if not (os.path.isdir(out) and os.listdir(out)):
            p = hf_hub_download(CONFIG["DATA_ID"], fn, repo_type="dataset", revision=S["revs"]["data"], local_dir=BPCC_DIR)
            with zipfile.ZipFile(p) as z:
                z.extractall(out)
    sf, listing = {}, []
    for fn in zips:
        kind = "flores" if re.search("flores", fn, re.I) else "in22"
        found, files, n_other = _sniff_bench(os.path.join(BENCH_DIR, os.path.splitext(os.path.basename(fn))[0]), kind)
        _merge_sets(sf, found)
        listing += files
        used = sorted({r["dir"] for rec in found.values() for r in rec["pairs"].values()})
        plain = sorted({c for rec in found.values() for c in rec["langs"]})
        print(f"{os.path.basename(fn)}: {len(files)} files | project pair directories {used} | other pair directories ignored: {n_other} | "
              f"plain per-language files for {plain}")
    bench, where, lines, missing = _build_bench(sf)
    in22_missing = [s for s in missing if s.startswith("in22")]
    if in22_missing:
        print("IN22 not (completely) parsed from the zip; trying the HF dataset fallback ...", {s: missing[s] for s in in22_missing})
        try:
            hf = load_in22_hf()
        except Exception as e:
            raise RuntimeError(f"Cannot parse IN22 (zip layout unknown, HF fallback failed: {e}). Files seen: {listing[:40]}") from None
        for s in in22_missing:
            sf[s] = {"pairs": {}, "langs": {c: [(f"HF dataset ({s})", ls)] for c, ls in hf[s].items()}}
        bench, where, lines, missing = _build_bench(sf)
    if missing:
        raise RuntimeError("Required benchmark pair(s) not found: " + "; ".join(f"{BENCH_LABEL[s]}: {ds}" for s, ds in missing.items()) + chr(10) +
                           f"Discovered relevant pair directories: {sorted({r['dir'] for rec in sf.values() for r in rec['pairs'].values()})}" + chr(10) +
                           f"Discovered language codes: {sorted({c for rec in sf.values() for c in rec['langs']} | {c for rec in sf.values() for k in rec['pairs'] for c in k})}" +
                           chr(10) + f"Files seen (first 40): {listing[:40]}")
    for s in BENCH_SETS:
        _check_alignment(bench, s)
    S["bench_lines"], S["bench"], S["bench_src"] = lines, bench, where
    return bench


def c07():
    bench = ensure_bench()
    mf_set("bench_counts", {s: {d: len(v) for d, v in dd.items()} for s, dd in bench.items()})
    mf_set("bench_sources", S["bench_src"])
    short = lambda d: "->".join(x.split("_")[0] for x in split_direction(d))  # noqa: E731
    print("BENCHMARK SUMMARY" + chr(10) + "-" * 60 + chr(10) + f"{'set':<11}{'direction':<12}{'n':>6}   source" + chr(10) + "-" * 60)
    for s in BENCH_SETS:
        for d in DIRECTIONS:
            print(f"{s:<11}{short(d):<12}{len(bench[s][d]):>6}   {S['bench_src'][s][d]}")
    print("-" * 60)
    for s in BENCH_SETS:
        print(f"{BENCH_LABEL[s]} pair mapping:")
        for d in CONFIG["PAIRS"]:
            print(f"  {d} <- {S['bench_src'][s][d]}")


timed("C07", c07)


In [ ]:
# C08a - BPCC readers (sniff TSV layout at run time, sample, delete the file)
import csv
import itertools

csv.field_size_limit(1 << 22)
_HDR = re.compile(r"^(eng|en|english|src|source|tgt|target|id|idx|index|score|labse|laser|domain|[a-z]{3}_[A-Z][a-z]{3})$", re.I)
_LANGCODE = re.compile(r"^[a-z]{3}_[A-Z][a-z]{3}$")


def _count_lines(path):
    n, last = 0, b"\n"
    with open(path, "rb") as f:
        while True:
            b = f.read(1 << 24)
            if not b:
                break
            n += b.count(b"\n")
            last = b[-1:]
    return n + (0 if last == b"\n" else 1)


def _iter_lines(path):
    with open(path, encoding="utf-8", errors="replace", newline="\n") as f:
        for l in f:
            yield l.rstrip("\r\n")


def _csv_records(path):
    """Quote-aware TSV records (a quoted field may contain tabs and newlines).  A record the csv module cannot parse is yielded as None
    (the caller counts it as malformed); the reader carries on with the next record."""
    errors = 0
    with open(path, encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f, delimiter="\t", quotechar='"')
        while True:
            try:
                yield next(reader)
                errors = 0
            except StopIteration:
                return
            except csv.Error:
                errors += 1
                if errors > 1000:
                    raise RuntimeError(f"{path}: more than 1000 consecutive unparsable records - the quoting of this TSV is not what this reader assumes")
                yield None


def _count_records(path):
    return sum(1 for _ in _csv_records(path))


def _score(x):
    try:
        return float(str(x).split()[0])
    except Exception:
        return float("nan")


def fetch(path, tries=3):
    """Download ONE file of the pinned BPCC revision (gated: uses the token); returns the local path."""
    for k in range(tries):
        try:
            return hf_hub_download(CONFIG["DATA_ID"], path, repo_type="dataset", revision=S["revs"]["data"], local_dir=BPCC_DIR, token=HF_TOKEN)
        except (GatedRepoError, HfHubHTTPError):
            raise
        except Exception as e:
            if k == tries - 1:
                raise
            print(f"   download retry {k + 1} for {path}: {type(e).__name__}")


def release(paths):
    for p in paths:
        try:
            os.remove(p)
        except OSError:
            pass


In [ ]:
# C08a1 - BPCC readers: TSV layout detection (explicit schema / ratio-based)
def _sniff_explicit(recs, lang, path):
    """FORMAT B - the file DECLARES its layout: columns src_lang | tgt_lang | src | tgt (+ optional score), with or without a header row.
    Returns the layout dict, or None when the file is not of this format (then the ratio-based detector of FORMAT A is used).
    Once this format is recognised the declared languages are verified and the script-ratio detector is never consulted."""
    first = [c.strip().lower() for c in recs[0]]
    names = {"src_lang", "tgt_lang", "src", "tgt"}
    if names <= set(first):
        header, col, body, ncols = True, {n: first.index(n) for n in names}, recs[1:], len(first)
    elif len(recs[0]) >= 4 and _LANGCODE.match(recs[0][0].strip()) and _LANGCODE.match(recs[0][1].strip()):
        header, col, body, ncols = False, {"src_lang": 0, "tgt_lang": 1, "src": 2, "tgt": 3}, recs, max(len(r) for r in recs)
    else:
        return None
    body = [r for r in body if len(r) == ncols]
    if len(body) < 3:
        raise RuntimeError(f"{path}: explicit src_lang/tgt_lang schema, but fewer than 3 well-formed records in the first 60: {[r[:4] for r in recs[:3]]}")
    sl, tl = {r[col["src_lang"]].strip() for r in body}, {r[col["tgt_lang"]].strip() for r in body}
    if sl == {"eng_Latn"} and tl == {lang}:
        swap = False
    elif sl == {lang} and tl == {"eng_Latn"}:
        swap = True  # declared the other way round: the English text is the tgt column
    else:
        raise RuntimeError(f"{path}: declared languages src_lang={sorted(sl)} tgt_lang={sorted(tl)} do not match eng_Latn <-> {lang}")
    sc_c = None
    if header:
        sc_c = next((c for c, nm in enumerate(first) if c not in col.values() and re.search(r"labse|laser|score|sim", nm)), None)
    en_c, in_c = (col["tgt"], col["src"]) if swap else (col["src"], col["tgt"])
    return {"header": header, "ncols": ncols, "en": en_c, "ind": in_c, "score": sc_c, "explicit": True, "swap": swap,
            "src_lang_col": col["src_lang"], "tgt_lang_col": col["tgt_lang"], "declared": ["eng_Latn" if not swap else lang, lang if not swap else "eng_Latn"]}


def sniff_tsv(path, lang):
    """Which TSV layout is this?  FORMAT B (explicit src_lang/tgt_lang/src/tgt columns, quote-aware) is tried first; otherwise FORMAT A:
    header? which column is English, which is the Indic side (by script), is there an inline 0-1 score column?"""
    recs = [r for r in itertools.islice(_csv_records(path), 60) if r is not None]
    if recs:
        ex = _sniff_explicit(recs, lang, path)
        if ex is not None:
            return ex
    rows = [l.split("\t") for l in itertools.islice(_iter_lines(path), 60)]
    if len(rows) < 6:
        raise RuntimeError(f"{path}: fewer than 6 lines - cannot sniff the TSV layout")
    widths = [len(r) for r in rows]
    ncols = max(set(widths), key=widths.count)
    header = all(len(c) <= 24 and len(c.split()) <= 2 for c in rows[0]) and any(_HDR.match(c.strip()) for c in rows[0])
    body = [r for r in rows[1 if header else 0:] if len(r) == ncols]
    if ncols < 2 or len(body) < 5:
        raise RuntimeError(f"{path}: expected >=2 tab-separated columns, first rows: {[r[:3] for r in rows[:3]]}")
    en_r = [float(np.mean([script_ratio(r[c], "eng_Latn") for r in body])) for c in range(ncols)]
    in_r = [float(np.mean([script_ratio(r[c], lang) for r in body])) for c in range(ncols)]
    en_c, in_c = int(np.argmax(en_r)), int(np.argmax(in_r))
    if en_c == in_c or en_r[en_c] < 0.6 or in_r[in_c] < 0.6:
        raise RuntimeError(f"{path}: cannot identify an English and a {lang} column (latin/script ratios {np.round(en_r, 2).tolist()} / "
                           f"{np.round(in_r, 2).tolist()}); first rows: {[[c[:30] for c in r] for r in rows[:3]]}")
    sc_c = None
    if header:  # 1) a column NAMED like a score wins
        sc_c = next((c for c, name in enumerate(rows[0]) if c not in (en_c, in_c) and re.search(r"labse|laser|score|sim", name, re.I)), None)
    if sc_c is None:  # 2) otherwise a column whose values all look like decimals in [0, 1.5] (integer ids / counts are rejected)
        sc_c = next((c for c in range(ncols) if c not in (en_c, in_c) and all("." in r[c] and -0.01 <= _score(r[c]) <= 1.5 for r in body)), None)
    return {"header": header, "ncols": ncols, "en": en_c, "ind": in_c, "score": sc_c, "explicit": False}


In [ ]:
# C08a2 - BPCC readers: TSV and aligned-file sources
def _frame(src, en, ind, lab):
    return pd.DataFrame({"pair": src["pair"], "subset": src["subset"], "is_human": src["human"], "src": en, "tgt": ind,
                         "labse": np.asarray(lab, dtype="float32")})


def read_tsv(src, rng, rep):
    p = fetch(src["files"][0])
    local = [p]
    try:
        sn = sniff_tsv(p, src["lang"])
        explicit = sn["explicit"]
        n = (_count_records(p) if explicit else _count_lines(p)) - int(sn["header"])  # explicit files: RECORDS (a quoted field may span lines)
        note, sc_it = "no score", None
        if sn["score"] is not None:
            note = f"inline score column {sn['score']}"
        elif src["score"]:
            sp = fetch(src["score"])
            local.append(sp)
            n_sc, first = _count_lines(sp), _score(next(_iter_lines(sp), ""))
            hdr = int(math.isnan(first))
            if n_sc - hdr == n:
                sc_it, note = itertools.islice(_iter_lines(sp), hdr, None), "labse_scores.txt (row-aligned)"
            else:
                note = f"labse_scores.txt IGNORED (lines {n_sc - hdr} != tsv rows {n})"
                print(f"   WARNING {src['subset']}/{src['lang']}: {note}")
        cap = None if src["human"] else CONFIG["MINED_PARSE_CAP"]
        keep = None
        if cap and n > cap:
            keep = np.zeros(n, bool)
            keep[rng.choice(n, cap, replace=False)] = True
        en, ind, lab, bad, lang_bad = [], [], [], 0, 0
        rows_it = _csv_records(p) if explicit else (l.split("\t") for l in _iter_lines(p))
        if sn["header"]:
            next(rows_it, None)
        need = max(sn["en"], sn["ind"], sn["score"] if sn["score"] is not None else 0, sn.get("src_lang_col", 0), sn.get("tgt_lang_col", 0)) + 1
        for i, r in enumerate(rows_it):
            s = next(sc_it, "") if sc_it is not None else None
            if keep is not None and (i >= n or not keep[i]):
                continue
            if r is None or len(r) < need:
                bad += 1
                continue
            if explicit and (r[sn["src_lang_col"]].strip() != sn["declared"][0] or r[sn["tgt_lang_col"]].strip() != sn["declared"][1]):
                lang_bad += 1  # a row whose declared languages differ from the file's: never used
                continue
            a, b = r[sn["en"]], r[sn["ind"]]
            if explicit:
                a, b = a.replace(chr(10), " ").replace(chr(13), " "), b.replace(chr(10), " ").replace(chr(13), " ")  # quoted fields may hold line breaks
            en.append(a)
            ind.append(b)
            lab.append(_score(r[sn["score"]]) if sn["score"] is not None else (_score(s) if s is not None else float("nan")))
        expected = int(keep.sum()) if keep is not None else n
        if explicit and expected > 1000 and len(en) < 0.90 * expected:  # e.g. a stray quote swallowing following records, or mislabelled rows
            raise RuntimeError(f"{p}: only {len(en):,} of {expected:,} records were usable ({bad} unparsable, {lang_bad} language-mismatching): the TSV is "
                               f"not parsed as intended (layout {sn}) - refusing to continue with a corrupted sample")
        print(f"   first rows ({note}, header={sn['header']}, explicit-schema={explicit}, cols en/ind/score={sn['en']}/{sn['ind']}/{sn['score']}):",
              [(a[:40], b[:30]) for a, b in zip(en[:2], ind[:2])], f"| malformed {bad}, language-mismatch {lang_bad}" if explicit else "")
        rep.append({**{k: src[k] for k in ("pair", "subset", "kind", "human")}, "status": "ok", "rows_in_file": n, "kept": len(en),
                    "malformed": bad, "language_mismatch": lang_bad, "score": note, "layout": sn})
        return _frame(src, en, ind, lab)
    finally:
        release(local)

def read_aligned(src, rng, rep):
    ps = [fetch(f) for f in src["files"]]
    local = list(ps)
    try:
        n1, n2 = _count_lines(ps[0]), _count_lines(ps[1])
        if n1 != n2:
            rep.append({**{k: src[k] for k in ("pair", "subset", "kind", "human")}, "status": f"dropped: unequal line counts {n1} vs {n2}"})
            return None
        sc_it, note = None, "no score"
        if src["score"]:
            sp = fetch(src["score"])
            local.append(sp)
            if _count_lines(sp) == n1:
                sc_it, note = _iter_lines(sp), "score file (row-aligned)"
        cap = None if src["human"] else CONFIG["MINED_PARSE_CAP"]
        keep = None
        if cap and n1 > cap:
            keep = np.zeros(n1, bool)
            keep[rng.choice(n1, cap, replace=False)] = True
        en, ind, lab = [], [], []
        for i, (a, b) in enumerate(zip(_iter_lines(ps[0]), _iter_lines(ps[1]))):
            s = next(sc_it, "") if sc_it is not None else None
            if keep is None or keep[i]:
                en.append(a)
                ind.append(b)
                lab.append(_score(s) if s is not None else float("nan"))
        rep.append({**{k: src[k] for k in ("pair", "subset", "kind", "human")}, "status": "ok", "rows_in_file": n1, "kept": len(en), "score": note})
        return _frame(src, en, ind, lab)
    finally:
        release(local)


**Changed:** before C08 the notebook checks whether `/kaggle/working/data/manifest.json` carries the current **prep signature** (data settings + pinned model / BPCC revisions + helper-code hash + `CODE_VERSION`) and N, and whether every file still matches its sha256. If so, parsing / filtering / selection (C08-C16a) are skipped with a printed message; otherwise everything is recomputed and the old files are overwritten. Every preparation step is now timed (`prep_timing_s` in the manifest). **Why:** after a kernel restart inside the same session a checkpoint can only be resumed if the pool is byte-identical, and re-running 1-2 h of preparation would also eat the training budget. **Effect:** none on a first run.

In [ ]:
# C08b - Parse BPCC into one DataFrame
def c08():
    if S.get("prep_cached"):
        return
    rng, report, chunks, held_gb = np.random.RandomState(CONFIG["SEED"]), [], [], 0.0
    for k, src in enumerate(S["sources"], 1):
        t0 = time.time()
        print(f"[{k}/{len(S['sources'])}] {src['subset']} / {src['lang']} ({src['kind']}, {src['size'] / 1e9:.2f} GB)")
        df = read_tsv(src, rng, report) if src["kind"] == "tsv" else read_aligned(src, rng, report)
        if df is not None and len(df):
            chunks.append(df)
            held_gb += float(df.memory_usage(deep=True).sum()) / 1e9  # running total: refuse BEFORE the memory spike of pd.concat
            if held_gb > 9.0:
                raise RuntimeError(f"parsed chunks already hold {held_gb:.1f} GB of RAM after {k} files and the filters keep several copies: "
                                   "lower CONFIG['MINED_PARSE_CAP']")
        print(f"   {0 if df is None else len(df):,} rows kept in {time.time() - t0:.0f}s | free raw disk {free_gb(RAW):.1f} GB")
    if not chunks:
        raise RuntimeError("no BPCC rows could be read - see the messages above (layout unknown - STOP)")
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    print(df.groupby(["pair", "subset", "is_human"]).size().to_string())
    gb = float(df.memory_usage(deep=True).sum()) / 1e9  # the later filters make 2-3 copies: refuse early instead of dying of OOM hours in
    print(f"parsed frame: {len(df):,} rows, {gb:.1f} GB in RAM (cap per mined file: MINED_PARSE_CAP={CONFIG['MINED_PARSE_CAP']:,})")
    if gb > 9.0:
        raise RuntimeError(f"parsed data needs {gb:.1f} GB of RAM and the filters keep several copies: lower CONFIG['MINED_PARSE_CAP']")
    S["df"] = df
    mf_set("parse", report)
    mf_set("rows_after_parse", {p: int((df.pair == p).sum()) for p in CONFIG["PAIRS"]})
    disk_report("after parse")


S["prep_cached"] = prep_cache_valid()
if S["prep_cached"]:
    print("[C08-C16] PROCESSED DATA REUSED: data/manifest.json carries this prep signature and every file matches its sha256 - BPCC parsing, filtering and selection are skipped")
timed("C08", c08)


In [ ]:
# C09 - Profile (histograms, ratio bounds, LaBSE, script ratio, IN22 token statistics)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
AUDIT_DIR = os.path.join(PREP_DIR, "audit")
os.makedirs(AUDIT_DIR, exist_ok=True)


def c09():
    if S.get("prep_cached"):
        return
    df, tok, bench_lines = S["df"], load_tok(), S["bench_lines"]
    df["en_w"] = df.src.str.split().str.len()
    df["in_w"] = df.tgt.str.split().str.len()
    hum = df[df.is_human] if df.is_human.any() else df
    bounds, hist_rows = {}, []
    for p in CONFIG["PAIRS"]:
        h = hum[(hum.pair == p) & (hum.en_w > 0)]
        if len(h) < 100:
            h = df[(df.pair == p) & (df.en_w > 0)]
        ratio = (h.in_w / h.en_w).values
        bounds[p] = [float(np.percentile(ratio, 1)), float(np.percentile(ratio, 99))]
        cnt, edges = np.histogram(np.clip(h.en_w.values, 0, 100), bins=np.arange(0, 105, 5))
        hist_rows += [{"pair": p, "bin_lo": int(e), "count": int(c)} for e, c in zip(edges[:-1], cnt)]
        print(f"{p}: word-ratio bounds (1st-99th pct of {'HUMAN' if df.is_human.any() else 'ALL'} data) = {bounds[p]}")
        if df[(df.pair == p)].labse.notna().any():
            q = df[(df.pair == p)].labse.quantile([.01, .1, .25, .5, .75, .9, .99]).round(3).to_dict()
            print(f"    LaBSE quantiles: {q}")
    pd.DataFrame(hist_rows).to_csv(os.path.join(AUDIT_DIR, "profile_english_word_hist.csv"), index=False)
    smp = df.sample(min(200_000, len(df)), random_state=CONFIG["SEED"])
    sr = [script_ratio(t, p.split("-")[1]) for t, p in zip(smp.tgt, smp.pair)]
    plt.figure(figsize=(6, 3.2))
    plt.hist(sr, bins=40, color="#3b6ea8")
    plt.xlabel("Indic-side script ratio (sample)")
    plt.ylabel("pairs")
    plt.tight_layout()
    plt.savefig(os.path.join(AUDIT_DIR, "profile_script_ratio.png"), dpi=120)
    plt.close()
    stats = {}
    for s in ("in22_gen", "in22_conv"):  # statistics only - IN22 never enters training or dev
        for code in LANGS:
            lines = bench_lines[s][code]
            nt = np.array([len(x) for x in tok(lines, add_special_tokens=False)["input_ids"]])
            nw = np.array([max(1, len(l.split())) for l in lines])
            stats[f"{s}/{code}"] = {"n": len(lines), "tok_mean": float(nt.mean()), "tok_p99": float(np.percentile(nt, 99)),
                                    "tok_per_word": float(nt.sum() / nw.sum())}
    print(pd.DataFrame(stats).T.round(2).to_string())
    en = bench_lines["in22_gen"]["eng_Latn"] + bench_lines["in22_conv"]["eng_Latn"]
    cnt = np.bincount([len_bucket(len(l.split())) for l in en], minlength=5).astype(float)
    S["in22_len_dist"] = (cnt / cnt.sum()).tolist()
    lo_w, hi_w = CONFIG["EN_WORDS"]  # BPCC's OWN length profile (human data, 3-60 words): the default target of the selection in C14
    hw = hum.en_w[(hum.en_w >= lo_w) & (hum.en_w <= hi_w)]
    hc = np.bincount([len_bucket(int(x)) for x in hw], minlength=5).astype(float)
    S["human_len_dist"] = (hc / hc.sum()).tolist()
    print("English length buckets (<=8, 9-14, 15-22, 23-35, 36-60)  IN22 (information only unless USE_IN22_LENGTH_MATCH):",
          np.round(S["in22_len_dist"], 3).tolist(), "| BPCC human data (used by default):", np.round(S["human_len_dist"], 3).tolist())
    mf_set("thresholds", {"ratio_bounds": bounds, "labse_min": CONFIG["LABSE_MIN"], "lid_min_prob": CONFIG["LID_MIN_PROB"],
                          "en_words": CONFIG["EN_WORDS"], "script_min_indic": 0.85, "script_min_english": 0.8,
                          "human_cap": CONFIG["HUMAN_CAP"], "source_cap": CONFIG["SOURCE_CAP"]})
    mf_set("in22_tokenizer_stats", stats)
    mf_set("in22_english_length_dist", S["in22_len_dist"])
    mf_set("human_english_length_dist", S["human_len_dist"])


timed("C09", c09)


In [ ]:
# C10 - Cheap filters with per-step counters
def c10():
    if S.get("prep_cached"):
        return
    df, log = S["df"], []
    bounds = S["manifest"]["thresholds"]["ratio_bounds"]

    def step(name, mask):
        nonlocal df
        out = df[np.asarray(mask, dtype=bool)]
        b = df.groupby(["pair", "subset"], sort=False).size()
        a = out.groupby(["pair", "subset"], sort=False).size()
        log.extend({"step": name, "pair": p, "subset": s, "before": int(n), "after": int(a.get((p, s), 0))} for (p, s), n in b.items())
        print(f"{name:<22}{len(df):>11,} -> {len(out):>11,}")
        df = out

    df = df.copy()
    df["src"], df["tgt"] = [norm_text(x) for x in df.src], [norm_text(x) for x in df.tgt]
    step("empty_after_norm", (df.src != "") & (df.tgt != ""))
    lo, hi = CONFIG["EN_WORDS"]
    enw, inw = df.src.str.split().str.len().values, df.tgt.str.split().str.len().values
    step("english_3_60_words", (enw >= lo) & (enw <= hi))
    enw, inw = df.src.str.split().str.len().values, df.tgt.str.split().str.len().values
    rl, rh = (df.pair.map(lambda p: bounds[p][i]).values for i in (0, 1))
    step("word_ratio_bounds", (inw / enw >= rl) & (inw / enw <= rh))
    codes = df.pair.str.split("-").str[1].values
    step("script_ratio", np.array([script_ratio(s, "eng_Latn") >= 0.8 and script_ratio(t, c) >= 0.85
                                   for s, t, c in zip(df.src, df.tgt, codes)]))
    step("urls_email_phone_html", np.array([not (has_noise(s) or has_noise(t)) for s, t in zip(df.src, df.tgt)]))
    step("digits_symbols_gt30pct", np.array([sym_digit_frac(s) <= 0.30 and sym_digit_frac(t) <= 0.30 for s, t in zip(df.src, df.tgt)]))
    step("src_equals_tgt", np.array([s.lower() != t.lower() for s, t in zip(df.src, df.tgt)]))
    step("numeral_check", np.array([numeral_check(s, t) for s, t in zip(df.src, df.tgt)]))
    S["df"] = df.reset_index(drop=True)
    pd.DataFrame(log).to_csv(os.path.join(AUDIT_DIR, "filter_counts_cheap.csv"), index=False)
    mf_set("filter_log_cheap", log)
    mf_set("rows_after_cheap_filters", {p: int((S["df"].pair == p).sum()) for p in CONFIG["PAIRS"]})


timed("C10", c10)


In [ ]:
# C11 - Mined data only: fastText LID + LaBSE, audit CSV
def c11():
    if S.get("prep_cached"):
        return
    import fasttext
    df, log = S["df"], []
    lid = fasttext.load_model(hf_hub_download(CONFIG["LID_ID"], "model.bin"))  # CC-BY-NC-4.0: filter/analysis tool only
    codes = df.pair.str.split("-").str[1].values
    mined = np.where(~df.is_human.values)[0]
    keep = np.ones(len(df), bool)
    tgt = df.tgt.values
    for i in range(0, len(mined), 50_000):
        idx = mined[i:i + 50_000]
        for j, (lab, pr) in zip(idx, lid_top1(lid, [tgt[k] for k in idx])):
            keep[j] = (lab == codes[j]) and pr >= CONFIG["LID_MIN_PROB"]
    after = df[keep]
    log.append({"step": "lid_mined", "before": len(df), "after": len(after), "mined_before": len(mined),
                "mined_after": int((~after.is_human).sum())})
    print(f"LID (mined only): {len(df):,} -> {len(after):,}")
    rng, rows = np.random.RandomState(CONFIG["SEED"]), []
    for code in INDIC:  # audit sample for a native reader, drawn BEFORE the LaBSE threshold is applied
        m = after[(after.pair == f"eng_Latn-{code}") & (~after.is_human) & after.labse.notna()]
        for lo, n in ((0.75, 34), (0.80, 33), (0.85, 33)):
            band = m[(m.labse >= lo) & (m.labse < lo + 0.05)]
            smp = band.sample(min(n, len(band)), random_state=int(rng.randint(0, 2 ** 31 - 1)))
            rows += [{"language": code, "band": f"[{lo:.2f},{lo + 0.05:.2f})", "labse": float(r.labse), "src": r.src, "tgt": r.tgt,
                      "native_judgement": "", "notes": ""} for r in smp.itertuples()]
    pd.DataFrame(rows).to_csv(os.path.join(AUDIT_DIR, "audit_mined_pairs.csv"), index=False, encoding="utf-8-sig")
    has_l, mined_m = after.labse.notna().values, ~after.is_human.values
    ok = np.where(has_l, after.labse.fillna(0).values >= CONFIG["LABSE_MIN"], CONFIG["MINED_WITHOUT_LABSE"] == "keep")  # mined rows only
    # starvation guard: each language needs about (1-HUMAN_CAP) x (pool + reserve + dev) mined candidates, ~half of which survive later filters
    need = int(2.0 * (1 - CONFIG["HUMAN_CAP"]) * (CONFIG["POOL"] // 3 + CONFIG["RESERVE_PER_LANG"] + 2 * CONFIG["DEV_PAIRS_PER_DIR"]))
    starved = {}
    if CONFIG["MINED_WITHOUT_LABSE"] == "drop":
        for code in INDIC:
            lang_m = (after.pair == f"eng_Latn-{code}").values & mined_m
            have = int((ok & lang_m).sum())
            if have < need:
                readd = CONFIG["ON_LABSE_STARVATION"] == "keep_unscored"
                if readd:  # keeps N, but this language then has weaker quality control (disclosed in the manifest / data card / model card)
                    ok = ok | (lang_m & ~has_l)
                starved[code] = {"mined_pairs_passing_labse": have, "needed": need, "policy": CONFIG["ON_LABSE_STARVATION"],
                                 "unscored_pairs_kept": int((lang_m & ~has_l).sum()) if readd else 0}
                print(f"WARNING {code}: only {have:,} scored mined pairs pass LaBSE (need ~{need:,}); policy {CONFIG['ON_LABSE_STARVATION']!r}: "
                      + (f"keeping {starved[code]['unscored_pairs_kept']:,} UNSCORED mined pairs - weaker quality control for this language"
                         if readd else "unscored pairs stay excluded - the pool (and N) may shrink"))
    final = after[~mined_m | ok]
    log.append({"step": "labse_min", "before": len(after), "after": len(final), "labse_min": CONFIG["LABSE_MIN"],
                "mined_without_score": int((mined_m & ~has_l).sum()), "policy": CONFIG["MINED_WITHOUT_LABSE"],
                "starved_languages_kept_unscored": starved})
    print(f"LaBSE >= {CONFIG['LABSE_MIN']} (unscored mined pairs: {CONFIG['MINED_WITHOUT_LABSE']}): {len(after):,} -> {len(final):,}; "
          f"audit CSV rows: {len(rows)}")
    S["df"] = final.reset_index(drop=True)
    mf_set("filter_log_mined", log)
    mf_set("rows_after_lid_labse", {p: int((S["df"].pair == p).sum()) for p in CONFIG["PAIRS"]})


timed("C11", c11)


In [ ]:
# C12 - Dedup + decontamination against ALL of IN22 and FLORES-22 dev
def c12():
    if S.get("prep_cached"):
        return
    df, log = S["df"], []
    df = df.sort_values("is_human", ascending=False, kind="stable")  # humans win ties
    df["sk"] = [dedup_key(s) for s in df.src]
    df["tk"] = [dedup_key(t) for t in df.tgt]
    n0 = len(df)
    df = df.drop_duplicates(["pair", "sk"]).drop_duplicates(["pair", "tk"])
    log.append({"step": "exact_dedup_within_pair_across_subsets", "before": n0, "after": len(df)})
    exact, grams = set(), set()
    for s in BENCH_SETS:
        for code, lines in S["bench_lines"][s].items():
            exact.update(dedup_key(l) for l in lines)
            for l in lines:
                grams |= word_ngrams(l, 8)  # sentences shorter than 8 words only match exactly
    ex = np.array([(a in exact) or (b in exact) for a, b in zip(df.sk, df.tk)])
    ng = np.array([(not word_ngrams(s, 8).isdisjoint(grams)) or (not word_ngrams(t, 8).isdisjoint(grams)) for s, t in zip(df.src, df.tgt)])
    log.append({"step": "decontaminate_exact", "removed": int(ex.sum())})
    log.append({"step": "decontaminate_8gram_only", "removed": int((ng & ~ex).sum())})
    S["df"] = df[~(ex | ng)].reset_index(drop=True)
    log.append({"step": "after_dedup_decontam", "after": len(S["df"])})
    print(pd.DataFrame(log).to_string(index=False))
    pd.DataFrame(log).to_csv(os.path.join(AUDIT_DIR, "dedup_decontam_counts.csv"), index=False)
    mf_set("filter_log_dedup", log)
    mf_set("rows_after_dedup_decontam", {p: int((S["df"].pair == p).sum()) for p in CONFIG["PAIRS"]})


timed("C12", c12)


In [ ]:
# C13 - Digit audit (ASCII vs native digits)
def c13():
    if S.get("prep_cached"):
        return
    """Default (USE_IN22_DIGIT_AUDIT=False): make the digit convention of the Indic training side CONSISTENT, using BPCC's own
    dominant form per language - the test set takes no part in the decision.  IN22's digit statistics are printed for information.
    USE_IN22_DIGIT_AUDIT=True restores the spec's variant: convert targets to IN22's dominant form when the forms differ by >20 points."""
    df, rows = S["df"], []
    for code in INDIC:
        ref = S["bench_lines"]["in22_gen"][code] + S["bench_lines"]["in22_conv"][code]
        a_r, n_r = digit_counts(ref)
        m = (df.pair == f"eng_Latn-{code}").values
        a_t, n_t = digit_counts(df.tgt[m])
        f_ref, f_tr = n_r / max(1, a_r + n_r), n_t / max(1, a_t + n_t)
        if CONFIG["USE_IN22_DIGIT_AUDIT"]:
            native, convert, basis = f_ref >= 0.5, (a_r + n_r >= 50 and abs(f_ref - f_tr) > 0.20), "IN22"
        else:
            native, convert, basis = f_tr >= 0.5, (a_t + n_t >= 50 and min(f_tr, 1 - f_tr) > 0.02), "BPCC dominant form"
        action = "none"
        if convert:
            conv = (lambda t, c=code: to_native_digits(t, c)) if native else (lambda t: to_ascii_digits(t))
            df.loc[m, "tgt"] = [conv(t) for t in df.tgt[m]]
            action = "targets -> native digits" if native else "targets -> ASCII digits"
        rows.append({"language": code, "train_native_frac": round(f_tr, 3), "in22_native_frac(info)": round(f_ref, 3),
                     "decision_basis": basis, "action": action})
    print(pd.DataFrame(rows).to_string(index=False))
    S["df"] = df
    mf_set("digit_audit", rows)


timed("C13", c13)


In [ ]:
# C14a - Selection helper (length-stratified, human-capped, source-capped)
def select_pairs(c, total, q, hcap, scap, rng):
    """Pick up to `total` rows (index labels) from candidate frame c following the 5-bucket English length distribution q; human share <= hcap;
    any subset <= scap per bucket.  Missing rows are back-filled from the rest.  q is derived from the TRAINING data (BPCC human data) by
    default; the IN22 test-set histogram is used only when CONFIG['USE_IN22_LENGTH_MATCH'] is True (default False: the evaluation set never
    shapes the training distribution)."""
    chosen = []
    seeds = lambda: int(rng.randint(0, 2 ** 31 - 1))  # noqa: E731
    for b, want in enumerate(largest_remainder(q, total)):
        sub = c[c.bucket == b]
        cap_b = int(math.ceil(scap * want))
        taken = 0
        for part, k in ((sub[sub.is_human], min(int(want * hcap), int(sub.is_human.sum()))), (sub[~sub.is_human], None)):
            k = (want - taken) if k is None else k
            if k <= 0 or len(part) == 0:
                continue
            alloc = waterfill(part.subset.value_counts().to_dict(), k, cap=cap_b)
            for s_, n in alloc.items():
                if n:
                    chosen.extend(part[part.subset == s_].sample(n, random_state=seeds()).index.tolist())
            taken += sum(alloc.values())
    need = total - len(chosen)
    if need > 0:  # relax bucket/source caps, keep the human cap
        rest = c.loc[~c.index.isin(chosen)]
        mined_rest = rest[~rest.is_human]
        extra = mined_rest.sample(min(need, len(mined_rest)), random_state=seeds()).index.tolist()
        chosen.extend(extra)
        need = total - len(chosen)
        h_left = max(0, int(hcap * total) - int(c.loc[chosen].is_human.sum()))
        hum_rest = rest[rest.is_human]
        if need > 0 and h_left > 0 and len(hum_rest):
            chosen.extend(hum_rest.sample(min(need, h_left, len(hum_rest)), random_state=seeds()).index.tolist())
    assert len(set(chosen)) == len(chosen)
    return chosen


In [ ]:
# C14b - Selection: dev first, then disjoint pool + reserve per language
def to_dir_rows(rows, direction):
    """Pair-format rows (src=English, tgt=Indic) -> training-direction rows."""
    en_first = split_direction(direction)[0] == "eng_Latn"
    return pd.DataFrame({"rid": rows.index.values, "direction": direction, "src": (rows.src if en_first else rows.tgt).values,
                         "tgt": (rows.tgt if en_first else rows.src).values, "subset": rows.subset.values,
                         "labse": rows.labse.values})


def c14():
    if S.get("prep_cached"):
        return
    df, rng = S["df"], np.random.RandomState(CONFIG["SEED"])
    use_in22 = CONFIG["USE_IN22_LENGTH_MATCH"]  # default False: the test set does not shape the training distribution
    q = S["in22_len_dist"] if use_in22 else S["human_len_dist"]
    df["en_w"] = df.src.str.split().str.len()
    df["bucket"] = df.en_w.map(len_bucket)
    hcap, scap = CONFIG["HUMAN_CAP"], CONFIG["SOURCE_CAP"]
    cand = {c: df[df.pair == f"eng_Latn-{c}"] for c in INDIC}
    n_dev, n_pool, n_res = 2 * CONFIG["DEV_PAIRS_PER_DIR"], CONFIG["POOL"] // 3, CONFIG["RESERVE_PER_LANG"]
    dev_idx = {c: select_pairs(cand[c], n_dev, q, hcap, scap, rng) for c in INDIC}
    dev_sk = set().union(*[set(df.loc[v, "sk"]) for v in dev_idx.values()])  # no dev English sentence may appear in ANY pool pair
    sel = {"pool": {}, "reserve": {}, "dev": {}}
    stats = {}
    for c in INDIC:
        left = cand[c].loc[~cand[c].index.isin(dev_idx[c]) & ~cand[c].sk.isin(dev_sk)]
        pick = select_pairs(left, n_pool + n_res, q, hcap, scap, rng)
        got = left.loc[pick].sample(frac=1, random_state=CONFIG["SEED"])
        got = got.sort_values(["bucket", "is_human", "subset"], kind="stable")  # shuffled, then stratified by (bucket, human, subset)
        res_pos = np.unique(np.linspace(0, len(got) - 1, min(n_res, len(got) // 10)).astype(int))
        reserve = got.iloc[res_pos]
        pool = got.drop(index=reserve.index)
        a, b = pool.iloc[0::2], pool.iloc[1::2]  # alternate -> both halves share the stratification
        d_rows = df.loc[dev_idx[c]].sample(frac=1, random_state=CONFIG["SEED"])
        h = len(d_rows) // 2
        sel["pool"][f"eng_Latn-{c}"], sel["pool"][f"{c}-eng_Latn"] = a, b
        sel["reserve"][f"eng_Latn-{c}"], sel["reserve"][f"{c}-eng_Latn"] = reserve.iloc[0::2], reserve.iloc[1::2]
        sel["dev"][f"eng_Latn-{c}"], sel["dev"][f"{c}-eng_Latn"] = d_rows.iloc[:h], d_rows.iloc[h:]
        stats[c] = {"selected": len(got), "pool": len(pool), "reserve": len(reserve), "dev": len(d_rows),
                    "human_share": round(float(got.is_human.mean()), 3), "eligible": len(left), "headroom": round(len(left) / max(1, len(got)), 2),
                    "mined_labse_mean": round(float(got.labse[~got.is_human].dropna().mean()), 3),
                    "mined_labse_p10": round(float(got.labse[~got.is_human].dropna().quantile(0.10)), 3),
                    "bucket_share": np.round(got.bucket.value_counts(normalize=True).sort_index().reindex(range(5), fill_value=0).values, 3).tolist(),
                    "top_subsets": got.subset.value_counts(normalize=True).round(3).head(6).to_dict()}
    for c_, st in stats.items():  # a selection that takes almost ALL eligible pairs dilutes quality: the mean LaBSE of the mined part shows it
        if st["headroom"] < 1.5:
            print(f"WARNING {c_}: only {st['eligible']:,} eligible pairs for {st['selected']:,} selected (headroom {st['headroom']}x): the selection is nearly "
                  f"exhaustive; mean LaBSE of the selected mined pairs {st['mined_labse_mean']} (10th percentile {st['mined_labse_p10']})")
    for c_, st in stats.items():  # the source cap is applied per length bucket: report the resulting GLOBAL share
        top = max(st["top_subsets"].values())
        if top > scap + 0.02:
            print(f"NOTE {c_}: the largest single subset supplies {top:.1%} of the selection (cap {scap:.0%} is enforced per length bucket; "
                  "relaxed when too few subsets exist)")
    ids = [r for part in sel.values() for v in part.values() for r in v.index]
    assert len(ids) == len(set(ids)), "pool / reserve / dev share pairs"
    pool_sk = set().union(*[set(v.sk) for v in sel["pool"].values()]) | set().union(*[set(v.sk) for v in sel["reserve"].values()])
    assert not (pool_sk & dev_sk), "a dev English sentence also occurs in the pool"
    per_dir = min(len(v) for v in sel["pool"].values())
    if 6 * per_dir < CONFIG["N"]:  # (the pool aims higher than N as headroom for the >512-token drops of C15)
        raise RuntimeError(f"the filtered data supplies only {per_dir:,} pool pairs per direction ({6 * per_dir:,} in total) so N={CONFIG['N']:,} "
                           "cannot be reached: the experiment is NOT run on a smaller dataset. Loosen a filter / raise CONFIG['MINED_PARSE_CAP'] and re-run.")
    if per_dir < CONFIG["POOL"] // 6:
        print(f"WARNING: data short - every direction scaled down equally to {per_dir:,} pool pairs (target {CONFIG['POOL'] // 6:,})")
        sel["pool"] = {k: v.iloc[:per_dir] for k, v in sel["pool"].items()}
    print(f"English length target ({'IN22' if use_in22 else 'BPCC human data'}):", np.round(q, 3).tolist())
    for c, st in stats.items():
        print(c, st)
    S["sel"] = sel
    mf_set("selection", {"per_language": stats, "pool_per_direction": per_dir, "disjoint_assertions": "passed",
                         "length_target": "IN22 histogram" if use_in22 else "BPCC human-data histogram", "length_target_dist": q})


timed("C14", c14)


In [ ]:
# C15 - Build examples (prompt and target tokenised SEPARATELY, + EOS)
def c15():
    if S.get("prep_cached"):
        return
    tok = load_tok()
    EOS = tok.eos_token_id
    assert tok.eos_token == EOS_TOKEN and tok.convert_tokens_to_ids(EOS_TOKEN) == EOS, f"unexpected EOS: {tok.eos_token}"
    assert tok.bos_token_id is None, "base tokenizer unexpectedly defines a BOS token"
    print("EOS id", EOS, "| pad id", tok.pad_token_id, "| BOS", tok.bos_token_id)
    MAXT = CONFIG["MAX_TOKENS"]

    def encode(rows):
        sn = [LANGS[split_direction(d)[0]]["name"] for d in rows.direction]
        tn = [LANGS[split_direction(d)[1]]["name"] for d in rows.direction]
        ids, pl = encode_examples(tok, sn, tn, rows.src.tolist(), rows.tgt.tolist(), EOS)
        out = rows.copy()
        out["input_ids"], out["prompt_len"], out["n_tok"] = ids, pl, [len(x) for x in ids]
        return out

    per_dir_target, final, dev, rep = CONFIG["POOL"] // 6, {}, {}, []
    for d in DIRECTIONS:
        base = encode(to_dir_rows(S["sel"]["pool"][d], d))
        keep = base[base.n_tok <= MAXT]
        dropped, refilled = len(base) - len(keep), 0
        want = min(per_dir_target, len(base))
        if len(keep) < want:
            res = encode(to_dir_rows(S["sel"]["reserve"][d], d))
            res = res[res.n_tok <= MAXT].iloc[:want - len(keep)]
            keep, refilled = pd.concat([keep, res], ignore_index=True), len(res)
        final[d] = keep
        dv = encode(to_dir_rows(S["sel"]["dev"][d], d))
        dev[d] = dv[dv.n_tok <= MAXT]
        rep.append({"direction": d, "pool_before": len(base), "dropped_gt512": dropped, "drop_rate": round(dropped / max(1, len(base)), 4),
                    "refilled": refilled, "pool_final": len(keep), "dev_final": len(dev[d]), "mean_tokens": round(float(keep.n_tok.mean()), 1)})
    print(pd.DataFrame(rep).to_string(index=False))
    m_dev = min(len(v) for v in dev.values())  # equal-sized dev sets across directions (>512-token drops can differ per direction)
    if m_dev < CONFIG["DEV_PAIRS_PER_DIR"]:
        print(f"NOTE: dev sets shrank to {m_dev} per direction (requested {CONFIG['DEV_PAIRS_PER_DIR']}) after the >{MAXT}-token filter; all directions use {m_dev}")
    if m_dev < 0.9 * CONFIG["DEV_PAIRS_PER_DIR"]:  # a dev set this much smaller than requested would make the dev-loss curves noisy / uneven
        raise RuntimeError(f"only {m_dev} dev pairs per direction survive the token filter (requested {CONFIG['DEV_PAIRS_PER_DIR']}, minimum "
                           f"{int(0.9 * CONFIG['DEV_PAIRS_PER_DIR'])}): raise CONFIG['DEV_PAIRS_PER_DIR'] and re-run the notebook")
    dev = {d: v.iloc[:m_dev] for d, v in dev.items()}
    m = min(len(v) for v in final.values())
    n_final = CONFIG["N"] if 6 * m >= CONFIG["N"] else (6 * m // CONFIG["GLOBAL_BATCH"]) * CONFIG["GLOBAL_BATCH"]
    if n_final < CONFIG["N"]:
        raise RuntimeError(f"only {n_final:,} examples survive the token filter (N={CONFIG['N']:,} is required): the experiment would silently shrink. "
                           "Raise CONFIG['RESERVE_PER_LANG'] / loosen a filter and re-run.")
    final = {d: v.iloc[:m] for d, v in final.items()}
    smp = pd.concat(final.values()).sample(20, random_state=CONFIG["SEED"])
    for r in smp.itertuples():
        pn, tn = (LANGS[x]["name"] for x in split_direction(r.direction))
        assert r.input_ids[-1] == EOS
        assert tok.decode(r.input_ids[:r.prompt_len], clean_up_tokenization_spaces=False) == build_prompt(pn, tn, r.src)
        assert tok.decode(r.input_ids[r.prompt_len:-1], clean_up_tokenization_spaces=False) == " " + r.tgt
    print("boundary decode test passed on 20 examples; prompt ends at '<Target>:', target = ' ' + text + EOS")
    S["final"], S["dev_final"], S["n_final"] = final, dev, n_final
    mf_set("examples", {"rows_per_direction": m, "dev_rows_per_direction": m_dev, "n_final": n_final, "n_nominal": CONFIG["N"], "drop_report": rep,
                        "eos_id": EOS, "pad_id": tok.pad_token_id})


timed("C15", c15)


In [ ]:
# C16a - Order (round-robin + blocks of 600), save, flores sample
def _to_frame(parts, order_col):
    df = pd.concat(parts, ignore_index=True)
    df["id"] = df.direction + "-" + df.rid.astype(str)
    df["input_ids"] = [np.asarray(x, dtype=np.int32) for x in df.input_ids]
    df["prompt_len"] = df.prompt_len.astype("int32")
    df["order"] = np.arange(len(df)) if order_col else -1
    return df[["id", "order", "direction", "src", "tgt", "subset", "labse", "prompt_len", "input_ids"]]


def c16a():
    if S.get("prep_cached"):
        return
    rng = np.random.RandomState(CONFIG["SEED"])
    per = [S["final"][d].sample(frac=1, random_state=int(rng.randint(0, 2 ** 31 - 1))).reset_index(drop=True) for d in DIRECTIONS]
    m = len(per[0])
    rr = pd.concat([p.assign(_k=np.arange(m) * 6 + j) for j, p in enumerate(per)]).sort_values("_k", kind="stable").reset_index(drop=True)
    perm = np.concatenate([b + rng.permutation(min(600, len(rr) - b)) for b in range(0, len(rr), 600)])  # each full 600-block is exactly balanced (100 per direction)
    pool = _to_frame([rr.iloc[perm].drop(columns="_k")], True)
    dev = _to_frame(list(S["dev_final"].values()), False)
    first = pool.iloc[:S["n_final"]].direction.value_counts()
    dev_max = float((first - S["n_final"] / 6).abs().max())  # blocks of 600 are exactly balanced; only the cut block can deviate
    if dev_max > max(0.005 * S["n_final"] / 6, 20):
        print(f"WARNING: first-N prefix is less balanced than expected (max deviation {dev_max:.0f}): {first.to_dict()}")
    pool.to_parquet(os.path.join(PREP_DIR, "pool.parquet"), index=False)
    dev.to_parquet(os.path.join(PREP_DIR, "dev_loss.parquet"), index=False)
    fl, lines = np.random.RandomState(CONFIG["SEED"]), []
    for d in DIRECTIONS:
        pairs = S["bench"]["flores_dev"][d]
        for i in sorted(fl.choice(len(pairs), CONFIG["FLORES_DEV_PER_DIR"], replace=False).tolist()):
            lines.append({"direction": d, "idx": i, "id": f"flores_dev-{i:04d}", "src": pairs[i][0], "ref": pairs[i][1]})
    atomic_save(os.path.join(PREP_DIR, "flores_dev_sample.jsonl"), "\n".join(json.dumps(x, ensure_ascii=False) for x in lines) + "\n")
    print("pool rows", len(pool), "| dev rows", len(dev), "| flores sample", len(lines), "| first-N direction counts", first.to_dict())
    mf_set("pool", {"rows": len(pool), "dev_rows": len(dev), "flores_sample_rows": len(lines), "n_final": S["n_final"],
                    "steps_final": S["n_final"] // CONFIG["GLOBAL_BATCH"]})
    disk_report("after prep save")


timed("C16a", c16a)


**Changed (C16b):** after preparation it evaluates the one authoritative TIME BUDGET (`time_plan`, shared with C21) and STOPS here if the training would not fit. **Why:** preparation now shares the 12 h session with training, so an over-long preparation is visible immediately instead of at the first training step. **Effect:** none on the data.

In [ ]:
# C16b - Manifest, data card, integrity check of the processed data
def c16b():
    if S.get("prep_cached"):
        require_time(time_plan("processed data reused", pre_reserve_h=CONFIG["PRE_TRAIN_H"]))
        return
    mf = dict(S["manifest"])
    mf.update({"prep_sig": prep_sig(), "prep_timing_s": S.get("timing"), "code_version": CONFIG["CODE_VERSION"], "helper_sha256": hashlib.sha256(UTILS_SRC.encode("utf-8")).hexdigest(),
               "seeds": {"python/numpy/torch": CONFIG["SEED"]}, "revisions": dict(S["revs"]), "versions": VERSIONS,
               "config": CONFIG, "n_final": S["n_final"], "steps_final": S["n_final"] // CONFIG["GLOBAL_BATCH"],
               "sha256": {f: sha256_file(os.path.join(PREP_DIR, f)) for f in DATA_FILES}, "built_at": time.strftime("%Y-%m-%d %H:%M:%S")})
    atomic_save(os.path.join(PREP_DIR, "manifest.json"), mf)
    card = f"""# p1-qwen3-indic-mt-data (local working data)
Training pairs for English<->Gujarati/Hindi/Marathi translation, derived from `{CONFIG['DATA_ID']}` @ `{S['revs']['data']}`.
* Sources: BPCC human subsets (e.g. seed/daily/wiki, CC-BY-4.0 where applicable) and BPCC-Mined (CC0). Exact mix: manifest.json -> selection.
* Filters: NFC/whitespace normalisation, 3-60 English words, human-data word-ratio bounds, script ratio, URL/e-mail/phone/HTML removal,
  symbol/digit share, src==tgt, numeral preservation; mined data additionally fastText LID (facebook/fasttext-language-identification,
  CC-BY-NC-4.0, used only as a filter) and LaBSE >= {CONFIG['LABSE_MIN']}; exact dedup; decontamination against ALL of IN22-Gen/Conv and FLORES-22 dev
  (exact + 8-gram on both sides).
* Counts after every step: manifest.json (never hard-coded). Order: balanced round-robin over 6 directions, shuffled in blocks of 600.
* IN22 / FLORES text is NOT part of this dataset (flores_dev_sample.jsonl is FLORES-22 *dev*, used only for monitoring).
* LaBSE starvation (languages that kept UNSCORED mined pairs because too few scored ones existed): {[k for k, v in S['manifest'].get('filter_log_mined', [{}, {}])[-1].get('starved_languages_kept_unscored', {}).items() if v.get('unscored_pairs_kept')] or 'none'}.
* Length target of the selection: {'IN22 histogram (USE_IN22_LENGTH_MATCH=True)' if CONFIG['USE_IN22_LENGTH_MATCH'] else 'BPCC human-data histogram (independent of IN22)'}; digit convention decided from {'IN22 (USE_IN22_DIGIT_AUDIT=True)' if CONFIG['USE_IN22_DIGIT_AUDIT'] else 'BPCC itself'}; mined pairs without a LaBSE score: {CONFIG['MINED_WITHOUT_LABSE']}.
"""
    atomic_save(os.path.join(PREP_DIR, "DATA_CARD.md"), card)
    verify_data_dir()
    S["pool_sha256"] = mf["sha256"]["pool.parquet"]
    print("processed data written and verified:", list(mf["sha256"]), "->", PREP_DIR)
    require_time(time_plan("after preparation", pre_reserve_h=CONFIG["PRE_TRAIN_H"]))  # fail NOW, before the model download and the 7 h of training

timed("C16b", c16b)


## Part 2 — MODEL AND TRAINING SETUP (no trial runs of any kind)

Correctness here rests on: the unit tests of C05e, the assertions below (ids, shapes, parameter counts, inspection of the *installed* transformers source) and the static review at the end of the notebook. **There is no smoke test, LR probe, throughput probe, trial training step or trial generation.**

**Loss alignment (exact).** For example *i* with padded length *T*, true length *T_i* and prompt length *p_i*, the hidden state at position *t* predicts token *t+1*. Target positions are `t ∈ [p_i−1, T_i−2]` with labels `ids[i, t+1]` → exactly `T_i − p_i` target tokens, the last being EOS. The mask is built from **lengths**, never from the pad id (pad id == EOS id for this tokenizer). Hidden states come from the *inner* model (final RMSNorm applied; asserted from the installed source), are gathered at target positions only, and the tied-embedding logits are computed in ~1K-token chunks with per-chunk `torch.utils.checkpoint`, cast to fp32, summed, and divided by the **total target tokens of the whole global batch**.

In [ ]:
# C17 - Tokenizer report + load pool/dev/flores (sha256-verified)
def c17():
    mf = verify_data_dir()
    S["prep_manifest"] = mf
    n_final = int(mf["n_final"])
    assert n_final % CONFIG["GLOBAL_BATCH"] == 0 and n_final <= CONFIG["POOL"]
    steps = n_final // CONFIG["GLOBAL_BATCH"]
    RUN.update(N=n_final, STEPS=steps, WARMUP=round(0.04 * steps), GEN_STEPS=[round(steps * f) for f in (0.25, 0.5, 0.75, 1.0)])
    assert n_final == CONFIG["N"], f"the processed data holds {n_final:,} examples, not the required N = {CONFIG['N']:,}"
    assert RUN["STEPS"] == CONFIG["STEPS"] and RUN["WARMUP"] == CONFIG["WARMUP_STEPS"] and RUN["GEN_STEPS"] == CONFIG["GEN_EVAL_STEPS"]
    pool = pd.read_parquet(os.path.join(PREP_DIR, "pool.parquet"))
    assert (pool["order"].values == np.arange(len(pool))).all()
    S["pool"] = pool.iloc[:n_final].reset_index(drop=True)
    S["dev"] = pd.read_parquet(os.path.join(PREP_DIR, "dev_loss.parquet"))
    S["flores"] = [json.loads(l) for l in open(os.path.join(PREP_DIR, "flores_dev_sample.jsonl"), encoding="utf-8")]
    tok = load_tok()
    EOS = tok.eos_token_id
    assert tok.eos_token == EOS_TOKEN and tok.convert_tokens_to_ids(EOS_TOKEN) == EOS and tok.bos_token_id is None
    S["EOS"] = EOS
    print(f"EOS id={EOS} pad id={tok.pad_token_id} BOS={tok.bos_token_id} | pool used: {len(S['pool']):,} rows | dev rows: {len(S['dev'])}")
    for r in S["pool"].sample(20, random_state=1).itertuples():
        pn, tn = (LANGS[x]["name"] for x in split_direction(r.direction))
        ids = list(r.input_ids)
        assert ids[-1] == EOS
        assert tok.decode(ids[:r.prompt_len], clean_up_tokenization_spaces=False) == build_prompt(pn, tn, r.src)
        assert tok.decode(ids[r.prompt_len:-1], clean_up_tokenization_spaces=False) == " " + r.tgt
    fert = {}
    for code in LANGS:  # fertility = tokens per whitespace word on the FLORES-22 dev sample
        lines = [r["src"] if split_direction(r["direction"])[0] == code else r["ref"] for r in S["flores"]
                 if code in split_direction(r["direction"])]
        n_t = sum(len(x) for x in tok(lines, add_special_tokens=False)["input_ids"])
        fert[code] = round(n_t / max(1, sum(len(l.split()) for l in lines)), 2)
    print("tokens per word (FLORES-22 dev sample):", fert)
    first = S["pool"].direction.value_counts().to_dict()
    print("direction counts in the first N:", first)
    S["tok_report"] = {"eos_id": EOS, "pad_id": tok.pad_token_id, "bos_id": tok.bos_token_id, "vocab": len(tok),
                       "fertility_tokens_per_word": fert, "boundary_test": "20/20 stored examples decode exactly",
                       "pool_prefix_direction_counts": first}
    atomic_save(os.path.join(TRAIN_DIR, "tokenizer_report.json"), S["tok_report"])


RUN = {}
c17()


In [ ]:
# C18 - Data pipeline (global batch -> length-grouped micro-batches)
class Batcher:
    """Serves the stored order: global batch g = rows [64g, 64g+64).  Micro-batches are built by sorting the global
    batch by length and grouping so that max_len * group_size <= budget (padded tokens).  Right padding + explicit
    attention mask; the loss mask is derived from LENGTHS (target_position_mask), never from the pad id."""

    def __init__(self, df, pad_id):
        self.ids = [np.asarray(x, dtype=np.int64) for x in df["input_ids"]]
        self.plen = df["prompt_len"].values.astype(np.int64)
        self.dir = np.array([DIRECTIONS.index(d) for d in df["direction"]], dtype=np.int64)
        self.pad = pad_id

    def __len__(self):
        return len(self.ids)

    def target_tokens(self, rows):
        return int(sum(len(self.ids[r]) - self.plen[r] for r in rows))

    def micro_batches(self, rows, budget):
        lens = [len(self.ids[r]) for r in rows]
        for g in group_by_budget(lens, budget):
            T = max(lens[i] for i in g)
            ids = torch.full((len(g), T), self.pad, dtype=torch.long)
            att = torch.zeros((len(g), T), dtype=torch.long)
            for k, i in enumerate(g):
                ids[k, :lens[i]] = torch.from_numpy(self.ids[rows[i]])
                att[k, :lens[i]] = 1
            yield {"input_ids": ids, "attention_mask": att, "length": torch.tensor([lens[i] for i in g]),
                   "prompt_len": torch.tensor([int(self.plen[rows[i]]) for i in g]),
                   "direction": torch.tensor([int(self.dir[rows[i]]) for i in g]),
                   "orig_idx": torch.tensor(g), "row": torch.tensor([rows[i] for i in g])}


def c18():
    S["train_batcher"] = Batcher(S["pool"], S["EOS"])
    S["dev_batcher"] = Batcher(S["dev"], S["EOS"])
    rows = list(range(CONFIG["GLOBAL_BATCH"]))
    seen, n_mask = [], 0
    for mb in S["train_batcher"].micro_batches(rows, CONFIG["MICRO_BATCH_TOKENS"]):
        B, T = mb["input_ids"].shape
        assert B * T <= CONFIG["MICRO_BATCH_TOKENS"] or B == 1
        m = target_position_mask(mb["prompt_len"], mb["length"], T)
        assert (m.sum(1) == mb["length"] - mb["prompt_len"]).all()
        assert (mb["attention_mask"].sum(1) == mb["length"]).all()
        last = mb["input_ids"][torch.arange(B), mb["length"] - 1]
        assert (last == S["EOS"]).all(), "the final target label of every example must be EOS"
        _, lab = gather_targets(torch.zeros(B, T, 1), mb["input_ids"], m)
        assert lab.numel() == int((mb["length"] - mb["prompt_len"]).sum())
        n_mask += int(m.sum())
        seen += mb["orig_idx"].tolist()
    assert sorted(seen) == rows and n_mask == S["train_batcher"].target_tokens(rows)
    print(f"batcher OK: first global batch = {len(rows)} examples, {n_mask} target tokens; dev examples {len(S['dev_batcher'])}")


c18()


In [ ]:
# C19a - build_model(n_layers, optimizer_kind)
PER_LAYER_PARAMS = 50.3e6  # Qwen3-1.7B decoder layer (spec section 3)


def _bnb_ok():
    try:  # import check only: nothing is constructed or run here (a failure of the REAL optimizer is handled in build_model)
        import bitsandbytes as bnb
        return hasattr(bnb.optim, "AdamW8bit")
    except Exception as e:
        print(f"bitsandbytes AdamW8bit unavailable ({type(e).__name__}: {e})")
        return False


def make_scaler(init_scale):
    try:
        return torch.amp.GradScaler("cuda", init_scale=init_scale)
    except AttributeError:  # older torch
        return torch.cuda.amp.GradScaler(init_scale=init_scale)


def rungs_after(n_layers, budget):
    """Remaining ladder rungs once (n_layers, budget) is in use, in ladder order."""
    return [r for r in CONFIG["OOM_LADDER"] if (r[0] == n_layers and r[1] < budget) or r[0] < n_layers]


def free_model(M):
    if M:
        for k in list(M):
            M[k] = None
    gc.collect()
    torch.cuda.empty_cache()


def build_model(n_layers, kind):
    """Load PRISTINE base weights from disk, freeze all but the top n_layers (+ final norm), build optimizer + scaler."""
    from transformers import AutoModelForCausalLM
    if kind == "adamw8bit" and not _bnb_ok():
        kind, n_layers = "adamw", min(n_layers, 8)
        print("-> optimizer fallback: torch AdamW with 8 trained layers")
    dev = torch.device("cuda:0")
    model = AutoModelForCausalLM.from_pretrained(MODEL_SRC_DIR, torch_dtype=torch.float16, attn_implementation="sdpa",
                                                 low_cpu_mem_usage=True)
    model.config.use_cache = False
    assert float(getattr(model.config, "attention_dropout", 0.0) or 0.0) == 0.0, "the design uses dropout 0 (and plain cross-entropy: no label smoothing)"
    model.to(dev)
    inner, layers = model.model, model.model.layers
    n_total = len(layers)
    assert n_total == model.config.num_hidden_layers == 28 and 0 < n_layers < n_total
    assert "self.norm" in inspect.getsource(type(inner).forward), \
        "installed Qwen3Model.forward does not apply self.norm: hidden states would be pre-norm. Pin transformers 4.57.x."
    assert model.lm_head.weight.data_ptr() == inner.embed_tokens.weight.data_ptr(), "embeddings are not tied"
    first = n_total - n_layers  # layers[first:] are trained; for n=10 -> layers[18:28] = layers 19..28
    for p in model.parameters():
        p.requires_grad_(False)
    for l in layers[first:]:
        l.float()
        l.requires_grad_(True)
    inner.norm.float()
    inner.norm.requires_grad_(True)
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for _, p in named)
    exp = n_layers * PER_LAYER_PARAMS
    emb = inner.embed_tokens.weight.numel()
    print(f"params: total {tot / 1e6:.1f}M | embeddings {emb / 1e6:.1f}M (frozen, tied head) | trainable {trn / 1e6:.1f}M "
          f"(layers[{first}:{n_total}] + norm) | frozen {(tot - trn) / 1e6:.1f}M")
    assert abs(trn - exp) / exp <= 0.02, f"trainable {trn / 1e6:.1f}M differs >2% from expected {exp / 1e6:.1f}M"
    assert all(p.dtype == torch.float32 for _, p in named) and not any(p.requires_grad for p in layers[:first].parameters())
    assert not inner.embed_tokens.weight.requires_grad and not model.lm_head.weight.requires_grad
    assert "gradient_checkpointing_kwargs" in inspect.signature(model.gradient_checkpointing_enable).parameters
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    for l in layers[:first]:  # frozen layers record no graph anyway: do not wrap them in checkpoint
        if hasattr(l, "gradient_checkpointing"):
            l.gradient_checkpointing = False
    params = [p for _, p in named]
    groups = [{"params": [p for p in params if p.ndim >= 2], "weight_decay": CONFIG["WD"]},
              {"params": [p for p in params if p.ndim < 2], "weight_decay": 0.0}]  # norm weights incl. q_norm/k_norm: no decay
    if kind == "adamw8bit":
        try:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit(groups, lr=CONFIG["LR_PEAK"], betas=CONFIG["BETAS"], eps=CONFIG["EPS"])
        except Exception as e:  # spec: bitsandbytes failure -> torch AdamW with 8 trained layers (pristine reload)
            print(f"-> AdamW8bit could not be created ({type(e).__name__}: {e}); falling back to torch AdamW with 8 trained layers")
            del model, inner, layers, named, params, groups
            gc.collect()
            torch.cuda.empty_cache()
            return build_model(min(n_layers, 8), "adamw")
    else:
        opt = torch.optim.AdamW(groups, lr=CONFIG["LR_PEAK"], betas=CONFIG["BETAS"], eps=CONFIG["EPS"])
    held = {id(p) for g in opt.param_groups for p in g["params"]}
    assert held == {id(p) for p in params}, "optimizer must hold exactly the trainable params"
    scaler = make_scaler(CONFIG["SCALER_INIT"])
    nd = sum(p.numel() for p in groups[1]["params"])
    print(f"optimizer={kind} | decay/no-decay params: {trn - nd:,}/{nd:,} | dtype frozen={layers[0].self_attn.q_proj.weight.dtype} "
          f"trained={layers[-1].self_attn.q_proj.weight.dtype} | GPU mem {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    return {"model": model, "opt": opt, "scaler": scaler, "trainable": named, "params": params, "n_layers": n_layers,
            "kind": kind, "first": first, "dev": dev}


In [ ]:
# C19b - Build the model (rung 0 of the recovery ladder)
def c19():
    fetch_model(tokenizer_only=False)  # pinned commit, weights included
    disk_report("after model download")
    n, budget = CONFIG["OOM_LADDER"][0]
    S["M"] = build_model(n, "adamw8bit")
    S["budget"] = budget if S["M"]["n_layers"] == n else 6000
    S["rungs"] = rungs_after(S["M"]["n_layers"], S["budget"])
    print(f"start: {S['M']['n_layers']} trained layers, optimizer {S['M']['kind']}, MICRO_BATCH_TOKENS={S['budget']}")


c19()


**Pre-training validation (new, C19c).** Pure assertions before the expensive training starts — *no forward pass, no trial step, no probe*: the trainable / frozen split (layers 18-27 + final norm only), optimizer parameter groups, the step-based LR curve, direction balance of the first N rows, pool ↔ dev ↔ benchmark disjointness re-checked on the **final stored pool**, prompt/inference parity and EOS-terminated targets on a random sample, and a synthetic full-logits reference of the loss with `pad == EOS`. **Why:** a wrong selection or mask would only be discovered after ~7 h. **Effect:** none on the model; failures stop the notebook here, in minutes.

In [ ]:
# C19c - Pre-training validation (assertions only)
def c19c():
    M, tok = S["M"], load_tok()
    names = [n for n, _ in M["trainable"]]
    layers_t = {int(n.split(".")[2]) for n in names if n.startswith("model.layers.")}
    assert layers_t == set(range(28 - M["n_layers"], 28)), f"trainable layers {sorted(layers_t)}"
    assert [n for n in names if not n.startswith("model.layers.")] == ["model.norm.weight"], "besides the top layers only the final norm may be trainable"
    assert not any(p.requires_grad for n, p in M["model"].named_parameters() if n not in set(names)), "a frozen parameter requires grad"
    tot, trn = sum(p.numel() for p in M["model"].parameters()), sum(p.numel() for _, p in M["trainable"])
    print(f"parameters: total {tot / 1e6:.1f}M | trainable {trn / 1e6:.1f}M ({100 * trn / tot:.1f}%) | frozen {(tot - trn) / 1e6:.1f}M | "
          f"trainable layers {min(layers_t)}-{max(layers_t)} (0-based) + final norm | embeddings and tied head frozen")
    ids_opt = [id(p) for g in M["opt"].param_groups for p in g["params"]]
    assert len(ids_opt) == len(set(ids_opt)) == len(M["trainable"]) and set(ids_opt) == {id(p) for _, p in M["trainable"]}, "optimizer parameter groups"
    assert [g["weight_decay"] for g in M["opt"].param_groups] == [CONFIG["WD"], 0.0] and all(p.dtype == torch.float32 for _, p in M["trainable"])
    st, wu = RUN["STEPS"], RUN["WARMUP"]
    lrs = [lr_schedule(s, st, wu, CONFIG["LR_PEAK"], CONFIG["LR_MIN"]) for s in range(st)]
    assert min(lrs) >= CONFIG["LR_MIN"] - 1e-15 and max(lrs) <= CONFIG["LR_PEAK"] + 1e-15
    assert abs(lrs[wu - 1] - CONFIG["LR_PEAK"]) < 1e-12 and abs(lrs[-1] - CONFIG["LR_MIN"]) < 1e-12
    assert all(a <= b + 1e-15 for a, b in zip(lrs[:wu], lrs[1:wu])) and all(a >= b - 1e-15 for a, b in zip(lrs[wu - 1:], lrs[wu:])), "LR is not warm-up then decay"
    print("LR (step-based):", {s: f"{lrs[s]:.2e}" for s in (0, 12, wu - 1, st // 4, st // 2, 3 * st // 4, st - 1)})
    pool, dev = S["pool"], S["dev"]
    cnt = pool.direction.value_counts()
    assert len(pool) == RUN["N"] == CONFIG["N"] and set(cnt.index) == set(DIRECTIONS) and (cnt - RUN["N"] / 6).abs().max() <= max(0.005 * RUN["N"] / 6, 20), cnt.to_dict()
    assert pool["id"].is_unique and dev["id"].is_unique and not (set(pool["id"]) & set(dev["id"])), "pool / dev share examples"
    en_key = lambda df: {dedup_key(s if split_direction(d)[0] == "eng_Latn" else t) for s, t, d in zip(df.src, df.tgt, df.direction)}  # noqa: E731
    assert not (en_key(pool) & en_key(dev)), "a dev English sentence also occurs in the training pool"
    exact = {dedup_key(l) for s in BENCH_SETS for ls in S["bench_lines"][s].values() for l in ls}
    leaked = sum((dedup_key(a) in exact) or (dedup_key(b) in exact) for a, b in zip(pool.src, pool.tgt))
    assert leaked == 0, f"{leaked} training examples are verbatim IN22 / FLORES sentences"
    assert pool["input_ids"].map(len).max() <= CONFIG["MAX_TOKENS"]
    for r in pool.sample(300, random_state=CONFIG["SEED"]).itertuples():
        a, b = (LANGS[x]["name"] for x in split_direction(r.direction))
        seq = list(r.input_ids)
        assert seq[:r.prompt_len] == tok(build_prompt(a, b, r.src), add_special_tokens=False)["input_ids"], "training prompt != the prompt generate_batch feeds"
        assert seq[r.prompt_len:-1] == tok([" " + r.tgt], add_special_tokens=False)["input_ids"][0] and seq[-1] == S["EOS"]
    # loss alignment on synthetic tensors with right padding by the EOS id (pad == EOS): full-logits reference vs chunked_ce
    g, V, H_ = torch.Generator().manual_seed(0), 40, 8
    ids = torch.randint(0, V - 1, (3, 9), generator=g)
    L, pl = torch.tensor([9, 7, 8]), torch.tensor([3, 2, 4])
    for b in range(3):
        ids[b, L[b] - 1:] = V - 1  # EOS at the end of the target, then pad == EOS
    hid, W = torch.randn(3, 9, H_, generator=g, dtype=torch.float64), torch.randn(V, H_, generator=g, dtype=torch.float64)
    m = target_position_mask(pl, L, 9)
    h, lab = gather_targets(hid, ids, m)
    ref = sum(torch.nn.functional.cross_entropy((hid[b, t] @ W.t())[None], ids[b, t + 1][None], reduction="sum") for b in range(3) for t in range(int(pl[b]) - 1, int(L[b]) - 1))
    assert abs(float(chunked_ce(h, lab, W, chunk=5, reduction="sum")) - float(ref)) < 1e-3 * max(1.0, abs(float(ref))) and int(m.sum()) == int((L - pl).sum())
    assert all(int(lab[int(m[:b].sum()) + int(L[b] - pl[b]) - 1]) == V - 1 for b in range(3)), "the last target label of every example must be EOS"
    print(f"EOS id {S['EOS']} == pad id {tok.pad_token_id}: masks come from lengths, so EOS is a loss target | loss == full-logits reference | "
          f"direction counts in the {RUN['N']:,} rows: {cnt.to_dict()}")
    assert free_gb(WORK) > 12, f"only {free_gb(WORK):.1f} GB free in /kaggle/working: checkpoints + the final model need about 12 GB"


c19c()


In [ ]:
# C20 - Greedy generation helper for dev evaluation (generate_batch lives in p1_utils)
def eval_generation(model, tag):
    """Greedy decoding of the 50 FLORES-22 dev sentences per direction -> {direction: chrF++}; prints 3 examples each."""
    tok = load_tok()
    gc.collect()
    torch.cuda.empty_cache()  # training leaves reserved-but-free blocks; generation needs room for the KV cache
    rows = S["flores"]
    items = [(r["src"], *split_direction(r["direction"])) for r in rows]
    outs = generate_batch(model, tok, items, batch_size=CONFIG["GEN_BATCH"], max_len=CONFIG["MAX_TOKENS"])
    per = {}
    for d in DIRECTIONS:
        idx = [i for i, r in enumerate(rows) if r["direction"] == d]
        per[d] = score_chrf([outs[i]["hyp"] for i in idx], [rows[i]["ref"] for i in idx], split_direction(d)[1])
        for i in idx[:3]:
            print(f"  [{tag}] {d}\n     src: {rows[i]['src'][:110]}\n     hyp: {outs[i]['hyp'][:110]}\n     ref: {rows[i]['ref'][:110]}")
    print(f"[{tag}] FLORES-dev chrF++:", {d: round(v, 1) for d, v in per.items()}, "| mean", round(float(np.mean(list(per.values()))), 2))
    return per


assert "max_len" in inspect.signature(generate_batch).parameters and "batch_size" in inspect.signature(generate_batch).parameters
print("generate_batch: left padding, greedy, eos=pad=<|endoftext|>, per-example caps (8*src+32 en->X, 0.5*src+32 X->en), order restored")


**Changed (C21a):** (1) `compute_wall` now uses the shared `time_plan` (the same function as C16b): it compares the time left in the session, after the post-training and safety reserves, with the time the run NEEDS (`N / 10.49 examples/s x 1.08`) and refuses to start training when it would end INCOMPLETE; (2) `mb_forward` also returns where the EOS-predicting positions are, so the EOS-token cross-entropy is monitored (`eos_loss` in the train log, `dev_eos_loss` at every dev evaluation). **Why:** a run that cannot finish should fail in minutes, not after hours, and "does the model learn to stop?" must be measurable rather than guessed. **Effect:** no change to the loss, gradients or updates.

In [ ]:
# C21a - Training helpers: wall clock, forward/backward, optimizer step
class OptimizerFailure(Exception):
    pass


TS = {}


def compute_wall(done_steps=0, trained_s=0.0):
    """Training wall cap in seconds of TOTAL training time (it includes `trained_s` already spent before a resume).  It comes from the SAME
    `time_plan` as the check in C16b - the model download / build / validation (the pre-training reserve) has already elapsed here, so it is
    part of the session time and is not subtracted a second time."""
    assert RUN["N"] == CONFIG["N"]
    plan = time_plan("before training", done_steps, trained_s, pre_reserve_h=0.0)
    S["time_plan"] = plan
    require_time(plan)
    return trained_s + 3600.0 * plan["training_window_h"]


def mb_forward(model, mb):
    """Per-target-token CE (fp32 vector) and the direction index of every target token, for one micro-batch."""
    dev = S["M"]["dev"]
    ids, am = mb["input_ids"].to(dev), mb["attention_mask"].to(dev)
    with torch.autocast("cuda", dtype=torch.float16):
        hid = model.model(input_ids=ids, attention_mask=am, use_cache=False).last_hidden_state  # final RMSNorm applied
        mask = target_position_mask(mb["prompt_len"], mb["length"], ids.shape[1]).to(dev)       # from lengths, not pad id
        h, labels = gather_targets(hid, ids, mask)
        lt = chunked_ce(h, labels, model.lm_head.weight, CONFIG["LOSS_CHUNK"], reduction="none")
    eos_pos = (torch.arange(ids.shape[1], device=dev)[None, :] == (mb["length"].to(dev) - 2)[:, None])[mask]  # the position that predicts EOS
    return lt, mb["direction"].to(dev)[mask.nonzero()[:, 0]], eos_pos


def forward_backward(rows, budget):
    """One global batch: sum-CE of every micro-batch / TOTAL target tokens of the global batch, scaled backward."""
    M, bt = S["M"], S["train_batcher"]
    model, scaler = M["model"], M["scaler"]
    denom = float(bt.target_tokens(rows))
    tot = torch.zeros((), device=M["dev"], dtype=torch.float32)
    d_sum = torch.zeros(len(DIRECTIONS), device=M["dev"], dtype=torch.float32)
    d_cnt = torch.zeros(len(DIRECTIONS), device=M["dev"], dtype=torch.float32)
    e_sum = torch.zeros((), device=M["dev"], dtype=torch.float32)
    e_cnt = torch.zeros((), device=M["dev"], dtype=torch.float32)
    for mb in bt.micro_batches(rows, budget):
        lt, td, ep = mb_forward(model, mb)
        loss_sum = lt.sum()
        scaler.scale(loss_sum / denom).backward()
        with torch.no_grad():
            tot += loss_sum.detach()
            d_sum.index_add_(0, td, lt.detach())
            d_cnt.index_add_(0, td, torch.ones_like(lt))
            e_sum += lt.detach()[ep].sum()
            e_cnt += ep.sum()
    TS["eos_loss"] = float(e_sum / e_cnt.clamp(min=1))  # mean CE of the EOS tokens of this global batch (monitoring only)
    per_dir = (d_sum / d_cnt.clamp(min=1)).tolist()
    return float(tot) / denom, per_dir, int(denom)


def optimizer_step(lr):
    """unscale_ -> clip -> step -> update -> zero_grad (scaler skips the step itself when grads are non-finite)."""
    M = S["M"]
    opt, scaler = M["opt"], M["scaler"]
    for g in opt.param_groups:
        g["lr"] = lr
    scaler.unscale_(opt)
    gn = float(torch.nn.utils.clip_grad_norm_(M["params"], CONFIG["CLIP"]))
    try:
        scaler.step(opt)
    except Exception as e:  # e.g. bitsandbytes kernel failure
        raise OptimizerFailure(f"{type(e).__name__}: {e}") from e
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return gn, math.isfinite(gn)


@torch.no_grad()
def dev_loss_eval():
    M, bt = S["M"], S["dev_batcher"]
    torch.cuda.empty_cache()
    M["model"].eval()
    s = torch.zeros(len(DIRECTIONS), device=M["dev"], dtype=torch.float64)
    c = torch.zeros_like(s)
    es = torch.zeros((), device=M["dev"], dtype=torch.float64)
    ec = torch.zeros_like(es)
    for i in range(0, len(bt), 256):
        for mb in bt.micro_batches(list(range(i, min(i + 256, len(bt)))), 8000):
            lt, td, ep = mb_forward(M["model"], mb)
            s.index_add_(0, td, lt.double())
            c.index_add_(0, td, torch.ones_like(lt, dtype=torch.float64))
            es += lt.double()[ep].sum()
            ec += ep.sum()
    M["model"].train()
    M["model"].config.use_cache = False
    per = (s / c.clamp(min=1)).tolist()
    S["dev_eos_loss"] = float(es / ec.clamp(min=1))  # mean CE of the EOS tokens of the dev set: how well "stop here" is learned
    return float(s.sum() / c.sum().clamp(min=1)), dict(zip(DIRECTIONS, per))


**Changed:** (1) the session start time is stored in `/kaggle/working/.session_t0`, so a kernel restart inside the same 12 h session no longer resets the clock; (2) `compute_wall(done_steps, trained_s)` counts training time already spent and only the examples still to be trained; (3) `try_resume` also restores the CUDA RNG, refreshes the OOM ladder for the configuration in use, asserts the trainable set, and on any failure rebuilds pristine weights instead of continuing with half-loaded state. **Why:** the old clock and wall accounting could end a resumed run early, and a half-loaded resume would have trained from corrupted state. **Effect:** none on an uninterrupted run.

In [ ]:
# C21b - Recovery ladder, local checkpoint / resume
def rebuild(n_layers, kind, budget, reason):
    """Free everything, reload PRISTINE base weights from disk, restart training from step 0 (frozen/trained flags rebuilt)."""
    print(f"[recovery] {reason} -> rebuild: layers={n_layers} optimizer={kind} micro-batch tokens={budget}")
    RUN_LOG.log(event="rebuild", reason=reason, n_layers=n_layers, kind=kind, budget=budget)
    free_model(S.get("M"))
    S["M"] = None
    gc.collect()
    torch.cuda.empty_cache()
    S["M"] = build_model(n_layers, kind)
    S["budget"] = budget
    TS.update(step=0, lr_scale=1.0, ema=None, skips=[], skip_total=0, halved_window=False, replayed=0, loss_spikes=0, spike_steps=[], skip_steps_all=[])


def next_rung(reason):
    if not S["rungs"]:
        raise RuntimeError(f"CUDA OOM on every rung of the recovery ladder ({reason}). Reduce MICRO_BATCH_TOKENS or the trained layers.")
    n, budget = S["rungs"].pop(0)
    kind = S["M"]["kind"] if S["M"] else "adamw8bit"
    rebuild(n, kind, budget, reason)


def run_signature():
    """Identity of a training run: every training-relevant setting (hyper-parameters, layers, optimizer, token limit, ...), the pinned
    model + dataset revisions and code identity (config_sig), and the exact pool.  A checkpoint is resumed ONLY if this matches."""
    payload = [config_sig(), RUN["N"], S["prep_manifest"]["sha256"]["pool.parquet"]]
    return hashlib.sha256(json.dumps(payload).encode()).hexdigest()[:16]


def save_local_ckpt(step):
    guard_working(7.0, "local checkpoint (old + new copy coexist briefly)")
    M = S["M"]
    state = {"sig": run_signature(), "step": step, "elapsed": time.time() - TS["t0"], "lr_scale": TS["lr_scale"], "ema": TS["ema"],
             "n_layers": M["n_layers"], "kind": M["kind"], "budget": S["budget"], "skips": TS["skips"], "skip_total": TS["skip_total"],
             "replayed": TS.get("replayed", 0), "loss_spikes": TS.get("loss_spikes", 0), "spike_steps": TS.get("spike_steps", []), "skip_steps_all": TS.get("skip_steps_all", []),
             "trained": {n: p.detach().cpu() for n, p in M["trainable"]}, "opt": M["opt"].state_dict(),
             "scaler": M["scaler"].state_dict(), "rng": {"py": random.getstate(), "np": np.random.get_state(),
                                                         "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all()}}
    atomic_save(os.path.join(CKPT_DIR, "last.pt"), lambda tmp: torch.save(state, tmp))
    print(f"[ckpt] step {step} saved ({os.path.getsize(os.path.join(CKPT_DIR, 'last.pt')) / 1e9:.2f} GB)")
    disk_report("ckpt")


def save_trained_weights():
    """Safety net: the trained layers (fp16, ~1 GB) are written to disk when training ends, BEFORE export, so a failure in the export /
    verification cells can never lose the result of a ~7 h run (the in-memory model is the only other copy)."""
    from safetensors.torch import save_file
    guard_working(1.5, "trained-layers file")
    M, path = S["M"], os.path.join(CKPT_DIR, "trained_layers_fp16.safetensors")
    atomic_save(path, lambda t: save_file({n: p.detach().to(torch.float16).cpu().contiguous() for n, p in M["trainable"]}, t,
                                          metadata={"step": str(TS["step"]), "stopped_by": str(TS.get("stopped_by")), "n_layers": str(M["n_layers"]),
                                                    "run_sig": run_signature()}))
    print(f"[safety] trained layers saved: {path} ({os.path.getsize(path) / 1e9:.2f} GB)")


def try_resume():
    path = os.path.join(CKPT_DIR, "last.pt")
    if not os.path.exists(path):
        return False
    try:
        st = torch.load(path, map_location="cpu", weights_only=False)
        if st["sig"] != run_signature():
            print("[resume] checkpoint belongs to a different run signature - ignored")
            return False
        if (st["n_layers"], st["kind"]) != (S["M"]["n_layers"], S["M"]["kind"]):
            rebuild(st["n_layers"], st["kind"], st["budget"], "resume with the checkpoint's layer count / optimizer")
        M = S["M"]
        for n, p in M["trainable"]:
            p.data.copy_(st["trained"][n])
        M["opt"].load_state_dict(st["opt"])
        M["scaler"].load_state_dict(st["scaler"])
        TS.update(step=st["step"], elapsed_prev=st["elapsed"], lr_scale=st["lr_scale"], ema=st["ema"], skips=st["skips"], skip_total=st["skip_total"],
                  replayed=st.get("replayed", 0), loss_spikes=st.get("loss_spikes", 0), spike_steps=st.get("spike_steps", []), skip_steps_all=st.get("skip_steps_all", []))
        S["budget"] = st["budget"]
        random.setstate(st["rng"]["py"]); np.random.set_state(st["rng"]["np"]); torch.set_rng_state(st["rng"]["torch"])
        torch.cuda.set_rng_state_all(st["rng"]["cuda"])
        S["rungs"] = rungs_after(M["n_layers"], S["budget"])
        assert [n for n, p in M["model"].named_parameters() if p.requires_grad] == [n for n, _ in M["trainable"]], "trainable set changed on resume"
        print(f"[resume] continuing from step {st['step']} (global batch rows {st['step'] * CONFIG['GLOBAL_BATCH']}.. next; nothing skipped, nothing repeated)")
        return True
    except Exception as e:
        print(f"[resume] failed ({type(e).__name__}: {e}) - discarding any half-loaded state and starting from pristine weights at step 0")
        rebuild(S["M"]["n_layers"], S["M"]["kind"], S["budget"], "failed resume")
        return False


In [ ]:
# C21b2 - Final-report printer (used by the end of training if it is incomplete, and by the last cell)
def print_final_report(status, reasons, sm, res=None, overall=None, deviations=(), human="PENDING"):
    """Concise 'PROJECT 1 - FINAL RUN' summary.  res = results.json (per direction), overall = overall.json; both may be None
    (training incomplete -> no evaluation was run)."""
    def f(v, p=2):
        return "n/a" if v is None else f"{v:.{p}f}"

    done, steps = sm.get("steps_done", 0), sm.get("steps", CONFIG["STEPS"])
    complete = done >= steps and sm.get("stopped_by") == "steps"
    sets = (("in22_gen", "IN22-Gen"), ("in22_conv", "IN22-Conv"))
    L = ["", "=" * 78, "PROJECT 1 - FINAL RUN", "", "Training:", f"  N = {CONFIG['N']:,}", f"  steps = {steps:,}",
         f"  completed = {'YES' if complete else 'NO'}   ({done:,} / {steps:,} = {'COMPLETE' if complete else 'INCOMPLETE'})",
         f"  epochs = 1 (examples seen: {sm.get('examples_consumed', 0):,}, epochs completed: {sm.get('epochs_completed')})",
         f"  optimizer updates: {done - sm.get('skipped_steps', 0):,} applied, {sm.get('skipped_steps', 0)} SKIPPED (fraction {sm.get('skipped_step_fraction', 0)}) | "
         f"batches replayed after fp16 overflow: {sm.get('replayed_batches', 0)} | high-loss batches kept (> 3x running average): {sm.get('loss_spike_warnings', 0)} | "
         f"final LR scale: {sm.get('lr_scale_final', 1.0)}", "",
         "Model:", "  Qwen3-1.7B-Base", f"  trained layers = {sm.get('trained_layers', CONFIG['TRAINED_LAYERS'])} + final norm", "  precision = fp16",
         f"  optimizer = {sm.get('optimizer', 'adamw8bit')}", ""]
    if res:
        L += ["Evaluation: IN22-Gen, IN22-Conv (all six directions, greedy decoding)", "", "chrF++  (95% bootstrap CI over test sentences)"]
        for d in DIRECTIONS:
            cells = []
            for s, nm in sets:
                x = res[d].get(s, {})
                ci = x.get("ci95")
                cells.append(f"{nm} {f(x.get('chrf_pp'))} [{f(ci[0]) if ci else 'n/a'}, {f(ci[1]) if ci else 'n/a'}]")
            L.append(f"  {d:<20}" + "   ".join(cells))
        for k, nm in (("in22_gen", "IN22-Gen"), ("in22_conv", "IN22-Conv"), ("all", "both sets")):
            o = (overall or {}).get(k)
            if o:
                ci = o["ci95_pooled"]
                L.append(f"  OVERALL {nm:<10} pooled {f(o['chrf_pp_pooled'])} [{f(ci[0])}, {f(ci[1])}]   (mean of the direction scores: {f(o['chrf_pp_macro_mean'])})")
        L += ["", "BLEU"]
        for d in DIRECTIONS:
            L.append(f"  {d:<20}" + "   ".join(f"{nm} {f(res[d].get(s, {}).get('bleu'))}" for s, nm in sets))
        for k, nm in (("in22_gen", "IN22-Gen"), ("in22_conv", "IN22-Conv"), ("all", "both sets")):
            o = (overall or {}).get(k)
            if o:
                L.append(f"  OVERALL {nm:<10} mean of the direction scores {f(o['bleu_macro_mean'])}")
        L += ["", "Sanity metrics (gates: script >= 0.98, empty+loop+leak <= 0.01, length ratio 0.7-1.4 for >= 95%)",
              f"  {'direction':<20}{'script':>8}{'empty/loop/leak':>17}{'len-ratio ok':>14}{'numerals':>10}{'acronyms':>10}{'lang-ID ok':>12}{'hit cap':>10}{'no EOS':>10}"]
        gates = []
        for d in DIRECTIONS:
            r = res[d]
            fl = r.get("failures") or {}
            bad = fl.get("empty_rate", 0) + fl.get("loop_rate", 0) + fl.get("leak_rate", 0) if fl else None
            L.append(f"  {d:<20}{f(r.get('script_acc'), 3):>8}{f(bad, 4):>17}{f(r.get('len_ratio_ok'), 3):>14}"
                     f"{f(r.get('numeral_preservation'), 3):>10}{f(r.get('acronym_copy_rate'), 3):>10}{f(r.get('lid_lang_acc'), 3):>12}"
                     f"{f(fl.get('unfinished_rate'), 4):>10}{f(fl.get('no_eos_rate'), 4):>10}")
            gates.append((r.get("script_acc") or 0) >= 0.98 and (bad is not None and bad <= 0.01) and (r.get("len_ratio_ok") or 0) >= 0.95)
        L.append(f"  gates passed in {sum(gates)} of {len(gates)} directions (sanity checks, not performance claims)")
        cur = sm.get("curve") or TS.get("curve") or []
        if cur:
            L += ["", "FLORES-22 dev chrF++ at training-progress checkpoints (same run; mean over the six directions)"]
            L += [f"  step {c['step']:>5} ({100 * c['frac']:.0f}%)  {sum(c[d] for d in DIRECTIONS) / len(DIRECTIONS):.2f}" for c in cur]
    else:
        L += ["Evaluation: NOT RUN (training did not complete)"]
    L += ["", f"Human evaluation: {human}", "", f"Final status: {status}"]
    L += [f"  - {r}" for r in reasons]
    if deviations:
        L += ["Deviations from the plan (disclose them with the results):"] + [f"  - {x}" for x in deviations]
    L += ["=" * 78]
    print("\n".join(L))


**Changed:** (1) an ordinary **high-loss batch is no longer discarded** (it used to be dropped when its loss exceeded 3 x the running average): it is logged (`loss_spike_warnings`) and trained on; only genuinely non-finite gradients are ever skipped. (2) A global batch whose fp16 gradients overflow (the scaler skips that update and lowers its scale) is **replayed** — the same rows, up to `NONFINITE_REPLAYS` times — so one update per batch is preserved; replays are counted (`replayed_batches`). (3) A batch that still fails is **skipped and reported** (`skipped_steps`, `skipped_step_fraction`, the step indices): ANY skipped step turns the final status into COMPLETE WITH DEVIATIONS (the old 1% tolerance is gone), as does a reduced LR scale. **Why:** high loss is not an error and dropping hard batches would silently change the training distribution; a skipped update is a real deviation from "3,900 updates". **Effect:** none on a clean run; summary, final report and `final_manifest.json` list all counters.

In [ ]:
# C21c - One optimizer step with OOM / optimizer-failure handling and guards
def do_step(step, rows, lr, budget):
    loss, per_dir, ntok = forward_backward(rows, budget)
    if TS["ema"] is not None and step >= 20 and loss > 3.0 * TS["ema"]:  # an ORDINARY high loss (hard / long / unusual batch) is not an error:
        TS["loss_spikes"] = TS.get("loss_spikes", 0) + 1                    # it is recorded and the batch is trained on like any other
        TS.setdefault("spike_steps", []).append(step)
        RUN_LOG.log(event="high_loss_kept", step=step, loss=loss, ema=TS["ema"])
        if TS["loss_spikes"] <= 20:
            print(f"[note] step {step}: loss {loss:.3f} > 3 x running average {TS['ema']:.3f} - batch KEPT (only non-finite gradients are ever skipped)")
    gn, finite = optimizer_step(lr)
    if finite:
        TS["ema"] = loss if TS["ema"] is None else 0.98 * TS["ema"] + 0.02 * loss
    return {"status": "ok" if finite else "skip_nonfinite", "loss": loss, "per_dir": per_dir, "gn": gn, "ntok": ntok}


def attempt_step(step, rows, lr):
    """Early (first 5 steps) OOM -> walk the ladder from scratch; later OOM -> retry THIS global batch once with a 25% smaller
    budget (no data skipped).  bitsandbytes step failure in the first steps -> torch AdamW with 8 layers."""
    for attempt in range(2):
        failure = None
        try:
            return do_step(step, rows, lr, S["budget"])
        except torch.cuda.OutOfMemoryError:
            failure = "oom"
        except OptimizerFailure as e:
            failure = f"optimizer: {e}"
        S["M"]["opt"].zero_grad(set_to_none=True)
        S["M"]["scaler"] = make_scaler(S["M"]["scaler"].get_scale())  # drop any half-finished unscale_ bookkeeping
        gc.collect()
        torch.cuda.empty_cache()
        if failure.startswith("optimizer"):
            if step >= 5 or S["M"]["kind"] != "adamw8bit":
                raise RuntimeError(f"optimizer failed at step {step}: {failure}")
            rebuild(8, "adamw", 6000, failure)  # reloads pristine weights; frozen/trained flags rebuilt for 8 layers
            S["rungs"] = rungs_after(8, 6000)
            return {"status": "restart"}
        if step < 5:
            next_rung(f"CUDA OOM at step {step}")
            return {"status": "restart"}
        if attempt == 0:
            S["budget"] = int(S["budget"] * 0.75)
            print(f"[recovery] mid-run OOM at step {step}: retrying the same global batch with micro-batch budget {S['budget']}")
            RUN_LOG.log(event="oom_retry", step=step, budget=S["budget"])
    raise RuntimeError(f"CUDA OOM twice at step {step} even after reducing the budget to {S['budget']}")


def note_skip(step):
    TS["skips"].append(step)
    TS["skip_total"] += 1
    TS.setdefault("skip_steps_all", []).append(step)
    recent = [s for s in TS["skips"] if s > step - 50]
    if len(recent) >= 3 and TS["lr_scale"] > 0.125:  # 3 skips within 50 steps: halve LR
        TS["lr_scale"] *= 0.5
        TS["skips"] = []
        print(f"[guard] 3 skipped steps within 50 -> LR scale now {TS['lr_scale']}")


def safe_eval(fn, *args):
    """Dev evaluations must never abort training: log, free memory, restore train mode, carry on."""
    try:
        return fn(*args)
    except Exception as e:
        print(f"WARNING: {fn.__name__} failed ({type(e).__name__}: {str(e)[:120]}); training continues")
        S["eval_failures"] = S.get("eval_failures", 0) + 1
        RUN_LOG.log(event="eval_failed", fn=fn.__name__, error=repr(e))
        gc.collect()
        torch.cuda.empty_cache()
        S["M"]["model"].train()
        S["M"]["model"].config.use_cache = False
        return None


def finish_training(summ):
    """After the loop (also after an error): final FLORES evaluation, learning curve, CSV log, and - when the run stopped before all planned
    steps without an error - a final local checkpoint so nothing trained is lost.  Returns a list of PROBLEMS (run_summary.json must be on disk)."""
    problems = []
    if summ["dev_eval_failures"]:
        print("!" * 78 + f"\n  WARNING: {summ['dev_eval_failures']} dev evaluation(s) FAILED during training - the dev-loss / FLORES "
              "curves are INCOMPLETE (see run_log.jsonl).\n" + "!" * 78)
    try:
        if TS.get("stopped_by") in ("time", "instability") or (TS["step"] not in RUN["GEN_STEPS"] and TS["step"] > 0):
            TS.setdefault("curve", []).append({"step": TS["step"], "frac": TS["step"] / RUN["STEPS"], "final": True,
                                               **eval_generation(S["M"]["model"], "final")})
        if TS.get("curve"):
            atomic_save(os.path.join(TRAIN_DIR, "learning_curve.json"), TS["curve"])
        recs = JsonlLogger(os.path.join(TRAIN_DIR, "train_log.jsonl")).read()
        pd.DataFrame(recs).to_csv(os.path.join(TRAIN_DIR, "train_log.csv"), index=False)
    except Exception as e:
        print(f"WARNING: final evaluation / log conversion failed (non-fatal): {type(e).__name__}: {e}")
    if TS.get("stopped_by") in ("time", "instability") and TS["step"] > 0:
        try:
            save_local_ckpt(TS["step"])
        except Exception as e:
            print(f"WARNING: final checkpoint could not be saved: {type(e).__name__}: {e}")
    if TS["step"] > 0 and not str(TS.get("stopped_by")).startswith("error"):
        try:
            save_trained_weights()
        except Exception as e:
            print(f"WARNING: trained-layers safety file could not be saved: {type(e).__name__}: {e}")
    if not os.path.exists(os.path.join(TRAIN_DIR, "run_summary.json")):
        problems.append("run_summary.json is missing")
    return problems


**Changed (C21d / `lr_schedule`):** the cosine progress used to be `max(step/STEPS, elapsed/WALL)`; it is now exactly `step`. **Why:** the clock term only wins when the run is projected to exceed its cap, so a 3,200-step run that finished inside its cap at a roughly constant rate (about 5.4-6 h of a 7 h cap) was effectively step-driven already — but on a slower machine the old rule would have decayed the LR early and *then still ended INCOMPLETE*; the strict INCOMPLETE policy makes the time term pointless. **Effect:** identical LR trajectory on every machine (peak 3e-5 at step 155, cosine to exactly 3e-6 at the last step); the wall cap now only stops the run.

In [ ]:
# C21d - Main training loop (step-based cosine LR, dev loss, gen checkpoints, local ckpt)
def train_loop():
    WALL = RUN["WALL_S"]
    log = JsonlLogger(os.path.join(TRAIN_DIR, "train_log.jsonl"))
    TS["t0"] = time.time() - TS.get("elapsed_prev", 0.0)
    S["M"]["model"].train()
    last_ckpt = time.time()
    t_win, step = time.time(), TS["step"]
    curve = []
    while step < RUN["STEPS"]:
        elapsed = time.time() - TS["t0"]
        if elapsed >= WALL:
            TS["stopped_by"] = "time"
            break
        lr = lr_schedule(step, RUN["STEPS"], RUN["WARMUP"], CONFIG["LR_PEAK"], CONFIG["LR_MIN"]) * TS["lr_scale"]
        rows = list(range(step * CONFIG["GLOBAL_BATCH"], (step + 1) * CONFIG["GLOBAL_BATCH"]))
        t_step = time.time()
        r = attempt_step(step, rows, lr)
        tries = 0
        while r["status"] == "skip_nonfinite" and tries < CONFIG["NONFINITE_REPLAYS"]:
            # fp16 overflow: the scaler skipped the update (weights and optimizer state untouched) and lowered its scale -> the SAME global batch is
            # computed again (nothing is skipped or repeated; one update per batch is preserved).  Counted in `replayed_batches`.
            tries += 1
            TS["replayed"] = TS.get("replayed", 0) + 1
            RUN_LOG.log(event="replay_nonfinite", step=step, attempt=tries)
            print(f"[guard] step {step}: non-finite gradients (loss-scale overflow, update skipped by the scaler) -> replaying the SAME global batch (attempt {tries})")
            r = attempt_step(step, rows, lr)
        if r["status"] == "restart":
            step, t_win, last_ckpt = TS["step"], time.time(), time.time()  # reset to step 0 on the rebuilt model
            continue
        if r["status"] != "ok":
            note_skip(step)
        dt = time.time() - t_step
        step += 1
        TS["step"] = step
        log.log(step=step, p=step / RUN["STEPS"], lr=lr, eos_loss=TS.get("eos_loss"), status=r["status"], loss=r["loss"], gn=r["gn"], tokens=r["ntok"], tok_s=r["ntok"] / dt,
                ex_s=CONFIG["GLOBAL_BATCH"] / dt, skipped=TS["skip_total"], peak_gb=torch.cuda.max_memory_allocated() / 1e9,
                budget=S["budget"], n_layers=S["M"]["n_layers"], lr_scale=TS["lr_scale"],
                **{f"loss_{d}": v for d, v in zip(DIRECTIONS, r["per_dir"])})
        if step % 50 == 0:
            print(f"step {step}/{RUN['STEPS']} loss {r['loss']:.4f} lr {lr:.2e} gn {r['gn']:.2f} "
                  f"{CONFIG['GLOBAL_BATCH'] * 50 / (time.time() - t_win):.2f} ex/s skipped {TS['skip_total']}")
            t_win = time.time()
        if step % 100 == 0:
            el = time.time() - TS["t0"]
            proj = el + (RUN["STEPS"] - step) * el / max(1, step - TS.get("start_step", 0))
            print(f"   projected finish {proj / 3600:.2f} h of {WALL / 3600:.2f} h cap")
            if proj > WALL * 1.15:
                print("   WARNING: projected finish exceeds the wall cap by >15%: the run will end INCOMPLETE at the cap (the LR schedule is step-based and does not adapt)")
        if step % CONFIG["DEV_LOSS_EVERY"] == 0:
            ev = safe_eval(dev_loss_eval)
            if ev:
                dl, per = ev
                log.log(step=step, dev_loss=dl, dev_eos_loss=S.get("dev_eos_loss"), **{f"dev_{d}": v for d, v in per.items()})
                print(f"   dev loss {dl:.4f} | dev EOS-token loss {S.get('dev_eos_loss', float('nan')):.4f}", {d: round(v, 3) for d, v in per.items()})
            win = [s for s in TS["skips"] if s > step - 500]
            if len(win) / 500 > 0.01:  # >1% skipped in a 500-step window
                if not TS["halved_window"]:
                    TS["lr_scale"] *= 0.5
                    TS["halved_window"] = True
                    print(f"[guard] >1% skipped steps in 500 -> LR halved (scale {TS['lr_scale']})")
                else:
                    TS["stopped_by"] = "instability"
                    break
        if step in RUN["GEN_STEPS"]:
            per_gen = safe_eval(eval_generation, S["M"]["model"], f"step {step}")
            if per_gen:
                curve.append({"step": step, "frac": step / RUN["STEPS"], **per_gen})
                atomic_save(os.path.join(TRAIN_DIR, "learning_curve.json"), curve)
        if time.time() - last_ckpt > 60 * CONFIG["LOCAL_CKPT_MIN"]:
            save_local_ckpt(step)
            last_ckpt = time.time()
    else:
        TS["stopped_by"] = "steps"
    TS["curve"] = curve
    if not TS.get("stopped_by"):
        TS["stopped_by"] = "steps"


def c21():
    TS.update(step=0, lr_scale=1.0, ema=None, skips=[], skip_total=0, halved_window=False, elapsed_prev=0.0, stopped_by=None,
              replayed=0, loss_spikes=0, spike_steps=[], skip_steps_all=[])
    try_resume()
    RUN["WALL_S"] = compute_wall(TS["step"], TS["elapsed_prev"])
    print(f"N={RUN['N']:,} STEPS={RUN['STEPS']} WARMUP={RUN['WARMUP']} | start step {TS['step']} | wall cap {RUN['WALL_S'] / 3600:.2f} h of total training time "
          f"(session elapsed {SESSION.hours():.2f} h)")
    TS["start_step"] = TS["step"]
    TS["stopped_by"] = None
    try:
        train_loop()
    except Exception as e:
        TS["stopped_by"] = f"error: {type(e).__name__}"
        raise
    finally:  # always leave a summary (and the logs) of whatever was trained
        done = TS["step"] * CONFIG["GLOBAL_BATCH"]
        summ = {"N": RUN["N"], "N_nominal": CONFIG["N"], "steps": RUN["STEPS"], "steps_done": TS["step"], "examples_consumed": done,
                "epochs_completed": round(done / RUN["N"], 4), "stopped_by": TS.get("stopped_by"), "trained_layers": S["M"]["n_layers"],
                "optimizer": S["M"]["kind"], "micro_batch_tokens": S["budget"], "wall_cap_h": round(RUN["WALL_S"] / 3600, 3),
                "skipped_steps": TS["skip_total"], "skipped_step_fraction": round(TS["skip_total"] / max(1, TS["step"]), 5), "skipped_step_indices": TS.get("skip_steps_all", [])[:100],
                "replayed_batches": TS.get("replayed", 0), "loss_spike_warnings": TS.get("loss_spikes", 0), "time_plan": S.get("time_plan"), "lr_scale_final": TS["lr_scale"], "session_h_at_end": round(SESSION.hours(), 3),
                "dev_eval_failures": S.get("eval_failures", 0), "complete_pass": TS["step"] >= RUN["STEPS"],
                "run_sig": run_signature(), "planned_layers": CONFIG["TRAINED_LAYERS"], "planned_optimizer": "adamw8bit",
                "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)}
        S["run_summary"] = summ
        atomic_save(os.path.join(TRAIN_DIR, "run_summary.json"), summ)
        print("run summary:", summ)
        S["train_problems"] = finish_training(summ)
    if S["train_problems"]:
        raise RuntimeError("training finished but its summary is not stored: " + "; ".join(S["train_problems"]))
    if not (summ["complete_pass"] and summ["stopped_by"] == "steps"):
        print_final_report("INCOMPLETE", [f"training stopped by '{summ['stopped_by']}' after {summ['steps_done']:,} / {summ['steps']:,} steps"], summ)
        raise RuntimeError(f"TRAINING INCOMPLETE: {summ['steps_done']:,} / {summ['steps']:,} steps (stopped by {summ['stopped_by']}). "
                           "This is not the planned single complete pass, so the model is NOT exported or evaluated as the final result. "
                           f"The last checkpoint is in {CKPT_DIR}.")


c21()


## Part 4 — EXPORT (only after all 3,900 steps)

Reached only if training completed all planned steps (otherwise the training cell has already stopped the notebook with an INCOMPLETE report). Trained fp32 master weights are cast to fp16 (the frozen part already is fp16), embeddings stay **tied**, and the model is saved as one `model.safetensors` (~3.4 GB) in `/kaggle/working/project1_output/final_model/` together with tokenizer, `generation_config` (eos = pad = `<|endoftext|>`, greedy), `prompt_template.json`, a model card and `export_manifest.json` (sha256 + file sizes). **Verification:** the saved model is reloaded from disk with `from_pretrained()` and must reproduce the in-memory fp16 model's greedy outputs on the six hand-written demo prompts *exactly*; it is then loaded once more in a **fresh Python process**. Nothing is uploaded anywhere.

In [ ]:
# C22a - Model card + helpers
import shutil

_SHA_CACHE = {}


def _sha_cached(path):
    st = os.stat(path)
    key = (path, st.st_size, st.st_mtime_ns)  # hashing the 3.4 GB model once per file version is enough
    if key not in _SHA_CACHE:
        _SHA_CACHE[key] = sha256_file(path)
    return _SHA_CACHE[key]


def load_run_summary():
    if S.get("run_summary"):
        return S["run_summary"]
    p = os.path.join(TRAIN_DIR, "run_summary.json")
    return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else {}


def write_model_card(results_md=""):
    s = load_run_summary()
    card = f"""---
license: apache-2.0
base_model: {CONFIG['MODEL_ID']}
language: [en, gu, hi, mr]
pipeline_tag: text-generation
tags: [translation, indic, gujarati, hindi, marathi]
---
# qwen3-1.7b-indic-mt (local Kaggle artifact; not published)

Sentence-level translation between **English and Gujarati / Hindi / Marathi** (six directions), obtained by *fully fine-tuning only the
top {s.get('trained_layers', CONFIG['TRAINED_LAYERS'])} transformer layers + final norm* of `{CONFIG['MODEL_ID']}` (embeddings, tied LM head and lower layers frozen).
No LoRA/PEFT. Plain completion prompt (the base model has no chat template):

```
Translate {{SRC}} to {{TGT}}.
{{SRC}}: <source sentence>
{{TGT}}: <translation><|endoftext|>
```
Names: English, Gujarati, Hindi, Marathi. The prompt ends at `{{TGT}}:` (no trailing space); decode greedily until `<|endoftext|>` and keep the first line.
See `prompt_template.json`. Scope: one sentence (up to ~60 words, 512 tokens incl. prompt). Non-goals: documents, speech, other languages, production serving.

## Training
* Examples seen (single non-repeating pass): **{s.get('examples_consumed', 'n/a')}** of N = {s.get('N', 'n/a')} -> **epochs_completed = {s.get('epochs_completed', 'n/a')}**, stopped_by = **{s.get('stopped_by', 'n/a')}**
  (if `time`, the wall-clock cap ended training early; the LR schedule is step-based, so a clock-stopped model would NOT be annealed).
* steps {s.get('steps_done', 'n/a')}/{s.get('steps', 'n/a')}, global batch 64, fp16 autocast + GradScaler, optimizer {s.get('optimizer', 'n/a')}, LR {CONFIG['LR_PEAK']} -> {CONFIG['LR_MIN']} cosine, custom loop.
* Data: `{CONFIG['DATA_ID']}` (BPCC): human subsets (some CC-BY-4.0, e.g. BPCC-H-Wiki / H-Daily) + BPCC-Mined (CC0), decontaminated against IN22 and FLORES-22 dev. Length distribution of the selection and digit convention: {'matched to IN22 statistics (USE_IN22_LENGTH_MATCH / USE_IN22_DIGIT_AUDIT were enabled; no IN22 sentence was used)' if (CONFIG['USE_IN22_LENGTH_MATCH'] or CONFIG['USE_IN22_DIGIT_AUDIT']) else 'derived from BPCC alone (IN22 only drove decontamination)'}.
  fastText LID (CC-BY-NC-4.0) was used only as a filter. Base model: Apache-2.0 (Qwen3).

## Limitations and caveats
* Bootstrap confidence intervals capture **test-set sampling only**, not training randomness (a single training run).
* Qwen3 pre-training may overlap IN22 / FLORES text; our decontamination only covers the fine-tuning data.
* `epochs_completed` may be below 1.0 if the clock stopped training. Automatic metrics are not a substitute for native-speaker review; human scores are reported, not gated.
* Single sentences only; numbers/URLs/named entities are usually copied but not guaranteed; Hindi vs Marathi confusion is tracked in `eval/`.
{results_md}
"""
    atomic_save(os.path.join(EXPORT_DIR, "README.md"), card)
    return card


def _export_valid():
    """The local export must be complete (file sizes + sha256 of the weights match export_manifest.json) AND from the current run."""
    mp = os.path.join(EXPORT_DIR, "export_manifest.json")
    if not os.path.exists(mp):
        return False
    mf = json.load(open(mp, encoding="utf-8"))
    if mf.get("run_sig") != run_signature():
        return False
    for f, size in mf["files"].items():
        fp = os.path.join(EXPORT_DIR, f)
        if not os.path.exists(fp) or os.path.getsize(fp) != size:
            return False
    return _sha_cached(os.path.join(EXPORT_DIR, "model.safetensors")) == mf["model_sha256"]


def check_export_weights(n_trained):
    """Freezing survives the export: a few frozen tensors are bit-identical to the pinned base model (cast to fp16 exactly as at load time), the
    first / last trained layers differ from it, and the tied head is stored once (no separate lm_head tensor).  Reads 6 tensors only."""
    from safetensors import safe_open
    base, new = os.path.join(MODEL_SRC_DIR, "model.safetensors"), os.path.join(EXPORT_DIR, "model.safetensors")
    if not (os.path.exists(base) and os.path.exists(new)):
        print("export weight check skipped: the base model is not a single model.safetensors file")
        return
    first = 28 - n_trained
    frozen = ["model.embed_tokens.weight", "model.layers.0.self_attn.q_proj.weight", f"model.layers.{first - 1}.mlp.down_proj.weight"]
    trained = [f"model.layers.{first}.mlp.up_proj.weight", "model.layers.27.self_attn.o_proj.weight"]
    with safe_open(base, "pt") as b, safe_open(new, "pt") as e:
        assert "lm_head.weight" not in set(e.keys()), "the tied output head was exported as a separate tensor"
        same = lambda k: torch.equal(b.get_tensor(k).to(torch.float16), e.get_tensor(k))  # noqa: E731
        bad_frozen, bad_trained = [k for k in frozen if not same(k)], [k for k in trained if same(k)]
    if bad_frozen or bad_trained:
        raise RuntimeError(f"export weight check failed: frozen tensors changed {bad_frozen}; trained tensors unchanged {bad_trained}")
    print(f"export weight check OK: frozen layers 0-{first - 1} + embeddings identical to the base model, layers {first}-27 trained, head tied")


FRESH_LOAD = """import sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
m = AutoModelForCausalLM.from_pretrained(sys.argv[1], torch_dtype=torch.float16)
t = AutoTokenizer.from_pretrained(sys.argv[1])
n = sum(p.numel() for p in m.parameters())
assert m.config.tie_word_embeddings and m.lm_head.weight.data_ptr() == m.model.embed_tokens.weight.data_ptr()
assert all(p.dtype == torch.float16 for p in m.parameters())
ids = t(chr(10).join(["Translate English to Hindi.", "English: Hello.", "Hindi:"]), add_special_tokens=False, return_tensors="pt")
try:
    with torch.no_grad():
        out = m(**ids).logits
    assert out.shape[-1] == m.config.vocab_size and bool(torch.isfinite(out).all())
    fwd = "finite logits"
except Exception as e:  # a CPU fp16 forward is a bonus check; the loading above is the requirement
    fwd = "CPU forward skipped (" + type(e).__name__ + ")"
print("fresh process load OK:", n, "parameters, fp16, tied embeddings,", fwd, ", eos", t.eos_token_id)
"""


**Changed:** after saving, `check_export_weights` compares a few tensors of the exported `model.safetensors` with the pinned base model: frozen embeddings / layers must be bit-identical, the first and last trained layers must differ, and no separate `lm_head` tensor may exist (tying). **Why:** it proves from the saved file — not from memory — that the freezing and weight tying of the experiment survived training and export. **Effect:** none on the model; a violation stops the notebook before evaluation.

In [ ]:
# C22c - Export (fp16, tied embeddings), parity check, local verification
def c22():
    from transformers import AutoModelForCausalLM
    tok = load_tok()
    EOS = tok.eos_token_id
    M = S["M"]
    model = M["model"]
    M["opt"], M["scaler"] = None, None
    for p in model.parameters():
        p.grad = None
    gc.collect()
    torch.cuda.empty_cache()
    model.gradient_checkpointing_disable()
    model = model.half().eval()  # trained fp32 master weights -> fp16; embeddings stay tied
    assert model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr()
    assert all(p.dtype == torch.float16 for p in model.parameters()), "export expects all-fp16 parameters"
    model.config.use_cache = True
    for k in ("torch_dtype", "dtype"):
        try:
            setattr(model.config, k, torch.float16)
        except Exception:
            pass
    gcfg = model.generation_config
    gcfg.eos_token_id, gcfg.pad_token_id, gcfg.do_sample, gcfg.num_beams = EOS, EOS, False, 1
    gcfg.temperature = gcfg.top_p = gcfg.top_k = None
    guard_working(4.5, "export")
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
    os.makedirs(EXPORT_DIR, exist_ok=True)
    model.save_pretrained(EXPORT_DIR, safe_serialization=True, max_shard_size="5GB")
    tok.save_pretrained(EXPORT_DIR)
    check_export_weights(load_run_summary().get("trained_layers") or CONFIG["TRAINED_LAYERS"])
    atomic_save(os.path.join(EXPORT_DIR, "prompt_template.json"), {
        "template": "Translate {src_name} to {tgt_name}.\n{src_name}: {src}\n{tgt_name}:", "target_prefix": " ", "stop": EOS_TOKEN,
        "names": {k: v["name"] for k, v in LANGS.items()}, "max_tokens": CONFIG["MAX_TOKENS"], "decoding": "greedy",
        "postprocess": "cut at the first newline, strip, NFC", "tokenize": "tokenizer(prompt, add_special_tokens=False)"})
    write_model_card()
    cfg_json = json.loads(json.dumps(CONFIG, default=str))
    try:
        import yaml
        atomic_save(os.path.join(TRAIN_DIR, "config_used.yaml"), yaml.safe_dump(cfg_json, sort_keys=False, allow_unicode=True))
    except ImportError:
        atomic_save(os.path.join(TRAIN_DIR, "config_used.yaml"), json.dumps(cfg_json, indent=1))  # JSON is valid YAML
    atomic_save(os.path.join(TRAIN_DIR, "env.json"), {"versions": VERSIONS, "python": platform.python_version(), "revisions": S["revs"],
                "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "seed": CONFIG["SEED"],
                "time": time.strftime("%Y-%m-%d %H:%M:%S")})
    atomic_save(os.path.join(TRAIN_DIR, "p1_utils_source.txt"), UTILS_SRC)
    files = sorted(f for f in os.listdir(EXPORT_DIR) if f != "README.md")  # README.md is rewritten with the results in stage 6: not size-verified
    atomic_save(os.path.join(EXPORT_DIR, "export_manifest.json"), {  # lets later stages verify the export is complete and from THIS run
        "run_sig": run_signature(), "model_sha256": _sha_cached(os.path.join(EXPORT_DIR, "model.safetensors")),
        "trained_layers": load_run_summary().get("trained_layers"), "optimizer": load_run_summary().get("optimizer"),  # the ACTUAL (post-recovery) setup
        "files": {f: os.path.getsize(os.path.join(EXPORT_DIR, f)) for f in files}})
    print("exported:", files, f"| {dir_gb(EXPORT_DIR):.2f} GB")
    # ---- parity: in-memory fp16 model vs the model reloaded from disk, greedy, 6 hand-written prompts
    items = [(DEMO_FIXTURES[split_direction(d)[0]], *split_direction(d)) for d in DIRECTIONS]
    mem = generate_batch(model, tok, items, batch_size=6, max_len=CONFIG["MAX_TOKENS"])
    reloaded = AutoModelForCausalLM.from_pretrained(EXPORT_DIR, torch_dtype=torch.float16).to("cuda:0").eval()
    assert reloaded.config.tie_word_embeddings and reloaded.lm_head.weight.data_ptr() == reloaded.model.embed_tokens.weight.data_ptr()
    disk = generate_batch(reloaded, tok, items, batch_size=6, max_len=CONFIG["MAX_TOKENS"])
    same = [a["hyp"] == b["hyp"] for a, b in zip(mem, disk)]
    print("parity (in-memory fp16 vs reloaded):", same)
    S["parity"] = {"identical": all(same), "outputs": [a["hyp"] for a in mem]}
    del reloaded, model
    M["model"] = None  # drop the last reference so the GPU memory is really released
    S["M"] = None
    gc.collect()
    torch.cuda.empty_cache()
    if not all(same):
        raise RuntimeError("parity check failed: reloaded model does not reproduce the in-memory greedy outputs")
    # the final model exists ONLY locally: verify the directory is complete, then load it in a FRESH Python process (CPU, fp16)
    miss = [f for f in ("model.safetensors", "config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
                        "prompt_template.json", "README.md", "export_manifest.json") if not os.path.exists(os.path.join(EXPORT_DIR, f))]
    if miss or not _export_valid():
        raise RuntimeError(f"the local export in {EXPORT_DIR} is incomplete or does not match its manifest (missing: {miss})")
    r = subprocess.run([sys.executable, "-c", FRESH_LOAD, EXPORT_DIR], capture_output=True, text=True, encoding="utf-8", errors="replace",
                       env={**os.environ, "CUDA_VISIBLE_DEVICES": ""}, timeout=1800)
    if r.returncode != 0:
        raise RuntimeError(f"the exported model cannot be loaded in a fresh Python process with from_pretrained(): {r.stderr[-500:]}")
    print(r.stdout.strip())
    mp = os.path.join(EXPORT_DIR, "export_manifest.json")
    emf = json.load(open(mp, encoding="utf-8"))
    emf["verification"] = {"parity_identical": bool(S["parity"]["identical"]), "fresh_process_load": True, "export_weight_check": True}
    atomic_save(mp, emf)  # (README.md and this file are not part of the size-verified file list)
    print(f"final model verified locally: {EXPORT_DIR} ({dir_gb(EXPORT_DIR):.2f} GB)")


c22()


## Part 5 — IN22-Gen / IN22-Conv GENERATION (IN22 is first used for model generation here)

IN22 was already used in Part 1, but only for decontamination and printed statistics; here it is used for the first time to *generate with* and, in Part 6, to *score* the model. Training memory is freed, then **one worker subprocess per visible GPU** (two on a T4 x2 session, ONE on a single T4; `CUDA_VISIBLE_DEVICES=0` / `1`) loads the *saved, hash-verified* model and processes the (direction × set) jobs — batches sorted by length, original order restored, greedy, fp16. Jobs are balanced greedily by size over the workers. Stale `samples_*.jsonl` files are deleted first, a job counts as done **only if its file parses and has exactly the expected ids** (exactly one prediction per benchmark sentence), workers have a timeout derived from the remaining session time, and missing jobs are retried once; incomplete output is never scored.

In [ ]:
# C23b - Generation workers (one per visible GPU), validated outputs
def job_complete(j):
    """A generation job counts as done ONLY if its file parses, has exactly the expected number of rows and exactly the expected ids
    (a stale / partial / foreign file is not accepted just because it exists)."""
    p = os.path.join(EVAL_DIR, f"samples_{j['direction']}_{j['set']}.jsonl")
    if not os.path.exists(p):
        return False
    try:
        ids = [json.loads(l)["id"] for l in open(p, encoding="utf-8") if l.strip()]
    except Exception:
        return False
    return ids == [r["id"] for r in j["rows"]]


def run_workers(jobs, n_gpu, tag):
    """Balance jobs by source size over GPUs, run one subprocess per GPU, return the list of missing job keys."""
    loads, buckets = [0] * n_gpu, [[] for _ in range(n_gpu)]
    for j in sorted(jobs, key=lambda j: -sum(len(r["src"]) for r in j["rows"])):
        k = loads.index(min(loads))
        buckets[k].append(j)
        loads[k] += sum(len(r["src"]) for r in j["rows"])
    procs = []
    for k, bj in enumerate(buckets):
        if not bj:
            continue
        jp, cp = os.path.join(SCRATCH, f"jobs_{tag}{k}.json"), os.path.join(SCRATCH, f"cfg_{tag}{k}.json")
        atomic_save(jp, bj)
        atomic_save(cp, {"jobs_path": jp, "model_dir": EXPORT_DIR, "out_dir": EVAL_DIR, "batch_size": CONFIG["GEN_BATCH"]})
        logf = open(os.path.join(TRAIN_DIR, f"worker_{tag}{k}.log"), "w", encoding="utf-8", errors="replace")
        procs.append((k, popen_utils(sys.executable, "worker", cp, logf, env={"CUDA_VISIBLE_DEVICES": str(k), "PYTHONUNBUFFERED": "1"}), logf))
        print(f"worker {k}: {len(bj)} jobs")
    deadline = time.time() + max(900, (session_left_h() - 1.25) * 3600)  # a hung worker must not eat the rest of the 12 h session
    for k, p, logf in procs:
        try:
            rc = p.wait(timeout=max(1, deadline - time.time()))
        except subprocess.TimeoutExpired:
            p.kill()
            rc = "killed after timeout"
        logf.close()
        if rc != 0:
            print(f"worker {k} ended with {rc}; tail of its log:" + chr(10) + "".join(open(logf.name, encoding="utf-8", errors="replace").readlines()[-8:]))
    return [j for j in jobs if not job_complete(j)]


def c23():
    S["M"] = None
    gc.collect()
    torch.cuda.empty_cache()
    assert _export_valid(), "the exported model in final_model/ does not verify against its manifest"
    bench = ensure_bench()  # first use of IN22 text since decontamination
    jobs = [{"direction": d, "set": s, "rows": [{"id": f"{s}-{i:04d}", "src": a, "ref": b} for i, (a, b) in enumerate(bench[s][d])]}
            for s in ("in22_gen", "in22_conv") for d in DIRECTIONS]
    for f in os.listdir(EVAL_DIR):  # stale predictions of any earlier attempt are never reused
        if f.startswith("samples_"):
            os.remove(os.path.join(EVAL_DIR, f))
    n_gpu = max(1, torch.cuda.device_count())
    print(f"{len(jobs)} jobs, {sum(len(j['rows']) for j in jobs):,} sentences, {n_gpu} GPU worker(s)")
    missing = run_workers(jobs, n_gpu, "a")
    if missing:
        print("retrying missing jobs once:", [(j["direction"], j["set"]) for j in missing])
        missing = run_workers(missing, 1, "r")
    if missing:
        raise RuntimeError(f"generation incomplete: {[(j['direction'], j['set']) for j in missing]}")
    for j in jobs:  # quick sanity chrF++ (the real metrics run in stage 6)
        rows = [json.loads(l) for l in open(os.path.join(EVAL_DIR, f"samples_{j['direction']}_{j['set']}.jsonl"), encoding="utf-8")]
        try:
            print(f"  {j['direction']:<20}{j['set']:<9} n={len(rows):<5} chrF++ {score_chrf([r['hyp'] for r in rows], [r['ref'] for r in rows], split_direction(j['direction'])[1]):.1f}"
                  f"  stopped by EOS/newline/cap {sum(r['stop'] == 'eos' for r in rows)}/{sum(r['stop'] == 'newline' for r in rows)}/{sum(r['stop'] == 'cap' for r in rows)}")
        except Exception as e:
            print("  sanity chrF++ failed:", type(e).__name__)
    assert all(job_complete(j) for j in jobs), "the IN22 generations on disk are incomplete"
    atomic_save(os.path.join(EVAL_DIR, "samples_meta.json"), {"run_sig": run_signature(), "model_sha256": _sha_cached(os.path.join(EXPORT_DIR, "model.safetensors"))})
    print(f"IN22 predictions saved in {EVAL_DIR}: {len(jobs)} files, exactly one prediction per benchmark sentence (ids verified)")
    disk_report("after generation")


c23()


## Part 6 — METRICS (chrF++ primary, BLEU secondary)

Metrics run in a subprocess of this interpreter (`sacrebleu==2.3.1`, `indic-nlp-library` and `fasttext` are installed in C02; the helper source is piped on stdin). **chrF++ and BLEU are mandatory** (overall = all six directions pooled into one corpus per set and over both sets, with a bootstrap CI; the overall BLEU is the mean of the direction scores): if they cannot be computed, or a sample file does not match the benchmark's rows/ids, the cell raises and the notebook stops (nothing incomplete is ever scored or reported). COMET is not part of this notebook.

**Protocol.** chrF++ = `CHRF(word_order=2)`, corpus-level per (direction × set). *English targets:* default tokenisation. *Indic targets:* hypothesis **and** reference are first normalised (indicnlp `IndicNormalizerFactory`) and tokenised (`trivial_tokenize`, `gu`/`hi`/`mr`) with `tok_indic` — the same preprocessing as AI4Bharat's IndicTrans2 evaluation scripts, so the scores are comparable with their published numbers. The signature is logged and the plain (untokenised) score is kept. BLEU = `BLEU(tokenize="none")` on the `tok_indic` text for Indic targets, default for English targets. The 95% bootstrap CI (1,000 resamples, seed 42) uses per-segment sufficient statistics via sacrebleu's private `_extract_corpus_statistics` / `_aggregate_and_compute`; the aggregate over all segments is asserted equal to `corpus_score` (±1e-6) for the pinned version, otherwise 300 resamples with multiprocessing are used. The same resampled indices are applied to hypotheses and references.

In [ ]:
# C24 - Metrics environment check
def c24():
    """chrF++ / BLEU / language-ID run in THIS interpreter (the packages were installed in C02).  Verify what the metrics depend on."""
    import sacrebleu
    assert sacrebleu.__version__ == "2.3.1", (f"sacrebleu {sacrebleu.__version__} is installed but the bootstrap CI uses private APIs verified "
                                              "only for 2.3.1: pip install sacrebleu==2.3.1")
    try:
        from indicnlp.tokenize import indic_tokenize  # noqa: F401  (Indic targets are normalised + tokenised, hypotheses AND references)
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory  # noqa: F401
    except ImportError:
        ok, err = _pip("indic-nlp-library", extra=("--no-deps",))
        assert ok, f"indic-nlp-library is required for the Indic chrF++/BLEU protocol and could not be installed: {err}"
        from indicnlp.tokenize import indic_tokenize  # noqa: F401,F811
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory  # noqa: F401,F811
    try:
        import fasttext  # noqa: F401
    except ImportError:
        print("WARNING: fasttext unavailable - language-ID accuracy and the Hindi/Marathi confusion matrix will be empty (chrF++/BLEU unaffected)")
    S["metrics_py"] = sys.executable
    print("metrics environment OK: sacrebleu", sacrebleu.__version__, "| Indic tokenisation available | interpreter:", sys.executable)


c24()


In [ ]:
# C25 - Run the metrics subprocess (chrF++ / BLEU / bootstrap CI / LID / failure counters)
def run_sub(cmd, cfg, timeout):
    cp = os.path.join(SCRATCH, f"cfg_{cmd}.json")
    atomic_save(cp, cfg)
    r = run_utils(S.get("metrics_py", sys.executable), cmd, cp, timeout=timeout, env={"CUDA_VISIBLE_DEVICES": "0"})
    print(r.stdout[-1800:])
    if r.returncode != 0:
        raise RuntimeError(f"{cmd} failed: {r.stderr[-600:]}")


def c25():
    have = [f for f in os.listdir(EVAL_DIR) if f.startswith("samples_") and f.endswith(".jsonl")]
    assert len(have) == 2 * len(DIRECTIONS), f"{len(have)} of {2 * len(DIRECTIONS)} IN22 prediction files exist in {EVAL_DIR}: refusing to score partial output"
    lid_path = hf_hub_download(CONFIG["LID_ID"], "model.bin")
    mdir = os.path.join(SCRATCH, "metrics_tmp")
    os.makedirs(mdir, exist_ok=True)
    bench = ensure_bench()  # the metrics refuse any sample file that does not have exactly the benchmark's rows (and ids)
    expected = {f"{d}|{s}": len(bench[s][d]) for d in DIRECTIONS for s in ("in22_gen", "in22_conv")}
    # chrF++ / BLEU are the primary results: a failure here is NOT swallowed - the metrics stage is recorded as FAILED
    # (and the end-of-run banner says the results are incomplete).
    run_sub("metrics-base", {"eval_dir": EVAL_DIR, "out_dir": mdir, "lid_path": lid_path, "seed": CONFIG["SEED"], "expected": expected,
                             "boot_n": CONFIG["BOOT_N"], "boot_fallback": CONFIG["BOOT_FALLBACK_N"]}, 3600)
    if not os.path.exists(os.path.join(mdir, "results_base.json")):
        raise RuntimeError("metrics-base finished without writing results_base.json")
    bad = [f"{d}/{s}: {v['error']}" for d, e in json.load(open(os.path.join(mdir, "results_base.json"), encoding="utf-8")).items()
           for s, v in e.items() if isinstance(v, dict) and "error" in v]
    if bad:
        raise RuntimeError(f"chrF++/BLEU could not be computed for {len(bad)} (direction, set) pairs, e.g. {bad[:2]}")


c25()


**Changed (C26a):** `results.md` gains a *Generation diagnostics* table (mean / median / p95 / max generated tokens, share of rows ended by EOS, by a new line, or by the length cap, mean hypothesis vs reference length) and `metrics/generation_diagnostics.csv`; the gate table shows "no EOS / hit cap". **Why:** this is the evidence needed to decide whether English->Indic failures come from the length cap, from weak EOS learning, or from verbose output — it is computed from the very outputs that are scored. **Effect:** none on scores.

In [ ]:
# C26a - Assemble results.json / results.md, worst-50, human sheets
def c26a():
    mdir = os.path.join(SCRATCH, "metrics_tmp")
    res = json.load(open(os.path.join(mdir, "results_base.json"), encoding="utf-8"))
    overall = json.load(open(os.path.join(mdir, "overall.json"), encoding="utf-8"))
    S["overall"] = overall
    atomic_save(os.path.join(METRICS_DIR, "overall.json"), overall)
    meta = {"chrf_signature_example": next(
        (res[d][s]["chrf_signature"] for d in DIRECTIONS for s in ("in22_gen", "in22_conv") if "chrf_signature" in res[d].get(s, {})), None)}
    atomic_save(os.path.join(METRICS_DIR, "results.json"), res)
    atomic_save(os.path.join(METRICS_DIR, "results_meta.json"), meta)
    segs = pd.read_json(os.path.join(mdir, "segments.jsonl"), lines=True)
    samples = pd.concat([pd.read_json(os.path.join(EVAL_DIR, f"samples_{d}_{s}.jsonl"), lines=True) for d in DIRECTIONS for s in ("in22_gen", "in22_conv")])
    full = segs.merge(samples[["id", "direction", "src", "ref", "hyp"]], on=["id", "direction"])
    human_meta = {}
    for d in DIRECTIONS:
        full[full.direction == d].nsmallest(50, "chrf_sent")[["id", "set", "chrf_sent", "src", "ref", "hyp"]].to_csv(
            os.path.join(METRICS_DIR, f"worst50_{d}.csv"), index=False, encoding="utf-8-sig")
        n = CONFIG["HUMAN_N_EN2X"] if d.startswith("eng_Latn") else CONFIG["HUMAN_N_X2EN"]
        sub, parts = full[full.direction == d], []
        groups = [g for _, g in sub.groupby(["set", "bucket"])]  # stratified random: set x source-length bucket, proportional
        for g, k in zip(groups, largest_remainder([len(g) for g in groups], n)):
            parts.append(g.sample(min(k, len(g)), random_state=CONFIG["SEED"]))
        picked = pd.concat(parts)
        if len(picked) < min(n, len(sub)):  # a stratum was smaller than its quota: top up at random so the sheet has exactly n rows
            rest = sub[~sub.id.isin(picked.id)]
            picked = pd.concat([picked, rest.sample(min(n, len(sub)) - len(picked), random_state=CONFIG["SEED"])])
        assert len(picked) == min(n, len(sub)), f"human sheet for {d} has {len(picked)} rows, expected {min(n, len(sub))}"
        human_meta[d] = {"requested": n, "rows": len(picked), "by_set": picked["set"].value_counts().to_dict()}
        sheet = picked[["id", "direction", "src", "hyp", "ref"]].rename(columns={"src": "source", "hyp": "hypothesis", "ref": "reference"})
        sheet["adequacy_1_5"], sheet["fluency_1_5"], sheet["notes"] = "", "", ""
        sheet.to_csv(os.path.join(HUMAN_DIR, f"human_eval_{d}.csv"), index=False, encoding="utf-8-sig")
    atomic_save(os.path.join(HUMAN_DIR, "human_eval_meta.json"), human_meta)
    fs = {d: {"failures": res[d].get("failures"), "script_acc": res[d].get("script_acc"), "len_ratio_ok": res[d].get("len_ratio_ok"),
              "lid_lang_acc": res[d].get("lid_lang_acc")} for d in DIRECTIONS}
    atomic_save(os.path.join(METRICS_DIR, "failure_summary.json"), fs)
    atomic_save(os.path.join(METRICS_DIR, "lid_confusion.json"), json.load(open(os.path.join(mdir, "lid_confusion.json"), encoding="utf-8")))
    shutil.copy2(os.path.join(mdir, "segments.jsonl"), os.path.join(METRICS_DIR, "segments.jsonl"))
    S["results"], S["segs"] = res, full
    def fmt(v, p=2):
        return "n/a" if v is None else f"{v:.{p}f}"

    rows, gates = [], []
    for d in DIRECTIONS:
        r = res[d]
        for s in ("in22_gen", "in22_conv"):
            x = r.get(s, {})
            ci = x.get("ci95")
            rows.append(f"| {d} | {s} | {x.get('n', 'n/a')} | {fmt(x.get('chrf_pp'))} | {('[' + fmt(ci[0]) + ', ' + fmt(ci[1]) + ']') if ci else 'n/a'} | "
                        f"{fmt(x.get('bleu'))} |")
        fl = r.get("failures") or {}
        bad = (fl.get("empty_rate", 0) + fl.get("loop_rate", 0) + fl.get("leak_rate", 0)) if fl else None
        gates.append(f"| {d} | {fmt(r.get('script_acc'), 3)} | {fmt(bad, 4)} | {fmt(r.get('len_ratio_ok'), 3)} | {fmt(r.get('lid_lang_acc'), 3)} | "
                     f"{fmt(fl.get('no_eos_rate'), 4)} / {fmt(fl.get('unfinished_rate'), 4)} |")
    gen_rows = [{"direction": d, "set": s, **{k: v for k, v in res[d][s]["generation"].items()}} for d in DIRECTIONS for s in ("in22_gen", "in22_conv")
                if isinstance(res[d].get(s), dict) and "generation" in res[d][s]]
    gdf = pd.DataFrame(gen_rows)
    gdf.to_csv(os.path.join(METRICS_DIR, "generation_diagnostics.csv"), index=False)
    gen_md = ("`" * 3 + chr(10) + gdf.round(3).to_string(index=False) + chr(10) + "`" * 3) if len(gdf) else "n/a"  # (to_markdown needs the optional tabulate package)
    md = ("## Results (IN22, greedy)\n\n| direction | set | n | chrF++ | 95% CI | BLEU |\n|---|---|---|---|---|---|\n" + "\n".join(rows) +
          "\n\n### Sanity gates (suggested: script >= 0.98, empty+loop+leak <= 0.01, length-ratio-ok >= 0.95)\n\n| direction | script acc | empty+loop+leak rate | len ratio ok | LID lang acc | no EOS (new line instead) / hit length cap |\n|---|---|---|---|---|---|\n" +
          "\n".join(gates) + "\n\n### Generation diagnostics (generated tokens; how decoding ended: eos / newline / cap)\n\n" + gen_md +
          f"\n\nchrF++ signature: `{meta['chrf_signature_example']}`.\n"
          "\n**Human evaluation: sheets were generated (`human_eval_*.csv`, adequacy/fluency 1-5 columns empty); human scores are PENDING - "
          "no human result is claimed here.**\n")
    atomic_save(os.path.join(METRICS_DIR, "results.md"), md)
    S["results_md"] = md
    print(md)


c26a()


In [ ]:
# C26b - Learning curve, training curves, confusion plot, model card with results
PAL = ["#2a6fb0", "#d9822b", "#3a9d6e", "#9b59b6", "#c0392b", "#7f8c8d"]


def style(ax, title, xl, yl):
    ax.set_title(title, loc="left", fontsize=10, fontweight="bold")
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.grid(axis="y", alpha=0.25)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


def c26b():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    pdir = os.path.join(METRICS_DIR, "plots")
    os.makedirs(pdir, exist_ok=True)
    curve_p = os.path.join(TRAIN_DIR, "learning_curve.json")
    if os.path.exists(curve_p):
        cv = pd.DataFrame(json.load(open(curve_p)))
        cv.to_csv(os.path.join(METRICS_DIR, "learning_curve.csv"), index=False)
        fig, ax = plt.subplots(figsize=(6.4, 3.6))
        for c, d in zip(PAL, DIRECTIONS):
            ax.plot(cv["step"], cv[d], marker="o", color=c, label=d)
        style(ax, "FLORES-22 dev chrF++ at training-progress checkpoints (50 sent./direction)", "optimizer step (last point = end of training)", "chrF++")
        ax.legend(fontsize=7, frameon=False, ncol=2)
        fig.tight_layout()
        fig.savefig(os.path.join(pdir, "learning_curve.png"), dpi=130)
        plt.close(fig)
        up = {d: bool(cv[d].is_monotonic_increasing) for d in DIRECTIONS}
        print("learning curve rises monotonically (per direction; a diagnostic only, translation quality may fluctuate - NOT a pass/fail gate):", up)
    log_p = os.path.join(TRAIN_DIR, "train_log.csv")
    if os.path.exists(log_p):
        tl = pd.read_csv(log_p)
        st = tl[tl["loss"].notna()] if "loss" in tl else tl
        fig, axs = plt.subplots(2, 2, figsize=(9, 5.4))
        axs[0, 0].plot(st["step"], st["loss"].rolling(20, min_periods=1).mean(), color=PAL[0])
        if "dev_loss" in tl:
            dv = tl[tl["dev_loss"].notna()]
            axs[0, 0].plot(dv["step"], dv["dev_loss"], "o-", color=PAL[1])
        style(axs[0, 0], "loss (train smoothed, dev dots)", "step", "CE / token")
        axs[0, 1].plot(st["step"], st["lr"], color=PAL[2])
        style(axs[0, 1], "learning rate", "step", "lr")
        axs[1, 0].plot(st["step"], st["gn"].clip(upper=st["gn"].quantile(0.99)), color=PAL[3], lw=0.8)
        style(axs[1, 0], "grad norm", "step", "norm")
        axs[1, 1].plot(st["step"], st["ex_s"].rolling(20, min_periods=1).mean(), color=PAL[4])
        style(axs[1, 1], "throughput", "step", "examples / s")
        fig.tight_layout()
        fig.savefig(os.path.join(pdir, "training_curves.png"), dpi=130)
        plt.close(fig)
    conf_p = os.path.join(METRICS_DIR, "lid_confusion.json")
    if os.path.exists(conf_p):
        conf = json.load(open(conf_p))
        labs = ["hin_Deva", "mar_Deva", "guj_Gujr", "eng_Latn"]
        rows = list(DIRECTIONS)  # every direction: the predicted language of the outputs (Hindi vs Marathi is the interesting pair)
        mat = np.array([[conf.get(d, {}).get(l, 0) for l in labs] for d in rows], dtype=float)
        mat = mat / np.maximum(mat.sum(1, keepdims=True), 1)
        fig, ax = plt.subplots(figsize=(6, 3.6))
        im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(labs)), [l.split("_")[0] for l in labs])
        ax.set_yticks(range(len(rows)), rows, fontsize=7)
        for i in range(len(rows)):
            for j in range(len(labs)):
                ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", color="white" if mat[i, j] > 0.5 else "black", fontsize=8)
        ax.set_title("LID of the outputs, per direction (rows) vs predicted language (columns)", loc="left", fontsize=9, fontweight="bold")
        fig.colorbar(im, ax=ax, fraction=0.04)
        fig.tight_layout()
        fig.savefig(os.path.join(pdir, "lid_confusion.png"), dpi=130)
        plt.close(fig)
    write_model_card("\n" + S.get("results_md", ""))
    print(f"plots, learning curve and the model card with the results are saved locally ({pdir}; {os.path.join(EXPORT_DIR, 'README.md')})")


c26b()


## Part 7 — INFERENCE DEMO

`translate("The government has announced a new scheme…", "English", "Hindi")` builds `Translate English to Hindi.\nEnglish: …\nHindi:`, generates greedily until EOS and returns the single translated line. The class below uses exactly the prompt/decoding code path of training-time evaluation and the IN22 workers (`p1_utils.generate_batch`). Longer inputs are split into sentences on `(?<=[.!?।])\s+`, translated in a batch and re-joined; empty input returns `""`; a sentence that does not fit the 512-token window raises a clear error; an unknown direction raises `ValueError`.

In [ ]:
# C27 - Translator class
class Translator:
    NAMES = ["English", "Gujarati", "Hindi", "Marathi"]
    SPLIT = re.compile(r"(?<=[.!?।])\s+")

    def __init__(self, model_dir=None, device=None):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        model_dir = model_dir or EXPORT_DIR
        device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.float16).to(device).eval()
        self.max_tokens = CONFIG["MAX_TOKENS"]

    def _codes(self, src, tgt):
        ok = src in self.NAMES and tgt in self.NAMES and src != tgt and "English" in (src, tgt)
        if not ok:
            raise ValueError(f"unsupported direction {src!r} -> {tgt!r}. Supported: English <-> Gujarati/Hindi/Marathi "
                             f"(names: {self.NAMES})")
        return NAME2CODE[src], NAME2CODE[tgt]

    def translate_detailed(self, texts, src, tgt, batch_size=16):
        """texts: str or list[str] -> list of dicts {output, n_gen_tok, seconds}; original order restored."""
        sc, tc = self._codes(src, tgt)
        single = isinstance(texts, str)
        texts = [texts] if single else list(texts)
        units, owner = [], []
        for i, t in enumerate(texts):
            for u in (x.strip() for x in self.SPLIT.split(norm_text(t))):
                if u:
                    units.append(u)
                    owner.append(i)
        for u in units:  # over-length guard: the prompt must leave room for a translation
            n = len(self.tok(build_prompt(src, tgt, u), add_special_tokens=False)["input_ids"])
            if n > self.max_tokens - 32:
                raise ValueError(f"input segment needs {n} prompt tokens (limit {self.max_tokens - 32}); split it into shorter sentences")
        outs = generate_batch(self.model, self.tok, [(u, sc, tc) for u in units], batch_size=batch_size, max_len=self.max_tokens) if units else []
        res = [{"output": "", "n_gen_tok": 0, "seconds": 0.0} for _ in texts]
        for o, i in zip(outs, owner):
            res[i]["output"] = (res[i]["output"] + " " + o["hyp"]).strip()
            res[i]["n_gen_tok"] += o["n_gen_tok"]
            res[i]["seconds"] += o["seconds"]
        return res

    def translate(self, texts, src, tgt, batch_size=16):
        out = [r["output"] for r in self.translate_detailed(texts, src, tgt, batch_size)]
        return out[0] if isinstance(texts, str) else out


def is_source_copy(src_text, out_text):
    from sacrebleu.metrics import CHRF
    return out_text.strip() == src_text.strip() or (bool(out_text) and CHRF(word_order=2).sentence_score(out_text, [src_text]).score >= 80.0)


def c27():
    S["translator"] = Translator()
    print("translator ready:", EXPORT_DIR)


c27()


In [ ]:
# C28 - Demo table for all six directions + failure modes
def c28():
    from sacrebleu.metrics import CHRF
    T = S["translator"]
    rows = []
    for fx_name, fx in (("fixture 1", DEMO_FIXTURES), ("fixture 2 (Aadhaar)", DEMO_FIXTURES_2)):
        for d in DIRECTIONS:
            sc, tc = split_direction(d)
            r = T.translate_detailed(fx[sc], LANGS[sc]["name"], LANGS[tc]["name"])[0]
            chrf = CHRF(word_order=2).sentence_score(r["output"], [fx[tc]]).score
            rows.append({"fixture": fx_name, "direction": d, "input": fx[sc], "output": r["output"], "generated tokens": r["n_gen_tok"],
                         "seconds": round(r["seconds"], 2), "chrF++ vs hand-written ref (illustration only)": round(chrf, 1)})
    pd.set_option("display.max_colwidth", 80, "display.width", 250)
    tab = pd.DataFrame(rows)
    print(tab.to_string(index=False))
    print("\nChrF++ here is an illustration on hand-written fixtures, NOT test data; native-speaker review is needed.")
    atomic_save(os.path.join(METRICS_DIR, "demo_table.csv"), tab.to_csv(index=False))
    print("\n--- failure modes ---")
    print("empty input        ->", repr(T.translate("", "English", "Hindi")), "| whitespace ->", repr(T.translate("   ", "English", "Hindi")))
    for label, fn in (("over-length input   ", lambda: T.translate("word " * 700, "English", "Hindi")),
                      ("wrong direction     ", lambda: T.translate("Hello.", "Hindi", "Marathi")),
                      ("unknown language    ", lambda: T.translate("Hello.", "English", "French"))):
        try:
            print(label, "->", fn())
        except ValueError as e:
            print(label, "-> ValueError:", e)
    hin = DEMO_FIXTURES["hin_Deva"]
    out = T.translate(hin, "English", "Hindi")  # a Hindi sentence declared as ENGLISH: shows what a wrong source-language label does
    print("wrong source label  -> Hindi text declared as English, output:", out, "| output flagged as a copy of the input:", is_source_copy(hin, out))


c28()


In [ ]:
# C30a - Final status logic
def final_status(sm, res):
    """(status, reasons, deviations).  COMPLETE only if ALL planned steps ran as one complete pass on the nominal N with the planned 10 layers /
    8-bit AdamW and the IN22 evaluation + chrF++/BLEU results exist for all six directions and both sets.  Any deviation from the plan (recovery
    fallback, failed monitoring evaluations, ANY skipped optimizer step, a reduced LR scale) gives COMPLETE WITH DEVIATIONS - never plain COMPLETE."""
    why, dev = [], []
    if not sm:
        why.append("no training run_summary.json")
    else:
        if not (sm.get("complete_pass") and sm.get("stopped_by") == "steps"):
            why.append(f"training stopped by {sm.get('stopped_by')!r} after {sm.get('steps_done')} / {sm.get('steps')} steps: NOT the planned complete single pass")
        if sm.get("N") != CONFIG["N"]:
            why.append(f"N is {sm.get('N')}, not the required {CONFIG['N']}")
        if sm.get("trained_layers") != CONFIG["TRAINED_LAYERS"] or sm.get("optimizer") != "adamw8bit":
            dev.append(f"recovery fallback changed the experiment: trained_layers={sm.get('trained_layers')}, optimizer={sm.get('optimizer')} "
                       f"(planned {CONFIG['TRAINED_LAYERS']} layers, 8-bit AdamW)")
        if sm.get("dev_eval_failures"):
            dev.append(f"{sm['dev_eval_failures']} monitoring evaluation(s) failed during training (the dev-loss / FLORES curves are incomplete)")
        if sm.get("skipped_steps", 0) > 0:  # no tolerance: a skipped update means a batch was seen without any optimisation step
            dev.append(f"{sm['skipped_steps']} of {sm.get('steps_done')} optimizer steps were SKIPPED (fraction {sm.get('skipped_step_fraction')}, steps "
                       f"{(sm.get('skipped_step_indices') or [])[:20]}): those batches were consumed without a weight update")
        if sm.get("lr_scale_final", 1.0) != 1.0:
            dev.append(f"the instability guard reduced the learning rate (final LR scale {sm.get('lr_scale_final')})")
        if sm.get("steps_done") and sm.get("steps_done") != sm.get("steps"):
            why.append(f"{sm.get('steps_done')} of {sm.get('steps')} steps done")
    ver = (json.load(open(os.path.join(EXPORT_DIR, "export_manifest.json"), encoding="utf-8")).get("verification") or {}) if os.path.exists(os.path.join(EXPORT_DIR, "export_manifest.json")) else {}
    if not (ver.get("parity_identical") and ver.get("fresh_process_load")):
        why.append("the export verification (in-memory vs reloaded greedy parity, fresh-process load) is not recorded as passed")
    missing = [f"samples_{d}_{s}.jsonl" for d in DIRECTIONS for s in ("in22_gen", "in22_conv") if not os.path.exists(os.path.join(EVAL_DIR, f"samples_{d}_{s}.jsonl"))]
    if missing:
        why.append(f"{len(missing)} of 12 IN22 prediction files missing")
    bad = [f"{d}/{s}" for d in DIRECTIONS for s in ("in22_gen", "in22_conv") if not isinstance((res or {}).get(d, {}).get(s), dict) or "chrf_pp" not in res[d][s]]
    if bad:
        why.append(f"chrF++/BLEU missing for {len(bad)} (direction, set) pairs")
    return ("INCOMPLETE" if why else "COMPLETE WITH DEVIATIONS" if dev else "COMPLETE"), why, dev


In [ ]:
# C30b - Manifests (final_manifest.json and copies of the small run records)
def build_final_manifest(status, why, dev, sm, res, overall):
    """Everything needed to identify this run: revisions, data / model hashes, configuration, N / steps, layers, optimizer, seed, counts, results."""
    def jl(p):
        return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else None

    prep_mf = jl(os.path.join(PREP_DIR, "manifest.json")) or {}
    exp_mf = jl(os.path.join(EXPORT_DIR, "export_manifest.json")) or {}
    sets = ("in22_gen", "in22_conv")
    return {
        "final_status": status, "reasons": why, "deviations": dev, "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "code_version": CONFIG["CODE_VERSION"], "helper_sha256": hashlib.sha256(UTILS_SRC.encode("utf-8")).hexdigest(), "config_sig": config_sig(),
        "model": {"base_model_id": CONFIG["MODEL_ID"], "base_model_revision": S["revs"].get("model"), "final_model_dir": EXPORT_DIR,
                  "final_model_sha256": exp_mf.get("model_sha256"), "trained_layers": exp_mf.get("trained_layers") or sm.get("trained_layers"),
                  "optimizer": exp_mf.get("optimizer") or sm.get("optimizer"), "precision": "fp16"},
        "data": {"bpcc_id": CONFIG["DATA_ID"], "bpcc_revision": S["revs"].get("data"), "bpcc_file_manifest_sha": S["revs"].get("bpcc_files"),
                 "processed_pool_sha256": (prep_mf.get("sha256") or {}).get("pool.parquet"),
                 "processed_manifest_sha256": sha256_file(os.path.join(PREP_DIR, "manifest.json")) if prep_mf else None},
        "training": {"N": sm.get("N"), "N_nominal": CONFIG["N"], "steps": sm.get("steps"), "steps_done": sm.get("steps_done"),
                     "global_batch": CONFIG["GLOBAL_BATCH"], "epochs": 1, "epochs_completed": sm.get("epochs_completed"), "stopped_by": sm.get("stopped_by"),
                     "skipped_steps": sm.get("skipped_steps"), "skipped_step_fraction": sm.get("skipped_step_fraction"), "skipped_step_indices": sm.get("skipped_step_indices"),
                     "replayed_batches": sm.get("replayed_batches"), "loss_spike_warnings": sm.get("loss_spike_warnings"), "lr_scale_final": sm.get("lr_scale_final"),
                     "time_plan": sm.get("time_plan"), "seed": CONFIG["SEED"], "configuration": {k: CONFIG[k] for k in sorted(_SIG_KEYS)}},
        "evaluation": {"counts": {d: {s: (res or {}).get(d, {}).get(s, {}).get("n") for s in sets} for d in DIRECTIONS},
                       "metrics": {d: {s: {k: res[d][s].get(k) for k in ("chrf_pp", "bleu", "ci95")} for s in sets if isinstance(res[d].get(s), dict)} for d in (res or {})},
                       "overall": overall}}


def write_manifests(status, why, dev, sm, res, overall):
    atomic_save(os.path.join(MAN_DIR, "final_manifest.json"), build_final_manifest(status, why, dev, sm, res, overall))
    atomic_save(os.path.join(MAN_DIR, "final_status.json"), {"final_status": status, "reasons": why, "deviations": dev, "run_summary": sm})
    for src, dst in ((os.path.join(PREP_DIR, "manifest.json"), "data_manifest.json"), (os.path.join(EXPORT_DIR, "export_manifest.json"), "export_manifest.json"),
                     (os.path.join(TRAIN_DIR, "run_summary.json"), "run_summary.json"), (os.path.join(TRAIN_DIR, "config_used.yaml"), "config_used.yaml"),
                     (os.path.join(TRAIN_DIR, "env.json"), "env.json"), (os.path.join(TRAIN_DIR, "tokenizer_report.json"), "tokenizer_report.json"),
                     (os.path.join(TRAIN_DIR, "p1_utils_source.txt"), "p1_utils_source.txt")):
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(MAN_DIR, dst))


In [ ]:
# C30 - Final report
def c30():
    try:  # the exported model must load from disk (never allowed to hide the report below)
        from transformers import AutoConfig, AutoTokenizer
        miss = [f for f in ["model.safetensors", "export_manifest.json", "config.json", "generation_config.json", "prompt_template.json", "README.md", "tokenizer.json"]
                if not os.path.exists(os.path.join(EXPORT_DIR, f))]
        cfg = AutoConfig.from_pretrained(EXPORT_DIR)
        _ = AutoTokenizer.from_pretrained(EXPORT_DIR)
        print(f"final_model/: missing files {miss or 'none'} | layers={cfg.num_hidden_layers} tie_word_embeddings={cfg.tie_word_embeddings}")
    except Exception as e:
        print(f"final model check raised {type(e).__name__}: {e}")
    sm = load_run_summary()
    cp = os.path.join(TRAIN_DIR, "learning_curve.json")
    sm = dict(sm, curve=json.load(open(cp, encoding="utf-8")) if os.path.exists(cp) else TS.get("curve"))
    rp, op = os.path.join(METRICS_DIR, "results.json"), os.path.join(METRICS_DIR, "overall.json")
    res = json.load(open(rp, encoding="utf-8")) if os.path.exists(rp) else None
    overall = json.load(open(op, encoding="utf-8")) if os.path.exists(op) else None
    status, why, dev = final_status(sm, res)
    sheets = [os.path.join(HUMAN_DIR, f"human_eval_{d}.csv") for d in DIRECTIONS]
    human = "PENDING (human_eval/*.csv generated, adequacy/fluency columns left blank)" if all(os.path.exists(p) for p in sheets) else "NOT GENERATED"
    write_manifests(status, why, dev, sm, res, overall)
    tl_p = os.path.join(TRAIN_DIR, "train_log.csv")
    if os.path.exists(tl_p):
        print("median training throughput:", round(float(pd.read_csv(tl_p)["ex_s"].median()), 2), "examples/s")
    S["translator"] = None
    S["M"] = None
    gc.collect()
    torch.cuda.empty_cache()
    print_final_report(status, why, sm, res, overall, dev, human)
    for d in os.listdir(OUT_DIR):
        if os.path.isdir(os.path.join(OUT_DIR, d)) and not os.listdir(os.path.join(OUT_DIR, d)):
            os.rmdir(os.path.join(OUT_DIR, d))
    print("outputs:", OUT_DIR, "->", sorted(os.listdir(OUT_DIR)))
    disk_report("end")


c30()


## REVIEW REPORT — static code review performed before delivery

**How it was checked.** The authoring environment has no GPU and no access to the gated data, so the notebook was **not run end to end**. It was checked by reading every cell, by `ast.parse` of every code cell, by pyflakes over the concatenated notebook, and by searches for removed constructs (stage framework, Hugging Face write calls, COMET, Kaggle-Dataset code). The pure helper functions (prompt / loss alignment, batching, LR schedule, tokenisation, metrics helpers, `waterfill`, ...) carry synthetic-data unit tests that run in C05e when the notebook starts; there is **no smoke test, trial run or probe of the real model**. Earlier review rounds (external reviews, checked claim by claim) produced the engineering already in the code: the real BPCC layout handling, decontamination, exact-N enforcement, target-only loss with EOS included and a length-based mask, validated generation files, mandatory chrF++/BLEU with bootstrap CIs.

| # | Item | Verdict | Evidence |
|---|---|---|---|
| 1 | Syntax | **PASS** | every code cell is `ast.parse`d at build time; pyflakes over the concatenated notebook reports no undefined names (remaining notes: intentional import probes) |
| 2 | Cross-cell consistency | **PASS** | table below (AST-generated); one linear order, every cell uses only names defined in earlier cells or in functions called later; CONFIG keys, directories and file names are used consistently |
| 3 | Index logic | **PASS** | `first = 28 - n_layers` → `layers[18:28]` = layers 19-28 for n=10 (parameter audit asserts ~503M trainable ±2%); `STEPS == N//64`, `WARMUP == round(0.04*STEPS)` asserted in C03; loss alignment checked by unit tests against a full-logits reference (value + gradient) and again on synthetic tensors with `pad == EOS` in C19c; count of loss tokens = Σ(T_i − p_i), last label EOS (C18 + unit test, pad id == EOS) |
| 4 | Shapes / dtypes | **PASS** (GPU dtype behaviour **NOT-STATICALLY-CHECKABLE**) | masks from lengths; chunked CE equals full-logits CE (value + gradient); logits `.float()` before CE; trained params fp32 / frozen fp16 asserted; export asserts all-fp16 and tied embeddings, reload keeps the tie |
| 5 | Optimizer / scaler order | **PASS** | `optimizer_step`: scale→backward (per micro-batch) → `unscale_` → clip → `step` → `update` → `zero_grad`; optimizer holds exactly the trainable params (id-set equality asserted); no frozen param has `requires_grad`; weight decay only on `ndim>=2` |
| 6 | Numerics | **PASS** (note) | denominator = total target tokens of the global batch (invariant to micro-batch grouping, tested); the LR schedule is a pure function of the optimizer step (unit-tested: warm-up, peak at step 155, monotone cosine, exactly LR_MIN at the last step, clamped to [LR_MIN, LR_PEAK]; recomputed in C19c). **Note:** linear warm-up would give values below `LR_MIN` for the first 15 steps; they are clamped to `LR_MIN` (0.4% of training, harmless) |
| 7 | Data logic | **PASS** | pool/reserve/dev disjointness + dev English keys absent from the pool asserted; dedup and decontamination (exact + 8-gram, both sides) re-checked on the final stored pool in C19c (no verbatim IN22 / FLORES sentence); by default IN22 text only feeds decontamination and the final generation (its length/digit statistics are printed for information; `USE_IN22_LENGTH_MATCH` / `USE_IN22_DIGIT_AUDIT` = `False`, so BPCC alone shapes the training length distribution and digit convention; unscored mined pairs are dropped unless that would starve a language); prompt/target boundary: `encode_examples` (training) and `generate_batch` (inference) both use `build_prompt` + `tokenizer(..., add_special_tokens=False)`; 20 stored examples decode back exactly |
| 8 | Generation logic | **PASS** | batched left-padded greedy output equals one-at-a-time output; order restored via `restore_order`; eval mode + `use_cache=True` during generation, previous mode restored in `finally`; caps `min(512−p, 8·src+32)` (en→X) and `min(512−p, 0.5·src+32)` (X→en); eos = pad = `<|endoftext|>`; no `stop_strings` |
| 9 | Metrics logic | **PASS** | Indic targets tokenised (hyp and ref), English untouched; id-aligned; bootstrap uses one set of resampled indices; private-API aggregate asserted == `corpus_score` (checked against sacrebleu 2.3.1) with the 300-resample multiprocessing fallback; **chrF++/BLEU are mandatory** (a failure stops the notebook with an error; sample files that do not match the benchmark's ids are refused) |
| 10 | Resource logic | **PASS** | raw downloads and `HF_HOME` under `/kaggle/temp` (fallback tmp); BPCC files are fetched one at a time and deleted; `check_disk` before downloads, `guard_working` before checkpoints / export; all deliverables are written directly into `project1_output/` (no duplicate copies of the model or checkpoint); gated downloads authenticated with `HF_TOKEN` only |
| 11 | Execution flow | **PASS** | one linear notebook: no `STAGES`, `FORCE`, stage markers, CPU/GPU branching, cross-session hand-over, Kaggle Dataset or Hugging Face upload; a failing cell raises and stops *Run All* (nothing is skipped silently); incomplete training (clock or instability) prints the final report as INCOMPLETE and stops before export / evaluation; the checkpoint carries a run signature (config + revisions + code + pool sha256) and is resumed only if it matches (model, optimizer, scaler, all RNGs, step; data position = step x 64, nothing skipped or repeated; a failed resume falls back to pristine weights); final status is COMPLETE only without deviations (COMPLETE WITH DEVIATIONS / INCOMPLETE otherwise) |
| 12 | Recovery logic | **PASS (by code inspection)** | early OOM -> pristine weights are reloaded and training restarts at step 0 on the next ladder rung; mid-run OOM -> the SAME global batch is recomputed with a 25% smaller micro-batch budget (no data skipped); an fp16 gradient overflow (update skipped by the scaler, state untouched) -> the SAME batch is replayed up to `NONFINITE_REPLAYS` times and counted; only a batch that still fails is SKIPPED, and any skipped step makes the final status COMPLETE WITH DEVIATIONS; ordinary high-loss batches are logged and trained on (never discarded); optimizer failure or missing bitsandbytes -> `AdamW` + 8 layers, recorded as a deviation; checkpoints restore weights, optimizer, scaler, step and all RNGs, and only load when the run signature matches |
| 13 | Unit tests | **PASS** | 6 groups on synthetic data in C05e (incl. accumulation invariance): every pure function of C05 (+ `lr_schedule`, `group_by_budget`, `waterfill`, bootstrap, `lid_top1` fallback) |

### Cell → names table (AST-generated; `<Cxx>` = defining cell)
`reads (earlier)` = defined by an earlier cell; `call-time` = defined in a later cell but used only inside functions at call time.

| cell | defines | reads (earlier) | call-time |
|---|---|---|---|
| C02 | BENCH_DIR, BPCC_DIR, CKPT_DIR, EVAL_DIR, EXPORT_DIR, HF_HOME, HUMAN_DIR, MAN_DIR, METRICS_DIR, MODEL_SRC_DIR, +35 more |  |  |
| C03 | CONFIG, c |  |  |
| C03b | i | torch<C02> |  |
| C04 | GatedRepoError, HF_TOKEN, HfApi, HfHubHTTPError, S, api, f, fetch_model, gated_test, get_secret, +5 more | BPCC_DIR<C02>, CONFIG<C03>, MODEL_SRC_DIR<C02>, hashlib<C02>, json<C02> |  |
| U01 | UTILS_SRC |  |  |
| U02 | UTILS_SRC |  |  |
| U03 | UTILS_SRC |  |  |
| U04 | UTILS_SRC |  |  |
| U05 | UTILS_SRC |  |  |
| U06 | UTILS_SRC |  |  |
| U07 | UTILS_SRC |  |  |
| U08 | UTILS_SRC |  |  |
| U09 | UTILS_SRC |  |  |
| U10 | UTILS_SRC |  |  |
| C05e | DEMO_FIXTURES, DEMO_FIXTURES_2, DIRECTIONS, EOS_TOKEN, INDIC, JsonlLogger, LANGS, NAME2CODE, SESSION, Timer, +33 more | CONFIG<C03>, T0<C02>, UTILS_SRC<U01>, sys<C02> |  |
| C05f | RUN_LOG, config_sig, popen_utils, prep_sig, run_utils, session_left_h | CONFIG<C03>, JsonlLogger<C05e>, S<C04>, SESSION<C05e>, TRAIN_DIR<C02>, UTILS_SRC<U01>, WORK<C02>, hashlib<C02>, json<C02>, +1 more |  |
| C05i | DATA_FILES, disk_report, guard_working, mf_set, prep_cache_valid, require_time, time_plan, timed, verify_data_dir | CONFIG<C03>, PREP_DIR<C02>, RAW<C02>, S<C04>, SESSION<C05e>, WORK<C02>, atomic_save<C05e>, dir_gb<C05e>, free_gb<C05e>, +4 more |  |
| C06 | HUMAN_RE, SCORE_RE, SKIP_STEM_RE, build_sources, c06, load_tok | CONFIG<C03>, INDIC<C05e>, MODEL_SRC_DIR<C02>, RAW<C02>, S<C04>, check_disk<C05e>, fetch_model<C04>, list_bpcc<C04>, mf_set<C05i>, +1 more |  |
| C07a | BENCH_LABEL, BENCH_SETS, LANG_RE, PAIR_DIR_RE, SKIP_EXT, zipfile |  |  |
| C07a2 |  | BENCH_LABEL<C07a>, LANGS<C05e>, LANG_RE<C07a>, PAIR_DIR_RE<C07a>, SKIP_EXT<C07a>, _bench_set_of<C07a>, _read_lines<C07a> |  |
| C07a3 |  | BENCH_LABEL<C07a>, BENCH_SETS<C07a>, DIRECTIONS<C05e>, INDIC<C05e>, LANGS<C05e>, _one_copy<C07a2>, numerals<C05e>, split_direction<C05e> |  |
| C07a4 | load_in22_hf | CONFIG<C03>, HF_TOKEN<C04>, LANGS<C05e> |  |
| C07b | c07, ensure_bench | BENCH_DIR<C02>, BENCH_LABEL<C07a>, BENCH_SETS<C07a>, BPCC_DIR<C02>, CONFIG<C03>, DIRECTIONS<C05e>, S<C04>, _build_bench<C07a3>, _check_alignment<C07a3>, +9 more |  |
| C08a | csv, fetch, itertools, release | BPCC_DIR<C02>, CONFIG<C03>, GatedRepoError<C04>, HF_TOKEN<C04>, HfHubHTTPError<C04>, S<C04>, hf_hub_download<C04> |  |
| C08a1 | sniff_tsv | _HDR<C08a>, _LANGCODE<C08a>, _csv_records<C08a>, _iter_lines<C08a>, _score<C08a>, itertools<C08a>, script_ratio<C05e> |  |
| C08a2 | read_aligned, read_tsv | CONFIG<C03>, _count_lines<C08a>, _count_records<C08a>, _csv_records<C08a>, _iter_lines<C08a>, _score<C08a>, fetch<C08a>, itertools<C08a>, math<C02>, +2 more |  |
| C08b | c08 | CONFIG<C03>, RAW<C02>, S<C04>, disk_report<C05i>, free_gb<C05e>, mf_set<C05i>, prep_cache_valid<C05i>, read_aligned<C08a2>, read_tsv<C08a2>, +2 more |  |
| C09 | AUDIT_DIR, c09, matplotlib, plt | CONFIG<C03>, LANGS<C05e>, PREP_DIR<C02>, S<C04>, len_bucket<C05e>, load_tok<C06>, mf_set<C05i>, script_ratio<C05e>, timed<C05i> |  |
| C10 | c10 | AUDIT_DIR<C09>, CONFIG<C03>, S<C04>, has_noise<C05e>, mf_set<C05i>, norm_text<C05e>, numeral_check<C05e>, script_ratio<C05e>, sym_digit_frac<C05e>, +1 more |  |
| C11 | c11 | AUDIT_DIR<C09>, CONFIG<C03>, INDIC<C05e>, S<C04>, hf_hub_download<C04>, lid_top1<C05e>, mf_set<C05i>, timed<C05i> |  |
| C12 | c12 | AUDIT_DIR<C09>, BENCH_SETS<C07a>, CONFIG<C03>, S<C04>, dedup_key<C05e>, mf_set<C05i>, timed<C05i>, word_ngrams<C05e> |  |
| C13 | c13 | CONFIG<C03>, INDIC<C05e>, S<C04>, digit_counts<C05e>, mf_set<C05i>, timed<C05i>, to_ascii_digits<C05e>, to_native_digits<C05e> |  |
| C14a | select_pairs | largest_remainder<C05e>, math<C02>, waterfill<C05e> |  |
| C14b | c14, to_dir_rows | CONFIG<C03>, INDIC<C05e>, S<C04>, len_bucket<C05e>, mf_set<C05i>, select_pairs<C14a>, split_direction<C05e>, timed<C05i> |  |
| C15 | c15 | CONFIG<C03>, DIRECTIONS<C05e>, EOS_TOKEN<C05e>, LANGS<C05e>, S<C04>, build_prompt<C05e>, encode_examples<C05e>, load_tok<C06>, mf_set<C05i>, +3 more |  |
| C16a | c16a | CONFIG<C03>, DIRECTIONS<C05e>, PREP_DIR<C02>, S<C04>, atomic_save<C05e>, disk_report<C05i>, json<C02>, mf_set<C05i>, timed<C05i> |  |
| C16b | c16b | CONFIG<C03>, DATA_FILES<C05i>, PREP_DIR<C02>, S<C04>, UTILS_SRC<U01>, VERSIONS<C02>, atomic_save<C05e>, hashlib<C02>, prep_sig<C05f>, +6 more |  |
| C17 | RUN, c17 | CONFIG<C03>, EOS_TOKEN<C05e>, LANGS<C05e>, PREP_DIR<C02>, S<C04>, TRAIN_DIR<C02>, atomic_save<C05e>, build_prompt<C05e>, json<C02>, +3 more |  |
| C18 | Batcher, c18 | CONFIG<C03>, DIRECTIONS<C05e>, S<C04>, gather_targets<C05e>, group_by_budget<C05e>, target_position_mask<C05e>, torch<C02> |  |
| C19a | PER_LAYER_PARAMS, build_model, free_model, make_scaler, rungs_after | CONFIG<C03>, MODEL_SRC_DIR<C02>, inspect<C02>, torch<C02> |  |
| C19b | c19 | CONFIG<C03>, S<C04>, build_model<C19a>, disk_report<C05i>, fetch_model<C04>, rungs_after<C19a> |  |
| C19c | c19c | BENCH_SETS<C07a>, CONFIG<C03>, DIRECTIONS<C05e>, LANGS<C05e>, RUN<C17>, S<C04>, WORK<C02>, build_prompt<C05e>, chunked_ce<C05e>, +8 more |  |
| C20 | eval_generation | CONFIG<C03>, DIRECTIONS<C05e>, S<C04>, generate_batch<C05e>, inspect<C02>, load_tok<C06>, score_chrf<C05e>, split_direction<C05e>, torch<C02> |  |
| C21a | OptimizerFailure, TS, compute_wall, dev_loss_eval, forward_backward, mb_forward, optimizer_step | CONFIG<C03>, DIRECTIONS<C05e>, RUN<C17>, S<C04>, chunked_ce<C05e>, gather_targets<C05e>, math<C02>, require_time<C05i>, target_position_mask<C05e>, +2 more |  |
| C21b | next_rung, rebuild, run_signature, save_local_ckpt, save_trained_weights, try_resume | CKPT_DIR<C02>, CONFIG<C03>, RUN<C17>, RUN_LOG<C05f>, S<C04>, atomic_save<C05e>, build_model<C19a>, config_sig<C05f>, disk_report<C05i>, +8 more |  |
| C21b2 | print_final_report | CONFIG<C03>, DIRECTIONS<C05e> |  |
| C21c | attempt_step, do_step, finish_training, note_skip, safe_eval | JsonlLogger<C05e>, OptimizerFailure<C21a>, RUN<C17>, RUN_LOG<C05f>, S<C04>, TRAIN_DIR<C02>, atomic_save<C05e>, eval_generation<C20>, forward_backward<C21a>, +8 more |  |
| C21d | c21, train_loop | CKPT_DIR<C02>, CONFIG<C03>, DIRECTIONS<C05e>, JsonlLogger<C05e>, RUN<C17>, RUN_LOG<C05f>, S<C04>, SESSION<C05e>, TRAIN_DIR<C02>, +15 more |  |
| C22a | FRESH_LOAD, check_export_weights, load_run_summary, shutil, write_model_card | CONFIG<C03>, EXPORT_DIR<C02>, MODEL_SRC_DIR<C02>, S<C04>, TRAIN_DIR<C02>, atomic_save<C05e>, json<C02>, run_signature<C21b>, sha256_file<C05e>, +1 more |  |
| C22c | c22 | CONFIG<C03>, DEMO_FIXTURES<C05e>, DIRECTIONS<C05e>, EOS_TOKEN<C05e>, EXPORT_DIR<C02>, FRESH_LOAD<C22a>, LANGS<C05e>, S<C04>, TRAIN_DIR<C02>, +21 more |  |
| C23b | c23, job_complete, run_workers | CONFIG<C03>, DIRECTIONS<C05e>, EVAL_DIR<C02>, EXPORT_DIR<C02>, S<C04>, SCRATCH<C02>, TRAIN_DIR<C02>, _export_valid<C22a>, _sha_cached<C22a>, +13 more |  |
| C24 | c24 | S<C04>, _pip<C02>, sys<C02> |  |
| C25 | c25, run_sub | CONFIG<C03>, DIRECTIONS<C05e>, EVAL_DIR<C02>, S<C04>, SCRATCH<C02>, atomic_save<C05e>, ensure_bench<C07b>, hf_hub_download<C04>, json<C02>, +2 more |  |
| C26a | c26a | CONFIG<C03>, DIRECTIONS<C05e>, EVAL_DIR<C02>, HUMAN_DIR<C02>, METRICS_DIR<C02>, S<C04>, SCRATCH<C02>, atomic_save<C05e>, json<C02>, +2 more |  |
| C26b | PAL, c26b, style | DIRECTIONS<C05e>, EXPORT_DIR<C02>, METRICS_DIR<C02>, S<C04>, TRAIN_DIR<C02>, json<C02>, matplotlib<C09>, plt<C09>, write_model_card<C22a> |  |
| C27 | Translator, c27, is_source_copy | CONFIG<C03>, EXPORT_DIR<C02>, NAME2CODE<C05e>, S<C04>, build_prompt<C05e>, generate_batch<C05e>, norm_text<C05e>, torch<C02> |  |
| C28 | c28 | DEMO_FIXTURES<C05e>, DEMO_FIXTURES_2<C05e>, DIRECTIONS<C05e>, LANGS<C05e>, METRICS_DIR<C02>, S<C04>, atomic_save<C05e>, is_source_copy<C27>, split_direction<C05e> |  |
| C30a | final_status | CONFIG<C03>, DIRECTIONS<C05e>, EVAL_DIR<C02>, EXPORT_DIR<C02>, json<C02> |  |
| C30b | build_final_manifest, write_manifests | CONFIG<C03>, DIRECTIONS<C05e>, EXPORT_DIR<C02>, MAN_DIR<C02>, PREP_DIR<C02>, S<C04>, TRAIN_DIR<C02>, UTILS_SRC<U01>, _SIG_KEYS<C05f>, +7 more |  |
| C30 | c30 | DIRECTIONS<C05e>, EXPORT_DIR<C02>, HUMAN_DIR<C02>, METRICS_DIR<C02>, OUT_DIR<C02>, S<C04>, TRAIN_DIR<C02>, disk_report<C05i>, final_status<C30a>, +5 more |  |

**Files:** no `.py` file is ever written: the helper source is the string `UTILS_SRC` (cells U01-U09), exec'd in C05e and piped on stdin to the generation workers and the metrics subprocess (`python - worker|metrics-base <cfg.json>`) · processed data → `/kaggle/working/data/` (`pool.parquet`, `dev_loss.parquet`, `flores_dev_sample.jsonl`, `manifest.json`, `DATA_CARD.md`, `audit/`) · everything else → `/kaggle/working/project1_output/` (`checkpoints/last.pt`, `final_model/`, `predictions/samples_<direction>_<set>.jsonl`, `metrics/`, `human_eval/`, `logs/`, `manifests/`).

### Assumptions that could NOT be checked statically (and what happens if they are wrong)
1. **BPCC TSV column layout** — the folder/file layout was verified from the live Hub listing (human subsets `bpcc-seed-*`, `nllb_seed`, `daily`, `wiki`, `ilci`, `massive`; mined `samanantar_v2`, `samanantar_v0.3_filtered`, `nllb_filtered`, `comparable`; `<subset>/<lang>.tsv`), but the files are gated so the **TSV columns could not be read**: `sniff_tsv` first recognises the explicit `src_lang | tgt_lang | src | tgt` schema (declared languages verified, quote-aware csv parsing, no ratio detector) and otherwise detects header / English column / Indic column (by script) / score column at run time, and anything unrecognised **stops with the first rows printed**.
2. **IN22 / FLORES zip internals** — not readable in advance (gated). The sniffer is **pair-aware**: a `LANG1-LANG2` directory is read as one aligned pair (equal line counts enforced, never truncated; files of different pair directories are never compared with each other), plain per-language files are the fallback, and pair directories that involve other languages (e.g. `eng_Latn-pan_Guru`) are ignored; with an HF-datasets fallback for IN22 (`sentence_<code>` fields; those datasets are auto-gated, accept their terms too); FLORES has no fallback and stops with a listing.
3. **Qwen3 in transformers 4.57** — verified against the 4.57.0 source (decoder layers, final norm inside `Qwen3Model.forward`, tied embeddings, 28 layers, hidden 2048, EOS 151643); runtime assertions remain as a second guard.
4. **fp16 on T4** — Qwen activations may overflow in fp16; the skip / halve / stop guards handle it, but nothing was observed on real hardware. bitsandbytes 8-bit + GradScaler interplay is standard but was not run here.
5. **Time budget (the main risk).** Preparation and training share ONE 12 h session. The training wall cap is `min(8 h, 12 − elapsed − 2 − 1)`; 249,600 examples need about 6.6 h of step time at the previously MEASURED 10.49 examples/s (about 7.1 h with 8% for dev evaluations and checkpoints), so preparation must finish within about 1.6 h of the session start (training then starts at about 1.86 h after the 0.25 h model download / validation reserve); ONE budget function, `time_plan`, is used by C16b (which stops right after preparation) and by C21 (`compute_wall`), with the pre-training, post-training (2 h) and safety (1 h) reserves. The session clock survives kernel restarts, and processed data with a matching prep signature + sha256 is reused instead of re-parsing BPCC. If the cap ends training before 3,900 steps the run is reported INCOMPLETE (no export, no evaluation); the last checkpoint is kept in `project1_output/checkpoints/`.
6. **Never executed here:** the full notebook (no GPU, no gated data in the authoring environment), the two-GPU worker launch, fastText label strings under NumPy 2, `/kaggle/temp` existence. The pure helper functions have synthetic-data unit tests in C05e (run when the notebook starts).
7. **Peak CPU RAM of preparation** depends on the number of mined subsets (`MINED_PARSE_CAP` keeps it bounded).

### Acceptance checklist
- [x] Single linear Kaggle-GPU notebook, `Run All`; GPU asserted in C03b  - [x] No `STAGES` / `FORCE` / `run_cell` / stage markers / CPU-GPU branching  - [x] No Kaggle Dataset, no cross-session hand-over
- [x] Hugging Face read/download only (`HF_TOKEN` for gated BPCC and the pinned model; no repository created, nothing uploaded)  - [x] BPCC, Qwen3, IN22/FLORES, language-ID downloaded automatically (pinned revisions)
- [x] Exactly N = 249,600 enforced (prep fails otherwise)  - [x] One training pass of 3,900 steps; top 10 layers + final norm; no LoRA / PEFT / Trainer / TRL  - [x] No smoke test, trial run, throughput probe, second run or hyper-parameter search
- [x] Incomplete training is reported INCOMPLETE and stops the notebook  - [x] Final model saved locally and reloaded with `from_pretrained()` (greedy parity + fresh-process load)  - [x] No COMET
- [x] IN22-Gen and IN22-Conv evaluated (exactly one prediction per sentence); FLORES-22 dev chrF++ at 25/50/75/100 % of the same run  - [x] chrF++, BLEU, 95 % bootstrap CI, sanity metrics  - [x] Human-evaluation CSVs with blank scores
- [x] Final report printed; outputs in `/kaggle/working/project1_output/` (`final_model/ predictions/ metrics/ human_eval/ logs/ manifests/ checkpoints/`)